In [ ]:
from IPython.display import Markdown, display

# Практическая тетрадь: Physics Attractor

Эта Jupyter-тетрадь предназначена для последовательного анализа и экспериментов.

Теория находится отдельно: [`docs/THEORY.md`](../docs/THEORY.md).

**Принцип работы:** выполняем один TODO, проверяем его результат и только
после этого переходим к следующему. SINDy будем использовать через PySINDy,
а не реализовывать вручную.

In [ ]:
# Импорты для всех запланированных этапов проекта.
from pathlib import Path
import warnings
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
# Ищем корень проекта, чтобы notebook одинаково работал при запуске
# из корня репозитория и из каталога notebooks/.
_archive_relative_path = Path("data/physics-attractor-time-series.zip")
_project_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (
        _candidate
        for _candidate in _project_candidates
        if (_candidate / "pyproject.toml").exists()
        and (_candidate / _archive_relative_path).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден data/physics-attractor-time-series.zip рядом с pyproject.toml. "
        "Сначала положи локальный архив датасета в папку data/."
    )

DATA_ARCHIVE = PROJECT_ROOT / _archive_relative_path
raw_df = pd.read_csv(DATA_ARCHIVE, compression="zip")

# Это исходные данные без преобразований. Их исходный порядок сохраняем,
# чтобы не потерять порядок регистрации наблюдений до проверки времени.

In [ ]:
dt = raw_df["time"].diff().dropna()
empty_columns = raw_df.columns[raw_df.isna().all()].tolist()

display(
    Markdown(
        f"""
        ## Датасет загружен

        - архив: `{_archive_relative_path}`;
        - строк: **{len(raw_df):,}**;
        - исходные столбцы: `{', '.join(raw_df.columns)}`;
        - полностью пустые столбцы: **{len(empty_columns)}**;
        - время: от **{raw_df['time'].min():.3f}** до **{raw_df['time'].max():.3f}**;
        - шаг времени: от **{dt.min():.3f}** до **{dt.max():.3f}**.

        Данные показаны без очистки. Ниже удаляем только технически пустые
        столбцы, нормализуем имена и отдельно проверяем, допустима ли сортировка.
        """
    )
)

In [ ]:
# Первые пять исходных строк — без очистки и преобразований.
raw_df.head()

In [ ]:
raw_df.isna().sum()

In [ ]:
# Короткие x1/y1/x2/y2 совпадают с математическими обозначениями в теории
# и станут понятными feature names в PySINDy. Названия диагностических
# полей не уточняем, пока их физический смысл неизвестен.
STATE_COLUMNS = ["x1", "y1", "x2", "y2"]
DIAGNOSTIC_COLUMNS = ["distance", "angle1", "angle2"]

_rename_columns = {
    "pos1x": "x1",
    "pos1y": "y1",
    "pos2x": "x2",
    "pos2y": "y2",
}

df = raw_df.dropna(axis="columns", how="all").copy()
df.columns = df.columns.str.strip()
df = df.rename(columns=_rename_columns)

_expected_columns = ["time", *STATE_COLUMNS, *DIAGNOSTIC_COLUMNS]
_missing_columns = sorted(set(_expected_columns) - set(df.columns))
_unexpected_columns = sorted(set(df.columns) - set(_expected_columns))
if _missing_columns or _unexpected_columns:
    raise ValueError(
        "Неожиданная схема данных. "
        f"Отсутствуют: {_missing_columns}; лишние: {_unexpected_columns}."
    )

# Явное преобразование не подменяет ошибки пропусками: любой текст или
# повреждённое число остановит ячейку с понятным исключением.
df = df.loc[:, _expected_columns].apply(pd.to_numeric, errors="raise")
df.index.name = "source_row"

df.head()

In [ ]:
data_dictionary = pd.DataFrame(
    [
        ("time", "время наблюдения", "единица времени неизвестна", "модель/ось"),
        ("x1", "x-координата объекта 1", "единица координат неизвестна", "состояние"),
        ("y1", "y-координата объекта 1", "единица координат неизвестна", "состояние"),
        ("x2", "x-координата объекта 2", "единица координат неизвестна", "состояние"),
        ("y2", "y-координата объекта 2", "единица координат неизвестна", "состояние"),
        ("distance", "исходный диагностический признак", "физический смысл неизвестен", "не моделировать"),
        ("angle1", "исходный угловой признак 1", "физический смысл неизвестен", "не моделировать"),
        ("angle2", "исходный угловой признак 2", "физический смысл неизвестен", "не моделировать"),
    ],
    columns=["column", "working_description", "units_or_limitation", "role"],
).set_index("column")

data_dictionary

In [ ]:
_time_diff = df["time"].diff()
_empty_column_count = int(raw_df.isna().all(axis="rows").sum())
_remaining_missing = int(df.isna().sum().sum())
_duplicate_rows = int(df.duplicated().sum())
_duplicate_times = int(df["time"].duplicated().sum())
_time_order_issues = int(_time_diff.dropna().le(0).sum())
_all_numeric = bool(df.dtypes.apply(pd.api.types.is_numeric_dtype).all())

quality_checks = pd.DataFrame(
    [
        ("Удалены полностью пустые столбцы", _empty_column_count, _empty_column_count == 8),
        ("Пропуски после очистки", _remaining_missing, _remaining_missing == 0),
        ("Полные дубликаты строк", _duplicate_rows, _duplicate_rows == 0),
        ("Повторяющиеся значения time", _duplicate_times, _duplicate_times == 0),
        ("Невозрастающие шаги time (dt <= 0)", _time_order_issues, _time_order_issues == 0),
        ("Все содержательные столбцы числовые", _all_numeric, _all_numeric),
    ],
    columns=["check", "observed", "passed"],
)

time_step_summary = pd.Series(
    {
        "min": _time_diff.dropna().min(),
        "median": _time_diff.dropna().median(),
        "max": _time_diff.dropna().max(),
        "unique_values": _time_diff.dropna().nunique(),
    },
    name="dt",
).to_frame()

In [ ]:
display(
    Markdown(
        r"""
        ## Результат очистки и решение по сортировке

        Координаты переименованы в `x1`, `y1`, `x2`, `y2`: эти имена
        соответствуют формулам в теории, компактны на графиках и напрямую
        подходят как имена признаков PySINDy. `time` оставлено явным, а
        `distance`, `angle1`, `angle2` не переименованы: их смысл пока не
        установлен, и новое содержательное имя создало бы ложную трактовку.

        **Глобальная сортировка не выполняется.** Исходный порядок строк —
        часть временных данных. В текущем файле `time` уже строго возрастает,
        поэтому сортировка ничего не исправит. Если в новом файле время
        сбросится (`dt <= 0`), это признак начала новой записи и будущая
        граница `segment_id`, а не повод перемешивать записи общей сортировкой.
        Сортировать по `time` можно только внутри уже подтверждённого сегмента,
        если отдельно установлено, что строки лишь записаны не по порядку и
        каждое значение времени уникально.
        """
    )
)
display(Markdown("### Проверки качества"))
display(quality_checks)
display(Markdown("### Неравномерность временного шага"))
display(time_step_summary)

## Координаты как функции времени

Серые траектории `x`–`y` выше — это только проекция движения на плоскость:
там не видно, в какой момент пройдена каждая точка. На графиках ниже по
горизонтали отложено настоящее `time`, поэтому можно проверить
непрерывность координат напрямую.

Слева — вся запись, справа — увеличенная окрестность одного из редких
больших шагов времени. Красная вертикаль показывает точку после паузы
записи.

In [ ]:
# Эти границы можно менять, чтобы рассмотреть любой фрагмент записи.
_time_start = 808
_time_end = 812
_zoom_df = df[df["time"].between(_time_start, _time_end)]
_zoom_candidates = _zoom_df[_zoom_df["time"].diff().gt(0.06)]

_time_fig, _time_axes = plt.subplots(2, 2, figsize=(14, 7), sharey="row")

for _row, (_x_column, _y_column, _object_name) in enumerate(
    [
        ("x1", "y1", "объект 1"),
        ("x2", "y2", "объект 2"),
    ]
):
    _overview_ax = _time_axes[_row, 0]
    _overview_ax.plot(df["time"], df[_x_column], linewidth=0.45, label="x")
    _overview_ax.plot(df["time"], df[_y_column], linewidth=0.45, label="y")
    _overview_ax.set_title(f"{_object_name}: вся запись")

    _zoom_ax = _time_axes[_row, 1]
    _zoom_ax.plot(_zoom_df["time"], _zoom_df[_x_column], label="x")
    _zoom_ax.plot(_zoom_df["time"], _zoom_df[_y_column], label="y")
    for _candidate_time in _zoom_candidates["time"]:
        _zoom_ax.axvline(
            _candidate_time,
            color="crimson",
            linestyle="--",
            linewidth=1.2,
            label="точка после dt > 0.06",
        )
    _zoom_ax.set_title(f"{_object_name}: увеличение {_time_start}–{_time_end}")

    for _ax in (_overview_ax, _zoom_ax):
        _ax.set_xlabel("time")
        _ax.set_ylabel("координата")
        _ax.grid(alpha=0.25)
        _ax.legend()

_time_fig.suptitle("Координаты объектов во времени")
_time_fig.tight_layout()

_time_fig

In [ ]:
_trajectory_fig, _axes = plt.subplots(1, 2, figsize=(12, 5))

for _ax, _x_column, _y_column, _title in [
    (_axes[0], "x1", "y1", "Траектория объекта 1"),
    (_axes[1], "x2", "y2", "Траектория объекта 2"),
]:
    _ax.plot(
        df[_x_column],
        df[_y_column],
        color="slategray",
        linewidth=0.5,
        alpha=0.7,
        label="траектория",
    )
    _ax.set_xlabel("x")
    _ax.set_ylabel("y")
    _ax.set_title(_title)
    _ax.set_aspect("equal", adjustable="box")
    _ax.grid(alpha=0.25)
    _ax.legend()

_trajectory_fig.suptitle("Движение в плоскости: время не является осью")
_trajectory_fig.tight_layout()

_trajectory_fig

## Промежуточный вывод: проверка времени

В исходном порядке строк время всегда возрастает: сбросов `time` нет.
Четыре сравнительно больших временных шага проверены на графиках координат
от времени и в плоскости `x`–`y`; заметных скачков положения не видно.

Эта проверка исключает разрывы самого времени, но ещё не доказывает
непрерывность траекторий. Дальше отдельно проверяем пространственный шаг между
соседними положениями объектов.

## Пространственный шаг между соседними наблюдениями

Для каждого объекта считаем евклидово расстояние между двумя соседними
положениями. Общий диагностический признак `spatial_step` — максимум из шагов
двух объектов: так разрыв будет заметен, даже если резко переместился только
один объект.

На этом этапе только рассчитываем признаки и смотрим на их распределение.
Порог разрыва и `segment_id` выберем отдельно после проверки графика и таблицы.

In [ ]:
step_df = df.copy()
step_df["dt"] = step_df["time"].diff()

step_df["step_object1"] = np.hypot(
    step_df["x1"].diff(),
    step_df["y1"].diff(),
)
step_df["step_object2"] = np.hypot(
    step_df["x2"].diff(),
    step_df["y2"].diff(),
)
step_df["spatial_step"] = step_df[
    ["step_object1", "step_object2"]
].max(axis="columns")

largest_spatial_steps = step_df.nlargest(15, "spatial_step")[
    ["time", "dt", "step_object1", "step_object2", "spatial_step"]
]

In [ ]:
# Крупнейшие шаги: здесь удобно искать разрыв между обычным движением
# и редкими пространственными скачками.
largest_spatial_steps

In [ ]:
_step_fig, _step_ax = plt.subplots(figsize=(14, 5))

_step_ax.plot(
    step_df["time"],
    step_df["step_object1"],
    linewidth=0.55,
    alpha=0.75,
    label="объект 1",
)
_step_ax.plot(
    step_df["time"],
    step_df["step_object2"],
    linewidth=0.55,
    alpha=0.75,
    label="объект 2",
)
_step_ax.set_yscale("log")
_step_ax.set_xlabel("time")
_step_ax.set_ylabel("пространственный шаг")
_step_ax.set_title("Шаг между соседними положениями объектов")
_step_ax.grid(alpha=0.25)
_step_ax.legend()
_step_fig.tight_layout()

_step_fig

### Локальная проверка крупнейших шагов

Выбери строку из списка, чтобы рассмотреть небольшой фрагмент движения вокруг
неё. Верхние графики показывают координаты от времени. Средние показывают путь
каждого объекта в плоскости `x`–`y`: синяя линия идёт до проверяемого шага,
оранжевая — после, а красная стрелка показывает переход между двумя соседними
точками, для которых рассчитан выбранный `spatial_step`.

Пределы осей `x` и `y` зафиксированы по всему датасету и не меняются при
переключении кандидата. Нижний график также использует один масштаб для всех
кандидатов и показывает величину шага рядом с выбранной точкой.

В списке оставлены 15 крупнейших шагов: так можно сравнить редкие большие
скачки с верхней границей обычного движения, не выбирая порог заранее.

In [ ]:
spatial_candidate_labels = [
    (
        f"{_rank:02d}. строка {_index}: "
        f"time={_row['time']:.3f}, "
        f"spatial_step={_row['spatial_step']:.3f}"
    )
    for _rank, (_index, _row) in enumerate(
        largest_spatial_steps.iterrows(),
        start=1,
    )
]

# Меняй число от 0 до 14 и повторно выполняй эту и следующие три ячейки,
# чтобы исследовать другой крупный пространственный шаг.
selected_candidate_rank = 0
print(spatial_candidate_labels[selected_candidate_rank])

In [ ]:
_candidate_rank = selected_candidate_rank
selected_step_index = int(largest_spatial_steps.index[_candidate_rank])
selected_step_row = step_df.loc[selected_step_index]

_selected_position = step_df.index.get_loc(selected_step_index)
_window_radius = 15
_window_start = max(0, _selected_position - _window_radius)
_window_stop = min(len(step_df), _selected_position + _window_radius + 1)
local_step_df = step_df.iloc[_window_start:_window_stop]

_neighbor_steps = local_step_df.loc[
    local_step_df.index != selected_step_index,
    "spatial_step",
].dropna()
typical_neighbor_step = _neighbor_steps.median()
selected_to_typical_ratio = (
    selected_step_row["spatial_step"] / typical_neighbor_step
)

In [ ]:
display(
    Markdown(
        f"""
        **Проверяемый шаг:** строка после скачка — `{selected_step_index}`,
        `time = {selected_step_row['time']:.3f}`, `dt = {selected_step_row['dt']:.3f}`.

        - объект 1: `{selected_step_row['step_object1']:.3f}`;
        - объект 2: `{selected_step_row['step_object2']:.3f}`;
        - общий `spatial_step`: **`{selected_step_row['spatial_step']:.3f}`**.

        Медиана соседних шагов: `{typical_neighbor_step:.3f}`. Выбранный шаг больше
        неё в **{selected_to_typical_ratio:.1f} раза**.
        """
    )
)

In [ ]:
_candidate_time = selected_step_row["time"]
_before_df = local_step_df[local_step_df.index < selected_step_index]
_after_df = local_step_df[local_step_df.index >= selected_step_index]
_previous_row = local_step_df.loc[selected_step_index - 1]

_x_min = step_df[["x1", "x2"]].min().min()
_x_max = step_df[["x1", "x2"]].max().max()
_y_min = step_df[["y1", "y2"]].min().min()
_y_max = step_df[["y1", "y2"]].max().max()
_x_padding = 0.03 * (_x_max - _x_min)
_y_padding = 0.03 * (_y_max - _y_min)

_local_fig = plt.figure(figsize=(14, 12), constrained_layout=True)
_local_grid = _local_fig.add_gridspec(
    3,
    2,
    height_ratios=[1, 1.15, 0.75],
)
_time_axes = [
    _local_fig.add_subplot(_local_grid[0, 0]),
    _local_fig.add_subplot(_local_grid[0, 1]),
]
_movement_axes = [
    _local_fig.add_subplot(_local_grid[1, 0]),
    _local_fig.add_subplot(_local_grid[1, 1]),
]
_local_step_ax = _local_fig.add_subplot(_local_grid[2, :])

for _ax, _x_column, _y_column, _title in [
    (_time_axes[0], "x1", "y1", "Объект 1: координаты от времени"),
    (_time_axes[1], "x2", "y2", "Объект 2: координаты от времени"),
]:
    _ax.plot(local_step_df["time"], local_step_df[_x_column], label="x")
    _ax.plot(local_step_df["time"], local_step_df[_y_column], label="y")
    _ax.axvline(
        _candidate_time,
        color="crimson",
        linestyle="--",
        linewidth=1.3,
        label="точка после шага",
    )
    _ax.set_xlabel("time")
    _ax.set_ylabel("координата")
    _ax.set_title(_title)
    _ax.grid(alpha=0.25)
    _ax.legend()

for _ax, _x_column, _y_column, _step_column, _object_name in [
    (_movement_axes[0], "x1", "y1", "step_object1", "Объект 1"),
    (_movement_axes[1], "x2", "y2", "step_object2", "Объект 2"),
]:
    _ax.plot(
        _before_df[_x_column],
        _before_df[_y_column],
        marker="o",
        markersize=2.5,
        color="steelblue",
        label="до шага",
    )
    _ax.plot(
        _after_df[_x_column],
        _after_df[_y_column],
        marker="o",
        markersize=2.5,
        color="darkorange",
        label="после шага",
    )
    _ax.annotate(
        "",
        xy=(selected_step_row[_x_column], selected_step_row[_y_column]),
        xytext=(_previous_row[_x_column], _previous_row[_y_column]),
        arrowprops={
            "arrowstyle": "-|>",
            "color": "crimson",
            "linewidth": 2.4,
            "mutation_scale": 16,
        },
    )
    _ax.scatter(
        _previous_row[_x_column],
        _previous_row[_y_column],
        facecolors="white",
        edgecolors="crimson",
        linewidths=1.8,
        s=52,
        zorder=3,
        label="точка до",
    )
    _ax.scatter(
        selected_step_row[_x_column],
        selected_step_row[_y_column],
        color="crimson",
        marker="X",
        s=62,
        zorder=3,
        label="точка после",
    )
    _ax.set_xlabel("x")
    _ax.set_ylabel("y")
    _ax.set_title(
        f"{_object_name}: выбранный шаг = "
        f"{selected_step_row[_step_column]:.3f}"
    )
    _ax.set_xlim(_x_min - _x_padding, _x_max + _x_padding)
    _ax.set_ylim(_y_min - _y_padding, _y_max + _y_padding)
    _ax.set_aspect("equal", adjustable="box")
    _ax.grid(alpha=0.25)
    _ax.legend()

_local_step_ax.plot(
    local_step_df["time"],
    local_step_df["step_object1"],
    marker="o",
    markersize=3,
    label="объект 1",
)
_local_step_ax.plot(
    local_step_df["time"],
    local_step_df["step_object2"],
    marker="o",
    markersize=3,
    label="объект 2",
)
_local_step_ax.axvline(
    _candidate_time,
    color="crimson",
    linestyle="--",
    linewidth=1.3,
    label="проверяемая точка",
)
_local_step_ax.scatter(
    [_candidate_time, _candidate_time],
    [
        selected_step_row["step_object1"],
        selected_step_row["step_object2"],
    ],
    color="crimson",
    marker="X",
    s=52,
    zorder=3,
)
_local_step_ax.set_ylim(0, step_df["spatial_step"].max() * 1.05)
_local_step_ax.set_xlabel("time")
_local_step_ax.set_ylabel("пространственный шаг")
_local_step_ax.set_title(
    "Локальные шаги: одинаковый масштаб для всех кандидатов"
)
_local_step_ax.grid(alpha=0.25)
_local_step_ax.legend()

_local_fig.suptitle(
    f"Локальное движение около time = {_candidate_time:.3f}"
)
_local_fig

## Разбиение на непрерывные сегменты

Локальная проверка показала, что первые десять крупнейших пространственных
шагов — настоящие переходы между разными участками записи, а начиная с
одиннадцатого движение остаётся непрерывным.

Чтобы не задавать порог на глаз, ниже ищем самый большой **относительный
разрыв** между соседними значениями в отсортированном `spatial_step`. Порог
помещаем внутрь найденного пустого интервала. Строка после каждого скачка
получает новый `segment_id` и становится первой строкой нового сегмента.

In [ ]:
_sorted_steps = step_df["spatial_step"].dropna().sort_values(ascending=False)
_neighbor_ratios = (
    _sorted_steps.iloc[:-1].to_numpy()
    / _sorted_steps.iloc[1:].to_numpy()
)
_gap_position = int(np.argmax(_neighbor_ratios))

break_count = _gap_position + 1
smallest_break_step = float(_sorted_steps.iloc[_gap_position])
largest_regular_step = float(_sorted_steps.iloc[_gap_position + 1])
spatial_break_threshold = float(
    np.sqrt(smallest_break_step * largest_regular_step)
)

segmented_df = step_df.copy()
segmented_df["is_spatial_break"] = segmented_df["spatial_step"].gt(
    spatial_break_threshold
)
segmented_df["segment_id"] = (
    segmented_df["is_spatial_break"].cumsum().astype(int)
)

break_rows = segmented_df.loc[
    segmented_df["is_spatial_break"],
    ["time", "spatial_step", "step_object1", "step_object2", "segment_id"],
].copy()
break_rows.index.name = "row_after_break"

segment_summary = (
    segmented_df.groupby("segment_id", as_index=False)
    .agg(
        row_start=("time", lambda _series: int(_series.index[0])),
        row_end=("time", lambda _series: int(_series.index[-1])),
        row_count=("time", "size"),
        time_start=("time", "min"),
        time_end=("time", "max"),
    )
)
segment_summary["duration"] = (
    segment_summary["time_end"] - segment_summary["time_start"]
)

In [ ]:
display(
    Markdown(
        f"""
        ### Вывод по разрывам

        По времени разрывов не обнаружено: `time` возрастает на всей записи,
        а редкие увеличенные `dt` не сопровождаются скачками координат.

        По пространственному шагу, наоборот, видны две чётко разделённые
        группы. Крупнейший обычный шаг равен `{largest_regular_step:.3f}`,
        самый маленький подтверждённый скачок —
        `{smallest_break_step:.3f}`, а между ними нет наблюдений. Поэтому
        значение `{spatial_break_threshold:.3f}`, расположенное внутри этого
        пустого интервала, используется как диагностический порог:
        `spatial_step > {spatial_break_threshold:.3f}` означает начало нового
        сегмента.

        Этот порог не является физической константой. Любое значение между
        `{largest_regular_step:.3f}` и `{smallest_break_step:.3f}` даст то же
        разбиение. В результате выделено **{break_count} переходов** и
        получено **{len(segment_summary)} непрерывных сегментов разной длины**.

        Дальнейшие графики, производные и модели нужно рассчитывать внутри
        каждого `segment_id`, не соединяя соседние сегменты через скачок.

        Ниже перечислены строки, с которых начинаются новые сегменты.
        """
    )
)
display(break_rows)
display(Markdown("### Размеры сегментов"))
display(segment_summary)

In [ ]:
_segment_fig, _segment_axes = plt.subplots(1, 2, figsize=(14, 6))
_segment_colors = plt.get_cmap("tab20", len(segment_summary))

for _segment_id, _segment in segmented_df.groupby("segment_id", sort=True):
    _color = _segment_colors(_segment_id)
    for _ax, _x_column, _y_column, _object_name in [
        (_segment_axes[0], "x1", "y1", "Объект 1"),
        (_segment_axes[1], "x2", "y2", "Объект 2"),
    ]:
        # Каждый segment_id рисуется отдельной линией: переходы между
        # сегментами намеренно не соединяются.
        _ax.plot(
            _segment[_x_column],
            _segment[_y_column],
            color=_color,
            linewidth=0.65,
            alpha=0.85,
            label=f"segment {_segment_id}",
        )
        _ax.scatter(
            _segment[_x_column].iloc[0],
            _segment[_y_column].iloc[0],
            color=_color,
            edgecolors="black",
            linewidths=0.35,
            s=22,
            zorder=3,
        )
        _ax.set_xlabel("x")
        _ax.set_ylabel("y")
        _ax.set_title(_object_name)
        _ax.set_aspect("equal", adjustable="box")
        _ax.grid(alpha=0.25)

_handles, _labels = _segment_axes[1].get_legend_handles_labels()
_segment_fig.legend(
    _handles,
    _labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    title="Непрерывные сегменты",
)
_segment_fig.suptitle(
    "Траектории после сегментации: линии не проходят через разрывы"
)
_segment_fig.tight_layout(rect=(0, 0, 0.88, 1))

_segment_fig

## TODO 1 — загрузка и аудит данных

- [x] Загрузить `data.csv` из локального архива в `data/`.
- [x] Удалить полностью пустые столбцы.
- [x] Проверить типы, пропуски, дубликаты и порядок по `time`.
- [x] Подтвердить диапазон и неравномерность временных шагов.
- [x] Не приписывать физический смысл `distance`, `angle1`, `angle2`.

**Ожидаемый артефакт:** короткая таблица data dictionary и две-три проверки
качества данных.

## TODO 2 — траектории и сбросы

- [x] Построить `x1`–`y1` и `x2`–`y2`.
- [x] Построить все четыре координаты от времени.
- [x] Проверить возможные разрывы по `time`.
- [x] Рассчитать пространственный шаг каждого объекта.
- [x] Выбрать порог разрыва и создать `segment_id`.
- [x] Проверить, что графики не соединяют разные сегменты.

## TODO 3 — скорости и фазовые проекции

- [x] На одном чистом сегменте вычислить скорости по реальному time.
- [x] Сравнить центральные разности и Savitzky–Golay.
- [x] Построить скорость от времени.
- [x] Построить x–v_x, y–v_y, центр и относительное движение.

**Стоп-критерий выполнен:** производные и переносимость единого `W = 11`
проверены на всех сегментах, а недостоверные краевые значения явно
исключены единой маской.

### TODO 3.A — доказательства перед выбором сглаживания

На этом диагностическом этапе итоговое окно Savitzky–Golay **не выбирается**.
Сначала сравнивается плотная сетка всех нечётных окон от 5 до 31, отдельно
доказывается допустимая граница производной, а затем повторяется контроль на
равномерной временной сетке.

Рабочая цепочка для каждого окна:

$$
X=(x_1,y_1,x_2,y_2)
\xrightarrow{\text{Savitzky--Golay}}
\widetilde X.
$$

$$
\widetilde X
\xrightarrow{\nabla_t}
\dot{\widetilde X}=(v_{x1},v_{y1},v_{x2},v_{y2}).
$$

`Savitzky–Golay` здесь только сглаживает координаты по номеру наблюдения,
а `np.gradient` вычисляет производную по настоящему, неравномерному `time`.
Центральная разность исходных координат остаётся baseline. Все сравнения
проводятся на одном непрерывном сегменте и на общей допустимой области.


In [ ]:
from scipy.signal import savgol_coeffs

TODO3_SEGMENT_ID = 1
TODO3_POSITION_COLUMNS = tuple(STATE_COLUMNS)
TODO3_VELOCITY_COLUMNS = ("vx1", "vy1", "vx2", "vy2")
SG_POLYORDER = 3
SG_WINDOW_CANDIDATES = tuple(range(5, 32, 2))
SG_DISPLAY_WINDOWS = (5, 11, 21, 31)
SG_EDGE_MODES = ("interp", "mirror", "nearest")

if any(_window % 2 == 0 for _window in SG_WINDOW_CANDIDATES):
    raise ValueError("Все окна Savitzky–Golay должны быть нечётными.")
if any(_window <= SG_POLYORDER for _window in SG_WINDOW_CANDIDATES):
    raise ValueError("Каждое окно должно быть длиннее порядка полинома.")
if not set(SG_DISPLAY_WINDOWS).issubset(SG_WINDOW_CANDIDATES):
    raise ValueError("Окна для подробных графиков должны входить в сетку.")

todo3_segment = segmented_df.loc[
    segmented_df["segment_id"].eq(TODO3_SEGMENT_ID),
    ["time", "segment_id", *TODO3_POSITION_COLUMNS],
].copy()
todo3_segment["segment_row"] = np.arange(len(todo3_segment), dtype=int)

todo3_time = todo3_segment["time"].to_numpy(dtype=float)
todo3_positions_raw = todo3_segment.loc[
    :, TODO3_POSITION_COLUMNS
].to_numpy(dtype=float)
todo3_dt = np.diff(todo3_time)

if len(todo3_segment) <= max(SG_WINDOW_CANDIDATES) + 2:
    raise ValueError("Выбранный сегмент слишком короткий для заданных окон.")
if not np.isfinite(todo3_positions_raw).all() or not np.isfinite(todo3_time).all():
    raise ValueError("В выбранном сегменте есть NaN или бесконечные значения.")
if not np.all(todo3_dt > 0):
    raise ValueError("Время внутри выбранного сегмента не возрастает строго.")

todo3_segment_summary = pd.DataFrame(
    {
        "value": [
            len(todo3_segment),
            todo3_time[0],
            todo3_time[-1],
            todo3_time[-1] - todo3_time[0],
            todo3_dt.min(),
            np.median(todo3_dt),
            todo3_dt.mean(),
            todo3_dt.max(),
            todo3_dt.std(ddof=1) / todo3_dt.mean(),
        ]
    },
    index=[
        "row_count",
        "time_start",
        "time_end",
        "duration",
        "dt_min",
        "dt_median",
        "dt_mean",
        "dt_max",
        "dt_coefficient_of_variation",
    ],
)

display(Markdown(f"### Проверка сегмента `{TODO3_SEGMENT_ID}`"))
display(todo3_segment_summary)
display(
    Markdown(
        f"Проверяются **{len(SG_WINDOW_CANDIDATES)} окон**: "
        f"`{SG_WINDOW_CANDIDATES[0]}, {SG_WINDOW_CANDIDATES[1]}, …, "
        f"{SG_WINDOW_CANDIDATES[-1]}`. На подробных графиках оставлены "
        f"только реперные окна `{SG_DISPLAY_WINDOWS}`, чтобы линии не "
        "перекрывали друг друга."
    )
)


In [ ]:
# Baseline: трёхточечная конечная разность с настоящими значениями time.
# np.gradient использует неравномерную формулу во внутренних точках и
# одностороннюю формулу второго порядка только на двух внешних концах.
todo3_velocity_central_values = np.gradient(
    todo3_positions_raw,
    todo3_time,
    axis=0,
    edge_order=2,
)
todo3_velocity_central = pd.DataFrame(
    todo3_velocity_central_values,
    index=todo3_segment.index,
    columns=TODO3_VELOCITY_COLUMNS,
)


def _todo3_smooth_and_differentiate(window, mode="interp"):
    # Return SG-smoothed positions and their actual-time gradient.
    _positions_sg = savgol_filter(
        todo3_positions_raw,
        window_length=window,
        polyorder=SG_POLYORDER,
        axis=0,
        mode=mode,
    )
    _velocity_sg = np.gradient(
        _positions_sg,
        todo3_time,
        axis=0,
        edge_order=2,
    )
    return _positions_sg, _velocity_sg


todo3_sg_candidates = {}
for _window in SG_WINDOW_CANDIDATES:
    _positions_sg, _velocity_sg = _todo3_smooth_and_differentiate(_window)
    todo3_sg_candidates[_window] = {
        "positions": _positions_sg,
        "velocity": _velocity_sg,
    }

if not np.isfinite(todo3_velocity_central_values).all():
    raise ValueError("Baseline-производная содержит NaN или бесконечность.")
if not all(
    np.isfinite(_result["positions"]).all()
    and np.isfinite(_result["velocity"]).all()
    for _result in todo3_sg_candidates.values()
):
    raise ValueError("Один из SG-кандидатов содержит NaN или бесконечность.")

display(
    Markdown(
        f"Baseline и **{len(SG_WINDOW_CANDIDATES)} SG-кандидатов** "
        "вычислены. Победитель намеренно ещё не назначен."
    )
)


### Математическая граница составного оператора

Пусть окно SG имеет длину \(W=2h+1\). Внутреннее сглаженное значение
\(\widetilde x_i\) зависит от исходных точек \(x_{i-h},\ldots,x_{i+h}\).
Следующий `np.gradient` использует
\(\widetilde x_{i-1},\widetilde x_i,\widetilde x_{i+1}\), поэтому полный
support производной равен

$$
x_{i-h-1},\ldots,x_{i+h+1}.
$$

Чтобы support целиком лежал внутри сегмента, необходимо

$$
h+1\le i\le N-h-2,
\qquad
m(W)=h+1=\frac{W+1}{2}.
$$

Это минимальный отступ, заданный самим алгоритмом. Для честного сравнения
разных окон ниже используется общая маска самого широкого кандидата.

In [ ]:
todo3_boundary_summary_rows = []
for _window in SG_WINDOW_CANDIDATES:
    _radius = (_window - 1) // 2
    _margin = _radius + 1
    _valid_count = len(todo3_segment) - 2 * _margin
    todo3_boundary_summary_rows.append(
        {
            "window": _window,
            "sg_radius_h": _radius,
            "gradient_expansion": 1,
            "valid_margin_m": _margin,
            "left_excluded_time": todo3_time[_margin] - todo3_time[0],
            "right_excluded_time": todo3_time[-1] - todo3_time[-1 - _margin],
            "valid_rows": _valid_count,
            "excluded_rows": 2 * _margin,
            "excluded_percent": 100 * 2 * _margin / len(todo3_segment),
        }
    )

todo3_boundary_summary = pd.DataFrame(todo3_boundary_summary_rows).set_index(
    "window"
)
todo3_comparison_margin = int(
    todo3_boundary_summary["valid_margin_m"].max()
)
todo3_distance_to_edge = np.minimum(
    np.arange(len(todo3_segment)),
    np.arange(len(todo3_segment))[::-1],
)
todo3_comparison_valid = (
    todo3_distance_to_edge >= todo3_comparison_margin
)

display(todo3_boundary_summary.style.format(precision=6))
display(
    Markdown(
        f"Общая область сравнения начинается на расстоянии "
        f"**{todo3_comparison_margin} наблюдений** от каждого края. "
        f"Она содержит **{todo3_comparison_valid.sum()}** из "
        f"**{len(todo3_segment)}** строк."
    )
)

In [ ]:
# Графическая и численная falsification-проверка границы.
# Меняем только искусственное продолжение за краем. Там, где весь support
# находится внутри сегмента, результат обязан совпасть до round-off error.
todo3_edge_proof_window = max(SG_WINDOW_CANDIDATES)
todo3_edge_velocity_by_mode = {}
for _mode in SG_EDGE_MODES:
    _, _velocity = _todo3_smooth_and_differentiate(
        todo3_edge_proof_window,
        mode=_mode,
    )
    todo3_edge_velocity_by_mode[_mode] = _velocity

todo3_edge_velocity_stack = np.stack(
    [todo3_edge_velocity_by_mode[_mode] for _mode in SG_EDGE_MODES],
    axis=0,
)
todo3_edge_component_spread = np.ptp(todo3_edge_velocity_stack, axis=0)
todo3_edge_pointwise_spread = todo3_edge_component_spread.max(axis=1)

todo3_edge_velocity_scale = max(
    1.0,
    float(np.abs(todo3_edge_velocity_stack).max()),
)
todo3_edge_tolerance = (
    100 * np.finfo(float).eps * todo3_edge_velocity_scale
)
todo3_edge_max_valid_spread = float(
    todo3_edge_pointwise_spread[todo3_comparison_valid].max()
)
todo3_edge_max_excluded_spread = float(
    todo3_edge_pointwise_spread[~todo3_comparison_valid].max()
)
todo3_edge_proof_passed = (
    todo3_edge_max_valid_spread <= todo3_edge_tolerance
)

todo3_edge_check = pd.DataFrame(
    {
        "value": [
            todo3_edge_proof_window,
            todo3_comparison_margin,
            todo3_edge_max_excluded_spread,
            todo3_edge_max_valid_spread,
            todo3_edge_tolerance,
            todo3_edge_proof_passed,
        ]
    },
    index=[
        "proof_window",
        "mathematical_margin",
        "max_spread_inside_excluded_zone",
        "max_spread_inside_valid_zone",
        "scale_aware_roundoff_tolerance",
        "boundary_invariance_passed",
    ],
)
display(todo3_edge_check)
display(
    Markdown(
        f"**Итог проверки:** максимум внутри исключаемой зоны — "
        f"`{todo3_edge_max_excluded_spread:.6g}`, внутри допустимой — "
        f"`{todo3_edge_max_valid_spread:.3e}`, scale-aware tolerance — "
        f"`{todo3_edge_tolerance:.3e}`. "
        f"Результат: **{'PASS' if todo3_edge_proof_passed else 'FAIL'}**."
    )
)

if not todo3_edge_proof_passed:
    raise AssertionError(
        "После математической границы производная зависит от edge mode."
    )

In [ ]:
_edge_plot_points = min(64, len(todo3_segment) // 2)
_edge_plot_indices_left = np.arange(_edge_plot_points)
_edge_plot_indices_right = np.arange(
    len(todo3_segment) - _edge_plot_points,
    len(todo3_segment),
)
_mode_colors = dict(zip(SG_EDGE_MODES, plt.cm.tab10.colors))

_edge_figure, _edge_axes = plt.subplots(1, 3, figsize=(18, 4.8))

for _mode in SG_EDGE_MODES:
    _velocity = todo3_edge_velocity_by_mode[_mode]
    _edge_axes[0].plot(
        todo3_time[_edge_plot_indices_left],
        _velocity[_edge_plot_indices_left, 0],
        label=_mode,
        color=_mode_colors[_mode],
        linewidth=1.5,
    )
    _edge_axes[1].plot(
        todo3_time[_edge_plot_indices_right],
        _velocity[_edge_plot_indices_right, 0],
        label=_mode,
        color=_mode_colors[_mode],
        linewidth=1.5,
    )

_edge_axes[0].axvspan(
    todo3_time[0],
    todo3_time[todo3_comparison_margin],
    color="tab:red",
    alpha=0.12,
    label="исключаемая область",
)
_edge_axes[1].axvspan(
    todo3_time[-1 - todo3_comparison_margin],
    todo3_time[-1],
    color="tab:red",
    alpha=0.12,
    label="исключаемая область",
)
_edge_axes[0].set_title("Начало сегмента: $v_{x1}$")
_edge_axes[1].set_title("Конец сегмента: $v_{x1}$")
for _axis in _edge_axes[:2]:
    _axis.set_xlabel("time")
    _axis.set_ylabel("coordinate / time")
    _axis.grid(alpha=0.25)
_edge_axes[0].legend()

_edge_distances = np.arange(_edge_plot_points)
_left_spread = todo3_edge_pointwise_spread[:_edge_plot_points]
_right_spread = todo3_edge_pointwise_spread[::-1][:_edge_plot_points]
_plot_floor = max(todo3_edge_tolerance / 10, np.finfo(float).tiny)
_edge_axes[2].semilogy(
    _edge_distances,
    np.maximum(_left_spread, _plot_floor),
    marker="o",
    markersize=3,
    linewidth=1.2,
    label="левый край",
)
_edge_axes[2].semilogy(
    _edge_distances,
    np.maximum(_right_spread, _plot_floor),
    marker="s",
    markersize=3,
    linewidth=1.2,
    label="правый край",
)
_edge_axes[2].axvline(
    todo3_comparison_margin,
    color="tab:red",
    linestyle="--",
    label=f"m = {todo3_comparison_margin}",
)
_edge_axes[2].axhline(
    todo3_edge_tolerance,
    color="black",
    linestyle=":",
    label="round-off tolerance",
)
_edge_axes[2].set_title("Зависимость от edge mode")
_edge_axes[2].set_xlabel("расстояние до края, наблюдений")
_edge_axes[2].set_ylabel("max spread по 4 компонентам")
_edge_axes[2].grid(alpha=0.25, which="both")
_edge_axes[2].legend()

_edge_figure.suptitle(
    "Проверка математической границы SG → actual-time gradient",
    y=1.02,
)
_edge_figure.tight_layout()
plt.show()

### Плотная сетка окон без назначения победителя

Одна метрика не может доказать истинность производной, потому что настоящие
скорости неизвестны. Поэтому для каждого нечётного окна от 5 до 31
одновременно проверяются:

- изменение координат относительно исходных данных;
- leave-one-out ошибка локального SG-полинома;
- согласие с baseline-производной;
- относительная шероховатость скорости;
- изменение результата при увеличении окна на два наблюдения.

Все величины считаются на общей области с отступом 16 наблюдений. Масштабные
ошибки нормируются отдельно для каждой координаты, чтобы объект или ось с
большим численным диапазоном не получили скрытого преимущества.

Дополнительно строится **локоть компромисса**. Ошибка координат (E(W)) и
шероховатость скорости (R(W)) независимо нормируются в диапазон ([0,1]).
Для каждой точки считается перпендикулярное расстояние до хорды, соединяющей
крайние окна. Максимум расстояния — data-driven точка наибольшего изгиба
trade-off curve. Это ориентир, а не доказательство правильности производной:
результат зависит от выбранных метрик и границ сканируемого диапазона.


In [ ]:
_valid = todo3_comparison_valid
_raw_valid = todo3_positions_raw[_valid]
_central_valid = todo3_velocity_central_values[_valid]
_time_valid = todo3_time[_valid]

_position_iqr = np.subtract(
    *np.percentile(_raw_valid, [75, 25], axis=0)
)
_position_iqr = np.maximum(_position_iqr, np.finfo(float).eps)
_central_rms = np.sqrt(np.mean(_central_valid**2, axis=0))
_central_rms = np.maximum(_central_rms, np.finfo(float).eps)


def _todo3_component_scaled_rmse(_difference, _scale):
    return np.sqrt(np.mean(_difference**2, axis=0)) / _scale


def _todo3_scaled_rmse(_difference, _scale):
    return float(np.mean(_todo3_component_scaled_rmse(_difference, _scale)))


def _todo3_tradeoff_chord_distance(_frame):
    _error = _frame["loo_position_nrmse_iqr"].to_numpy(dtype=float)
    _roughness = _frame["relative_velocity_roughness"].to_numpy(dtype=float)
    _error_span = np.ptp(_error)
    _roughness_span = np.ptp(_roughness)
    if _error_span <= 0 or _roughness_span <= 0:
        raise ValueError("Для поиска локтя нужны непостоянные метрики.")
    _x = (_error - _error.min()) / _error_span
    _y = (_roughness - _roughness.min()) / _roughness_span
    _dx = _x[-1] - _x[0]
    _dy = _y[-1] - _y[0]
    _denominator = np.hypot(_dx, _dy)
    return np.abs(
        _dx * (_y - _y[0]) - _dy * (_x - _x[0])
    ) / _denominator


todo3_window_metric_rows = []
_previous_window = None
for _window in SG_WINDOW_CANDIDATES:
    _positions = todo3_sg_candidates[_window]["positions"][_valid]
    _velocity = todo3_sg_candidates[_window]["velocity"][_valid]
    _position_residual = _raw_valid - _positions

    # Для линейного SG-smoother leave-one-out residual равен
    # обычному residual / (1 - leverage). В общей внутренней области
    # leverage — центральный коэффициент одного и того же фильтра.
    _center_leverage = float(
        savgol_coeffs(
            _window,
            SG_POLYORDER,
            deriv=0,
            use="conv",
        )[_window // 2]
    )
    _loo_residual = _position_residual / (1 - _center_leverage)

    _correlations = [
        np.corrcoef(_central_valid[:, _column], _velocity[:, _column])[0, 1]
        for _column in range(len(TODO3_POSITION_COLUMNS))
    ]
    _velocity_acceleration = (
        np.diff(_velocity, axis=0) / np.diff(_time_valid)[:, None]
    )
    _velocity_rms = np.sqrt(np.mean(_velocity**2, axis=0))
    _velocity_rms = np.maximum(_velocity_rms, np.finfo(float).eps)
    _relative_roughness = float(
        np.mean(
            np.sqrt(np.mean(_velocity_acceleration**2, axis=0))
            * np.median(todo3_dt)
            / _velocity_rms
        )
    )

    if _previous_window is None:
        _delta_previous = np.nan
    else:
        _previous_velocity = todo3_sg_candidates[_previous_window][
            "velocity"
        ][_valid]
        _previous_rms = np.sqrt(
            np.mean(_previous_velocity**2, axis=0)
        )
        _symmetric_velocity_scale = np.maximum(
            0.5 * (_velocity_rms + _previous_rms),
            np.finfo(float).eps,
        )
        _delta_previous = _todo3_scaled_rmse(
            _velocity - _previous_velocity,
            _symmetric_velocity_scale,
        )

    todo3_window_metric_rows.append(
        {
            "window": _window,
            "half_window_time_median": (
                ((_window - 1) // 2) * np.median(todo3_dt)
            ),
            "position_change_nrmse_iqr": _todo3_scaled_rmse(
                _position_residual,
                _position_iqr,
            ),
            "loo_position_nrmse_iqr": _todo3_scaled_rmse(
                _loo_residual,
                _position_iqr,
            ),
            "central_velocity_correlation_mean": float(
                np.mean(_correlations)
            ),
            "central_velocity_difference_nrmse": _todo3_scaled_rmse(
                _velocity - _central_valid,
                _central_rms,
            ),
            "relative_velocity_roughness": _relative_roughness,
            "delta_from_previous_window_nrmse": _delta_previous,
        }
    )
    _previous_window = _window

todo3_window_metrics = pd.DataFrame(todo3_window_metric_rows).set_index(
    "window"
)
todo3_window_metrics["tradeoff_chord_distance"] = (
    _todo3_tradeoff_chord_distance(todo3_window_metrics)
)
todo3_tradeoff_knee_window = int(
    todo3_window_metrics["tradeoff_chord_distance"].idxmax()
)
_knee_position = SG_WINDOW_CANDIDATES.index(todo3_tradeoff_knee_window)
todo3_tradeoff_neighbor_windows = SG_WINDOW_CANDIDATES[
    max(0, _knee_position - 1) : _knee_position + 2
]

display(todo3_window_metrics.style.format(precision=6, na_rep="—"))


In [ ]:
_metric_figure, _metric_axes = plt.subplots(1, 4, figsize=(19, 4.6))
_metric_windows = todo3_window_metrics.index.to_numpy()

_metric_axes[0].plot(
    _metric_windows,
    todo3_window_metrics["loo_position_nrmse_iqr"],
    marker="o",
)
_metric_axes[0].set_title("Leave-one-out ошибка координат")
_metric_axes[0].set_ylabel("NRMSE / IQR")

_metric_axes[1].plot(
    _metric_windows,
    todo3_window_metrics["relative_velocity_roughness"],
    marker="o",
)
_metric_axes[1].set_title("Шероховатость скорости")
_metric_axes[1].set_ylabel("dimensionless RMS")

_metric_axes[2].plot(
    _metric_windows,
    todo3_window_metrics["delta_from_previous_window_nrmse"],
    marker="o",
)
_metric_axes[2].set_title("Изменение при W − 2 → W")
_metric_axes[2].set_ylabel("symmetric velocity NRMSE")

for _axis in _metric_axes[:3]:
    _axis.axvline(
        todo3_tradeoff_knee_window,
        color="tab:red",
        linestyle="--",
        alpha=0.8,
        label=f"локоть W={todo3_tradeoff_knee_window}",
    )
    _axis.set_xlabel("SG window length")
    _axis.set_xticks(_metric_windows[::2])
    _axis.grid(alpha=0.25)
_metric_axes[0].legend()

_metric_axes[3].plot(
    todo3_window_metrics["loo_position_nrmse_iqr"],
    todo3_window_metrics["relative_velocity_roughness"],
    marker="o",
)
_knee_row = todo3_window_metrics.loc[todo3_tradeoff_knee_window]
_metric_axes[3].scatter(
    [_knee_row["loo_position_nrmse_iqr"]],
    [_knee_row["relative_velocity_roughness"]],
    color="tab:red",
    marker="X",
    s=90,
    zorder=3,
    label=f"локоть W={todo3_tradeoff_knee_window}",
)
for _window in (
    SG_WINDOW_CANDIDATES[0],
    todo3_tradeoff_knee_window,
    SG_WINDOW_CANDIDATES[-1],
):
    _row = todo3_window_metrics.loc[_window]
    _metric_axes[3].annotate(
        f"W={_window}",
        (_row["loo_position_nrmse_iqr"], _row["relative_velocity_roughness"]),
        xytext=(5, 5),
        textcoords="offset points",
    )
_metric_axes[3].set_title("Trade-off: координаты ↔ скорость")
_metric_axes[3].set_xlabel("LOO position NRMSE / IQR")
_metric_axes[3].set_ylabel("velocity roughness")
_metric_axes[3].grid(alpha=0.25)
_metric_axes[3].legend()

_metric_figure.suptitle("Плотная сетка нечётных SG-окон 5…31", y=1.02)
_metric_figure.tight_layout()
plt.show()

_minimum_delta = todo3_window_metrics[
    "delta_from_previous_window_nrmse"
].dropna().min()
_minimum_delta_window = int(
    todo3_window_metrics[
        "delta_from_previous_window_nrmse"
    ].idxmin()
)
display(
    Markdown(
        f"**Data-driven локоть:** `W = {todo3_tradeoff_knee_window}`. "
        f"Ближайшие точки сетки `{todo3_tradeoff_neighbor_windows}` — "
        "только локальная область для обсуждения, не доверительный интервал. "
        f"Изменение соседних производных уменьшается до "
        f"`{_minimum_delta:.3f}` при `W = {_minimum_delta_window}`, но "
        "внутри диапазона не выходит на очевидное нулевое плато. Ниже "
        "сравним это изменение с чувствительностью к временной сетке."
    )
)


In [ ]:
_candidate_colors = dict(
    zip(SG_DISPLAY_WINDOWS, plt.cm.viridis(np.linspace(0.08, 0.88, len(SG_DISPLAY_WINDOWS))))
)
_zoom_center = 0.5 * (todo3_time[0] + todo3_time[-1])
_zoom_half_width = 0.025 * (todo3_time[-1] - todo3_time[0])
_zoom_mask = (
    _valid
    & (todo3_time >= _zoom_center - _zoom_half_width)
    & (todo3_time <= _zoom_center + _zoom_half_width)
)

_velocity_figure, _velocity_axes = plt.subplots(
    4,
    2,
    figsize=(16, 13),
    sharex="col",
)

for _row, (_position_name, _velocity_name) in enumerate(
    zip(TODO3_POSITION_COLUMNS, TODO3_VELOCITY_COLUMNS)
):
    _velocity_axes[_row, 0].plot(
        todo3_time[_valid],
        todo3_velocity_central_values[_valid, _row],
        color="0.55",
        alpha=0.55,
        linewidth=0.8,
        label="central raw",
    )
    _velocity_axes[_row, 1].plot(
        todo3_time[_zoom_mask],
        todo3_velocity_central_values[_zoom_mask, _row],
        color="0.55",
        alpha=0.55,
        linewidth=0.9,
        label="central raw",
    )
    for _window in SG_DISPLAY_WINDOWS:
        _velocity = todo3_sg_candidates[_window]["velocity"][:, _row]
        _velocity_axes[_row, 0].plot(
            todo3_time[_valid],
            _velocity[_valid],
            color=_candidate_colors[_window],
            linewidth=1.0,
            label=f"SG {_window}",
        )
        _velocity_axes[_row, 1].plot(
            todo3_time[_zoom_mask],
            _velocity[_zoom_mask],
            color=_candidate_colors[_window],
            linewidth=1.2,
            label=f"SG {_window}",
        )

    _velocity_axes[_row, 0].set_ylabel(
        f"{_velocity_name}, coordinate / time"
    )
    _velocity_axes[_row, 0].grid(alpha=0.2)
    _velocity_axes[_row, 1].grid(alpha=0.2)

_velocity_axes[0, 0].set_title("Весь segment 1")
_velocity_axes[0, 1].set_title("Центральные 5% времени")
_velocity_axes[-1, 0].set_xlabel("time")
_velocity_axes[-1, 1].set_xlabel("time")
_velocity_axes[0, 1].legend(ncol=2)
_velocity_figure.suptitle(
    "Baseline и реперные SG-окна: компоненты скорости",
    y=1.0,
)
_velocity_figure.tight_layout()
plt.show()


In [ ]:
_phase_methods = ["central raw", *[f"SG {_w}" for _w in SG_DISPLAY_WINDOWS]]
_phase_figure, _phase_axes = plt.subplots(
    4,
    len(_phase_methods),
    figsize=(18, 15),
)

for _row, (_position_name, _velocity_name) in enumerate(
    zip(TODO3_POSITION_COLUMNS, TODO3_VELOCITY_COLUMNS)
):
    _position_series = [
        todo3_positions_raw[:, _row],
        *[
            todo3_sg_candidates[_window]["positions"][:, _row]
            for _window in SG_DISPLAY_WINDOWS
        ],
    ]
    _velocity_series = [
        todo3_velocity_central_values[:, _row],
        *[
            todo3_sg_candidates[_window]["velocity"][:, _row]
            for _window in SG_DISPLAY_WINDOWS
        ],
    ]
    _row_x_min = min(_values[_valid].min() for _values in _position_series)
    _row_x_max = max(_values[_valid].max() for _values in _position_series)
    _row_y_min = min(_values[_valid].min() for _values in _velocity_series)
    _row_y_max = max(_values[_valid].max() for _values in _velocity_series)

    for _column, (_method, _positions, _velocity) in enumerate(
        zip(_phase_methods, _position_series, _velocity_series)
    ):
        _axis = _phase_axes[_row, _column]
        _color = "0.55" if _column == 0 else _candidate_colors[
            SG_DISPLAY_WINDOWS[_column - 1]
        ]
        _axis.plot(
            _positions[_valid],
            _velocity[_valid],
            color=_color,
            alpha=0.8,
            linewidth=0.65,
        )
        _axis.set_xlim(_row_x_min, _row_x_max)
        _axis.set_ylim(_row_y_min, _row_y_max)
        _axis.grid(alpha=0.18)
        if _row == 0:
            _axis.set_title(_method)
        if _row == len(TODO3_POSITION_COLUMNS) - 1:
            _axis.set_xlabel(_position_name)
        if _column == 0:
            _axis.set_ylabel(
                f"{_position_name}–{_velocity_name}\n"
                f"{_velocity_name}, coordinate / time"
            )

_phase_figure.suptitle(
    "Предварительные фазовые проекции на общих шкалах",
    y=1.0,
)
_phase_figure.tight_layout()
plt.show()


### Контроль на равномерной временной сетке

Этот контроль не заменяет основной расчёт по реальному `time`. Он
проверяет, не является ли наблюдаемый выбор окна следствием вариации
временного шага.

1. На том же интервале строится равномерная сетка из того же числа точек,
   поэтому её шаг равен среднему исходному `dt`.
2. Каждая исходная координата линейно интерполируется на эту сетку.
   Линейная интерполяция выбрана как прозрачная и не создающая overshoot.
3. На равномерной сетке повторяются SG-сглаживание и производная для всех
   окон 5…31.
4. Результат интерполируется обратно в исходные моменты времени и
   сравнивается с основной actual-time pipeline на общей внутренней
   области.

Разность двух pipelines включает и эффект интерполяции, поэтому это
консервативная оценка чувствительности, а не чистая оценка ошибки
`np.gradient`. Отдельный round-trip координат показывает, насколько
велик сам интерполяционный вклад.


In [ ]:
todo3_uniform_time = np.linspace(
    todo3_time[0],
    todo3_time[-1],
    len(todo3_time),
)
todo3_uniform_dt = float(todo3_uniform_time[1] - todo3_uniform_time[0])
todo3_positions_uniform_raw = np.column_stack(
    [
        np.interp(
            todo3_uniform_time,
            todo3_time,
            todo3_positions_raw[:, _column],
        )
        for _column in range(len(TODO3_POSITION_COLUMNS))
    ]
)

todo3_uniform_valid = np.zeros(len(todo3_uniform_time), dtype=bool)
todo3_uniform_valid[
    todo3_comparison_margin : -todo3_comparison_margin
] = True
todo3_resampling_valid = (
    todo3_comparison_valid
    & (todo3_time >= todo3_uniform_time[todo3_comparison_margin])
    & (todo3_time <= todo3_uniform_time[-1 - todo3_comparison_margin])
)

todo3_positions_uniform_roundtrip = np.column_stack(
    [
        np.interp(
            todo3_time,
            todo3_uniform_time,
            todo3_positions_uniform_raw[:, _column],
        )
        for _column in range(len(TODO3_POSITION_COLUMNS))
    ]
)
todo3_uniform_roundtrip_position_nrmse = _todo3_scaled_rmse(
    (
        todo3_positions_uniform_roundtrip
        - todo3_positions_raw
    )[todo3_resampling_valid],
    _position_iqr,
)

todo3_uniform_candidates = {}
for _window in SG_WINDOW_CANDIDATES:
    _positions_uniform_sg = savgol_filter(
        todo3_positions_uniform_raw,
        window_length=_window,
        polyorder=SG_POLYORDER,
        axis=0,
        mode="interp",
    )
    _velocity_uniform_sg = np.gradient(
        _positions_uniform_sg,
        todo3_uniform_time,
        axis=0,
        edge_order=2,
    )
    _positions_uniform_back = np.column_stack(
        [
            np.interp(
                todo3_time,
                todo3_uniform_time,
                _positions_uniform_sg[:, _column],
            )
            for _column in range(len(TODO3_POSITION_COLUMNS))
        ]
    )
    _velocity_uniform_back = np.column_stack(
        [
            np.interp(
                todo3_time,
                todo3_uniform_time,
                _velocity_uniform_sg[:, _column],
            )
            for _column in range(len(TODO3_POSITION_COLUMNS))
        ]
    )
    todo3_uniform_candidates[_window] = {
        "positions": _positions_uniform_sg,
        "velocity": _velocity_uniform_sg,
        "positions_on_original_time": _positions_uniform_back,
        "velocity_on_original_time": _velocity_uniform_back,
    }

if not all(
    np.isfinite(_array).all()
    for _candidate in todo3_uniform_candidates.values()
    for _array in _candidate.values()
):
    raise ValueError("Контроль на равномерной сетке содержит NaN или inf.")

display(
    Markdown(
        f"Равномерный шаг: `{todo3_uniform_dt:.6f}`; средний "
        f"исходный шаг: `{todo3_dt.mean():.6f}`. Round-trip ошибка "
        f"координат: `{todo3_uniform_roundtrip_position_nrmse:.6f}` "
        "NRMSE / IQR."
    )
)


In [ ]:
_uniform_raw_valid = todo3_positions_uniform_raw[todo3_uniform_valid]
_uniform_time_valid = todo3_uniform_time[todo3_uniform_valid]
_uniform_position_iqr = np.maximum(
    np.subtract(
        *np.percentile(_uniform_raw_valid, [75, 25], axis=0)
    ),
    np.finfo(float).eps,
)

todo3_uniform_metric_rows = []
todo3_resampling_metric_rows = []
_previous_uniform_velocity = None

for _window in SG_WINDOW_CANDIDATES:
    _uniform_candidate = todo3_uniform_candidates[_window]
    _uniform_positions = _uniform_candidate["positions"][todo3_uniform_valid]
    _uniform_velocity = _uniform_candidate["velocity"][todo3_uniform_valid]
    _uniform_position_residual = _uniform_raw_valid - _uniform_positions
    _center_leverage = float(
        savgol_coeffs(
            _window,
            SG_POLYORDER,
            deriv=0,
            use="conv",
        )[_window // 2]
    )
    _uniform_loo_residual = (
        _uniform_position_residual / (1 - _center_leverage)
    )
    _uniform_velocity_rms = np.maximum(
        np.sqrt(np.mean(_uniform_velocity**2, axis=0)),
        np.finfo(float).eps,
    )
    _uniform_acceleration = (
        np.diff(_uniform_velocity, axis=0) / todo3_uniform_dt
    )
    _uniform_roughness = float(
        np.mean(
            np.sqrt(np.mean(_uniform_acceleration**2, axis=0))
            * todo3_uniform_dt
            / _uniform_velocity_rms
        )
    )
    if _previous_uniform_velocity is None:
        _uniform_delta_previous = np.nan
    else:
        _previous_uniform_rms = np.sqrt(
            np.mean(_previous_uniform_velocity**2, axis=0)
        )
        _uniform_symmetric_scale = np.maximum(
            0.5 * (_uniform_velocity_rms + _previous_uniform_rms),
            np.finfo(float).eps,
        )
        _uniform_delta_previous = _todo3_scaled_rmse(
            _uniform_velocity - _previous_uniform_velocity,
            _uniform_symmetric_scale,
        )

    todo3_uniform_metric_rows.append(
        {
            "window": _window,
            "loo_position_nrmse_iqr": _todo3_scaled_rmse(
                _uniform_loo_residual,
                _uniform_position_iqr,
            ),
            "relative_velocity_roughness": _uniform_roughness,
            "delta_from_previous_window_nrmse": _uniform_delta_previous,
        }
    )
    _previous_uniform_velocity = _uniform_velocity

    _actual_positions = todo3_sg_candidates[_window]["positions"][todo3_resampling_valid]
    _actual_velocity = todo3_sg_candidates[_window]["velocity"][todo3_resampling_valid]
    _uniform_positions_back = _uniform_candidate[
        "positions_on_original_time"
    ][todo3_resampling_valid]
    _uniform_velocity_back = _uniform_candidate[
        "velocity_on_original_time"
    ][todo3_resampling_valid]
    _actual_velocity_rms = np.maximum(
        np.sqrt(np.mean(_actual_velocity**2, axis=0)),
        np.finfo(float).eps,
    )
    _velocity_grid_difference_by_component = _todo3_component_scaled_rmse(
        _uniform_velocity_back - _actual_velocity,
        _actual_velocity_rms,
    )
    _velocity_grid_correlations = np.array(
        [
            np.corrcoef(
                _actual_velocity[:, _column],
                _uniform_velocity_back[:, _column],
            )[0, 1]
            for _column in range(len(TODO3_POSITION_COLUMNS))
        ]
    )

    todo3_resampling_metric_rows.append(
        {
            "window": _window,
            "position_grid_difference_nrmse_iqr": _todo3_scaled_rmse(
                _uniform_positions_back - _actual_positions,
                _position_iqr,
            ),
            "velocity_grid_difference_nrmse_mean": float(
                _velocity_grid_difference_by_component.mean()
            ),
            "velocity_grid_difference_nrmse_max": float(
                _velocity_grid_difference_by_component.max()
            ),
            "velocity_grid_correlation_mean": float(
                _velocity_grid_correlations.mean()
            ),
            "velocity_grid_correlation_min": float(
                _velocity_grid_correlations.min()
            ),
        }
    )

todo3_uniform_window_metrics = pd.DataFrame(
    todo3_uniform_metric_rows
).set_index("window")
todo3_uniform_window_metrics["tradeoff_chord_distance"] = (
    _todo3_tradeoff_chord_distance(todo3_uniform_window_metrics)
)
todo3_uniform_tradeoff_knee_window = int(
    todo3_uniform_window_metrics[
        "tradeoff_chord_distance"
    ].idxmax()
)

todo3_resampling_metrics = pd.DataFrame(
    todo3_resampling_metric_rows
).set_index("window")
todo3_resampling_metrics[
    "window_change_to_grid_sensitivity_ratio"
] = (
    todo3_window_metrics["delta_from_previous_window_nrmse"]
    / todo3_resampling_metrics[
        "velocity_grid_difference_nrmse_mean"
    ]
)
todo3_resampling_metrics["plateau_at_grid_sensitivity_scale"] = (
    todo3_resampling_metrics[
        "window_change_to_grid_sensitivity_ratio"
    ]
    <= 1
)

display(Markdown("#### Метрики на равномерной сетке"))
display(todo3_uniform_window_metrics.style.format(precision=6, na_rep="—"))
display(Markdown("#### Чувствительность actual-time pipeline к resampling"))
display(todo3_resampling_metrics.style.format(precision=6, na_rep="—"))


In [ ]:
_grid_figure, _grid_axes = plt.subplots(2, 3, figsize=(18, 9))
_uniform_metric_windows = todo3_uniform_window_metrics.index.to_numpy()

for _axis, _column, _title, _ylabel in [
    (
        _grid_axes[0, 0],
        "loo_position_nrmse_iqr",
        "LOO ошибка координат",
        "NRMSE / IQR",
    ),
    (
        _grid_axes[0, 1],
        "relative_velocity_roughness",
        "Шероховатость скорости",
        "dimensionless RMS",
    ),
    (
        _grid_axes[0, 2],
        "delta_from_previous_window_nrmse",
        "Изменение при W − 2 → W",
        "symmetric velocity NRMSE",
    ),
]:
    _axis.plot(
        _metric_windows,
        todo3_window_metrics[_column],
        marker="o",
        label="actual-time pipeline",
    )
    _axis.plot(
        _uniform_metric_windows,
        todo3_uniform_window_metrics[_column],
        marker="s",
        label="uniform-grid control",
    )
    _axis.set_title(_title)
    _axis.set_ylabel(_ylabel)
    _axis.legend()

_grid_axes[1, 0].plot(
    _metric_windows,
    todo3_resampling_metrics[
        "position_grid_difference_nrmse_iqr"
    ],
    marker="o",
    label="между pipelines",
)
_grid_axes[1, 0].axhline(
    todo3_uniform_roundtrip_position_nrmse,
    color="0.35",
    linestyle="--",
    label="raw round-trip",
)
_grid_axes[1, 0].set_title("Чувствительность координат к сетке")
_grid_axes[1, 0].set_ylabel("NRMSE / IQR")
_grid_axes[1, 0].legend()

_grid_axes[1, 1].plot(
    _metric_windows,
    todo3_resampling_metrics[
        "velocity_grid_difference_nrmse_mean"
    ],
    marker="o",
    label="среднее по компонентам",
)
_grid_axes[1, 1].plot(
    _metric_windows,
    todo3_resampling_metrics[
        "velocity_grid_difference_nrmse_max"
    ],
    marker="s",
    label="максимум по компонентам",
)
_grid_axes[1, 1].set_title("Чувствительность скорости к сетке")
_grid_axes[1, 1].set_ylabel("velocity NRMSE")
_grid_axes[1, 1].legend()

_grid_axes[1, 2].plot(
    _metric_windows,
    todo3_resampling_metrics["velocity_grid_correlation_mean"],
    marker="o",
    label="среднее по компонентам",
)
_grid_axes[1, 2].plot(
    _metric_windows,
    todo3_resampling_metrics["velocity_grid_correlation_min"],
    marker="s",
    label="минимум по компонентам",
)
_grid_axes[1, 2].set_title("Корреляция скоростей двух pipelines")
_grid_axes[1, 2].set_ylabel("Pearson correlation")
_grid_axes[1, 2].legend()

for _axis in _grid_axes.flat:
    _axis.set_xlabel("SG window length")
    _axis.set_xticks(_metric_windows[::2])
    _axis.grid(alpha=0.25)

_grid_figure.suptitle(
    "Actual-time pipeline и контроль на равномерной сетке",
    y=1.01,
)
_grid_figure.tight_layout()
plt.show()

_comparison_window = todo3_tradeoff_knee_window
_comparison_row = todo3_resampling_metrics.loc[_comparison_window]
_finite_sensitivity_ratios = todo3_resampling_metrics[
    "window_change_to_grid_sensitivity_ratio"
].dropna()
display(
    Markdown(
        f"**Локоть совпал:** actual-time `W = "
        f"{todo3_tradeoff_knee_window}`, uniform-grid `W = "
        f"{todo3_uniform_tradeoff_knee_window}`. При `W = "
        f"{_comparison_window}` средняя разница скоростей двух "
        f"pipelines равна `{_comparison_row['velocity_grid_difference_nrmse_mean']:.3f}` "
        f"NRMSE, минимальная корреляция компоненты — "
        f"`{_comparison_row['velocity_grid_correlation_min']:.4f}`. "
        "Для консервативной проверки плато сравниваем изменение "
        "`W − 2 → W` с чувствительностью к сетке того же окна. Во "
        "всём диапазоне отношение больше единицы: от "
        f"`{_finite_sensitivity_ratios.min():.2f}` до "
        f"`{_finite_sensitivity_ratios.max():.2f}`. Поэтому ни одно "
        "окно 5…31 нельзя назвать стабильным на уровне grid-sensitivity."
    )
)


In [ ]:
_uniform_overlay_figure, _uniform_overlay_axes = plt.subplots(
    2,
    2,
    figsize=(15, 8),
    sharex=True,
    sharey=True,
)

for _axis, _window in zip(
    _uniform_overlay_axes.flat,
    SG_DISPLAY_WINDOWS,
):
    _actual_velocity = todo3_sg_candidates[_window]["velocity"][:, 0]
    _uniform_velocity_back = todo3_uniform_candidates[_window][
        "velocity_on_original_time"
    ][:, 0]
    _axis.plot(
        todo3_time[_zoom_mask],
        _actual_velocity[_zoom_mask],
        linewidth=1.4,
        label="actual time",
    )
    _axis.plot(
        todo3_time[_zoom_mask],
        _uniform_velocity_back[_zoom_mask],
        linewidth=1.2,
        linestyle="--",
        label="uniform → original time",
    )
    _axis.set_title(f"SG window = {_window}")
    _axis.set_xlabel("time")
    _axis.set_ylabel("vx1, coordinate / time")
    _axis.grid(alpha=0.25)

_uniform_overlay_axes[0, 0].legend()
_uniform_overlay_figure.suptitle(
    "Локальное сравнение временных сеток: центральные 5% segment 1",
    y=1.01,
)
_uniform_overlay_figure.tight_layout()
plt.show()


### TODO 3.A.5 — переносимость окна между сегментами

Окно `W = 11` уже выбрано на `segment 1`; следующий блок не подбирает его
заново и не назначает разные окна отдельным сегментам. Это exploratory
portability audit: на каждом из 11 сегментов независимо повторяется полная
trade-off curve для окон 5…31, а локальная чувствительность подробно
сравнивается для соседних значений `W = 9, 11, 13`. Максимум расстояния до
хорды — детерминированная эвристика на этой сетке, а не статистически
единственный истинный локоть.

Все окна внутри сегмента сравниваются на одной row-index области с отступом
для максимального `W = 31`. Одинаковое окно означает одинаковое число samples,
но не строго одинаковый физический интервал времени; он показан отдельно.
LOO-метрика относится только к координатному SG-smoother, не к производной и
не к будущей SINDy-модели.

Поскольку group split ещё не зафиксирован, этот all-segment audit не является
слепой model validation. В TODO 4 выбор preprocessing нужно повторить только
на train-сегментах, `W = 9/11/13` сравнить на validation и не менять его по
результатам test.


In [ ]:
TODO3_CROSS_SEGMENT_REFERENCE_WINDOW = todo3_tradeoff_knee_window
TODO3_LOCAL_SENSITIVITY_WINDOWS = (9, 11, 13)

if TODO3_CROSS_SEGMENT_REFERENCE_WINDOW != todo3_uniform_tradeoff_knee_window:
    raise AssertionError(
        "Reference window должно совпадать для двух временных pipelines."
    )
if TODO3_LOCAL_SENSITIVITY_WINDOWS != (
    TODO3_CROSS_SEGMENT_REFERENCE_WINDOW - 2,
    TODO3_CROSS_SEGMENT_REFERENCE_WINDOW,
    TODO3_CROSS_SEGMENT_REFERENCE_WINDOW + 2,
):
    raise AssertionError(
        "Локальная sensitivity-сетка должна окружать reference window."
    )

_cross_segment_margin = (max(SG_WINDOW_CANDIDATES) + 1) // 2
if _cross_segment_margin != int(
    todo3_boundary_summary["valid_margin_m"].max()
):
    raise AssertionError("Не совпала граница общей row-index mask.")
_cross_segment_rows = []
_cross_segment_metric_frames = []

for _segment_id, _segment in segmented_df.groupby(
    "segment_id",
    sort=True,
):
    _segment_time = _segment["time"].to_numpy(dtype=float)
    _segment_positions = _segment.loc[
        :,
        list(TODO3_POSITION_COLUMNS),
    ].to_numpy(dtype=float)
    _segment_size = len(_segment)
    _segment_dt_median = float(np.median(np.diff(_segment_time)))
    _segment_distance_to_edge = np.minimum(
        np.arange(_segment_size),
        np.arange(_segment_size)[::-1],
    )
    _segment_valid = (
        _segment_distance_to_edge >= _cross_segment_margin
    )

    if not np.all(np.diff(_segment_time) > 0):
        raise AssertionError(
            f"time не возрастает строго в segment_id={_segment_id}."
        )
    if not np.isfinite(_segment_positions).all():
        raise AssertionError(
            f"Координаты не конечны в segment_id={_segment_id}."
        )
    if _segment_size <= max(SG_WINDOW_CANDIDATES) + 2:
        raise AssertionError(
            f"segment_id={_segment_id} слишком короткий для SG-сетки."
        )
    if _segment_valid.sum() < 2:
        raise AssertionError(
            f"segment_id={_segment_id} слишком короткий для audit."
        )

    _raw_valid = _segment_positions[_segment_valid]
    _time_valid = _segment_time[_segment_valid]
    _position_iqr = np.maximum(
        np.subtract(
            *np.percentile(_raw_valid, [75, 25], axis=0)
        ),
        np.finfo(float).eps,
    )
    _segment_metric_rows = []
    _local_velocity_candidates = {}

    for _window in SG_WINDOW_CANDIDATES:
        _positions = savgol_filter(
            _segment_positions,
            window_length=_window,
            polyorder=SG_POLYORDER,
            axis=0,
            mode="interp",
        )
        _velocity = np.gradient(
            _positions,
            _segment_time,
            axis=0,
            edge_order=2,
        )
        _positions_valid = _positions[_segment_valid]
        _velocity_valid = _velocity[_segment_valid]
        _position_residual = _raw_valid - _positions_valid
        _center_leverage = float(
            savgol_coeffs(
                _window,
                SG_POLYORDER,
                deriv=0,
                use="conv",
            )[_window // 2]
        )
        _loo_residual = _position_residual / (1 - _center_leverage)
        _acceleration = (
            np.diff(_velocity_valid, axis=0)
            / np.diff(_time_valid)[:, None]
        )
        _velocity_rms = np.maximum(
            np.sqrt(np.mean(_velocity_valid**2, axis=0)),
            np.finfo(float).eps,
        )
        _relative_roughness = float(
            np.mean(
                np.sqrt(np.mean(_acceleration**2, axis=0))
                * _segment_dt_median
                / _velocity_rms
            )
        )
        _segment_metric_rows.append(
            {
                "segment_id": int(_segment_id),
                "window": _window,
                "row_count": _segment_size,
                "valid_rows": int(_segment_valid.sum()),
                "loo_position_nrmse_iqr": _todo3_scaled_rmse(
                    _loo_residual,
                    _position_iqr,
                ),
                "relative_velocity_roughness": _relative_roughness,
            }
        )
        if _window in TODO3_LOCAL_SENSITIVITY_WINDOWS:
            _local_velocity_candidates[_window] = _velocity_valid

    _segment_metrics = pd.DataFrame(_segment_metric_rows).set_index(
        "window"
    )
    _segment_metrics["tradeoff_chord_distance"] = (
        _todo3_tradeoff_chord_distance(_segment_metrics)
    )
    _segment_metrics = _segment_metrics.reset_index()
    _cross_segment_metric_frames.append(_segment_metrics)

    _chord_ranking = _segment_metrics.sort_values(
        "tradeoff_chord_distance",
        ascending=False,
    ).reset_index(drop=True)
    _knee_window = int(_chord_ranking.loc[0, "window"])
    _runner_up_window = int(_chord_ranking.loc[1, "window"])
    _chord_distance_gap = float(
        _chord_ranking.loc[0, "tradeoff_chord_distance"]
        - _chord_ranking.loc[1, "tradeoff_chord_distance"]
    )
    _reference_metrics = _segment_metrics.loc[
        _segment_metrics["window"].eq(
            TODO3_CROSS_SEGMENT_REFERENCE_WINDOW
        )
    ].iloc[0]
    _audit_row = {
        "segment_id": int(_segment_id),
        "row_count": _segment_size,
        "valid_rows": int(_segment_valid.sum()),
        "tradeoff_knee_window": _knee_window,
        "runner_up_window": _runner_up_window,
        "chord_distance_gap": _chord_distance_gap,
        "reference_is_knee": (
            _knee_window == TODO3_CROSS_SEGMENT_REFERENCE_WINDOW
        ),
        "w11_loo_position_nrmse_iqr": _reference_metrics[
            "loo_position_nrmse_iqr"
        ],
        "w11_relative_velocity_roughness": _reference_metrics[
            "relative_velocity_roughness"
        ],
        "dt_median": _segment_dt_median,
        "w11_sg_half_window_time": (
            (TODO3_CROSS_SEGMENT_REFERENCE_WINDOW - 1)
            / 2
            * _segment_dt_median
        ),
        "w11_composite_support_time": (
            (TODO3_CROSS_SEGMENT_REFERENCE_WINDOW + 1)
            / 2
            * _segment_dt_median
        ),
    }

    for _left_window, _right_window in zip(
        TODO3_LOCAL_SENSITIVITY_WINDOWS[:-1],
        TODO3_LOCAL_SENSITIVITY_WINDOWS[1:],
    ):
        _left_velocity = _local_velocity_candidates[_left_window]
        _right_velocity = _local_velocity_candidates[_right_window]
        _left_rms = np.sqrt(np.mean(_left_velocity**2, axis=0))
        _right_rms = np.sqrt(np.mean(_right_velocity**2, axis=0))
        _symmetric_scale = np.maximum(
            0.5 * (_left_rms + _right_rms),
            np.finfo(float).eps,
        )
        _audit_row[
            f"velocity_delta_{_left_window}_{_right_window}_nrmse"
        ] = _todo3_scaled_rmse(
            _right_velocity - _left_velocity,
            _symmetric_scale,
        )

    _cross_segment_rows.append(_audit_row)

todo3_cross_segment_window_metrics = pd.concat(
    _cross_segment_metric_frames,
    ignore_index=True,
).set_index(["segment_id", "window"])
todo3_cross_segment_summary = pd.DataFrame(
    _cross_segment_rows
).sort_values("segment_id", ignore_index=True)

_knee_counts = (
    todo3_cross_segment_summary["tradeoff_knee_window"]
    .value_counts()
    .sort_index()
)
_dominant_knee = int(_knee_counts.idxmax())
_reference_support_count = int(
    todo3_cross_segment_summary["reference_is_knee"].sum()
)
_reference_support_fraction = (
    _reference_support_count / len(todo3_cross_segment_summary)
)
_reference_row_support_fraction = (
    todo3_cross_segment_summary.loc[
        todo3_cross_segment_summary["reference_is_knee"],
        "valid_rows",
    ].sum()
    / todo3_cross_segment_summary["valid_rows"].sum()
)
_sensitive_segment_ids = todo3_cross_segment_summary.loc[
    ~todo3_cross_segment_summary["reference_is_knee"],
    "segment_id",
].tolist()
_alternative_knees = sorted(
    set(
        todo3_cross_segment_summary.loc[
            ~todo3_cross_segment_summary["reference_is_knee"],
            "tradeoff_knee_window",
        ]
    )
)

if not np.isfinite(
    todo3_cross_segment_window_metrics.to_numpy(dtype=float)
).all():
    raise AssertionError("Cross-segment audit содержит NaN или inf.")
if _dominant_knee != TODO3_CROSS_SEGMENT_REFERENCE_WINDOW:
    raise AssertionError(
        "W=11 не является самым частым локтем между сегментами."
    )
display(
    todo3_cross_segment_summary.style.format(
        {
            "w11_loo_position_nrmse_iqr": "{:.6f}",
            "w11_relative_velocity_roughness": "{:.6f}",
            "velocity_delta_9_11_nrmse": "{:.6f}",
            "velocity_delta_11_13_nrmse": "{:.6f}",
            "chord_distance_gap": "{:.6f}",
            "dt_median": "{:.6f}",
            "w11_sg_half_window_time": "{:.6f}",
            "w11_composite_support_time": "{:.6f}",
        }
    )
)
display(
    _knee_counts.rename("segment_count").to_frame()
)


In [ ]:
_cross_figure, _cross_axes_grid = plt.subplots(
    2,
    2,
    figsize=(16, 10),
)
_cross_axes = _cross_axes_grid.ravel()
_cross_ids = todo3_cross_segment_summary["segment_id"]
_cross_colors = np.where(
    todo3_cross_segment_summary["reference_is_knee"],
    "tab:blue",
    "tab:orange",
)

_cross_axes[0].scatter(
    _cross_ids,
    todo3_cross_segment_summary["tradeoff_knee_window"],
    color=_cross_colors,
    s=55,
)
_cross_axes[0].axhline(
    TODO3_CROSS_SEGMENT_REFERENCE_WINDOW,
    color="tab:red",
    linestyle="--",
    label="reference W=11",
)
_cross_axes[0].set_title("Максимум chord-distance каждого сегмента")
_cross_axes[0].set_ylabel("SG window length")
_cross_axes[0].legend()

_cross_axes[1].bar(
    _cross_ids,
    todo3_cross_segment_summary["w11_loo_position_nrmse_iqr"],
    color=_cross_colors,
)
_cross_axes[1].axhline(
    todo3_cross_segment_summary[
        "w11_loo_position_nrmse_iqr"
    ].median(),
    color="0.25",
    linestyle=":",
    label="median",
)
_cross_axes[1].set_title("Сохранение координат при W=11")
_cross_axes[1].set_ylabel("LOO NRMSE / IQR")
_cross_axes[1].legend()

_cross_axes[2].plot(
    _cross_ids,
    todo3_cross_segment_summary[
        "velocity_delta_9_11_nrmse"
    ],
    marker="o",
    label="W 9 → 11",
)
_cross_axes[2].plot(
    _cross_ids,
    todo3_cross_segment_summary[
        "velocity_delta_11_13_nrmse"
    ],
    marker="s",
    label="W 11 → 13",
)
_cross_axes[2].set_title("Локальная чувствительность скорости")
_cross_axes[2].set_ylabel("symmetric velocity NRMSE")
_cross_axes[2].legend()

for _segment_id, _metrics in (
    todo3_cross_segment_window_metrics.reset_index()
    .groupby("segment_id", sort=True)
):
    _cross_axes[3].plot(
        _metrics["window"],
        _metrics["tradeoff_chord_distance"],
        marker="o",
        markersize=2.5,
        linewidth=0.9,
        alpha=0.75,
        label=f"segment {_segment_id}",
    )
_cross_axes[3].set_title("Полные chord-distance curves")
_cross_axes[3].set_xlabel("SG window length")
_cross_axes[3].set_ylabel("normalized chord distance")
_cross_axes[3].set_xticks(SG_WINDOW_CANDIDATES[::2])
_cross_axes[3].grid(alpha=0.25)
_cross_axes[3].legend(ncols=2, fontsize=8)

for _axis in _cross_axes[:3]:
    _axis.set_xlabel("segment_id")
    _axis.set_xticks(_cross_ids)
    _axis.grid(alpha=0.25)

_cross_figure.suptitle(
    "Переносимость единого SG-окна между сегментами",
    y=1.02,
)
_cross_figure.tight_layout()
plt.show()

_sensitive_text = ", ".join(map(str, _sensitive_segment_ids))
_alternative_text = ", ".join(map(str, _alternative_knees))
_w11_loo_median = todo3_cross_segment_summary[
    "w11_loo_position_nrmse_iqr"
].median()
_w11_loo_max_row = todo3_cross_segment_summary.loc[
    todo3_cross_segment_summary[
        "w11_loo_position_nrmse_iqr"
    ].idxmax()
]
_physical_half_window_range = todo3_cross_segment_summary[
    "w11_sg_half_window_time"
].agg(["min", "max"])
_physical_support_range = todo3_cross_segment_summary[
    "w11_composite_support_time"
].agg(["min", "max"])
if np.isclose(
    _physical_half_window_range["min"],
    _physical_half_window_range["max"],
) and np.isclose(
    _physical_support_range["min"],
    _physical_support_range["max"],
):
    _physical_window_text = (
        f"Физическое SG-полуокно по median(dt) одинаково на всех "
        f"сегментах: `{_physical_half_window_range['min']:.3f}`, "
        f"а составной support SG → gradient равен "
        f"`{_physical_support_range['min']:.3f}` единицы времени."
    )
else:
    _physical_window_text = (
        f"Физическое SG-полуокно меняется от "
        f"`{_physical_half_window_range['min']:.3f}` до "
        f"`{_physical_half_window_range['max']:.3f}`, а составной "
        f"support SG → gradient — от "
        f"`{_physical_support_range['min']:.3f}` до "
        f"`{_physical_support_range['max']:.3f}` единицы времени."
    )
_local_velocity_delta_columns = [
    "velocity_delta_9_11_nrmse",
    "velocity_delta_11_13_nrmse",
]
_local_velocity_delta_values = todo3_cross_segment_summary.loc[
    :,
    _local_velocity_delta_columns,
].to_numpy(dtype=float)
_smallest_chord_gap_row = todo3_cross_segment_summary.loc[
    todo3_cross_segment_summary["chord_distance_gap"].idxmin()
]
display(
    Markdown(
        f"**Cross-segment результат:** `W = 11` даёт максимальную "
        "chord-distance на "
        f"**{_reference_support_count} из "
        f"{len(todo3_cross_segment_summary)} сегментов** "
        f"при равном весе (`{100 * _reference_support_fraction:.1f}%`) "
        f"и на `{100 * _reference_row_support_fraction:.1f}%` валидных "
        f"строк. На сегментах `{_sensitive_text}` максимум равен "
        f"`{_alternative_text}`. Поэтому `W = 11` сохраняется как "
        "единое доминирующее по этой эвристике решение, а `W = 9` и "
        "`W = 13` — как симметричные sensitivity-альтернативы. "
        f"Их различие со скоростями `W = 11` лежит в диапазоне "
        f"`{_local_velocity_delta_values.min():.3f}–"
        f"{_local_velocity_delta_values.max():.3f}` NRMSE и не является "
        "малым. "
        f"Медианная LOO-ошибка координат при `W = 11` равна "
        f"`{_w11_loo_median:.6f}` NRMSE / IQR. Максимум наблюдается "
        f"на коротком segment "
        f"`{int(_w11_loo_max_row['segment_id'])}`: "
        f"`{_w11_loo_max_row['w11_loo_position_nrmse_iqr']:.6f}`; "
        "его нужно отдельно контролировать при validation и rollout. "
        f"{_physical_window_text} "
        f"Минимальный отрыв победителя от второго кандидата равен "
        f"`{_smallest_chord_gap_row['chord_distance_gap']:.6f}` на "
        f"segment `{int(_smallest_chord_gap_row['segment_id'])}`, что "
        "дополнительно подчёркивает эвристический характер выбора."
    )
)


### Вывод по выбору окна

**Рабочее решение:** для дальнейшего расчёта скоростей выбирается
Savitzky–Golay с `window_length = 11` и `polyorder = 3`.

Выбор опирается на несколько согласованных наблюдений:

- data-driven локоть trade-off curve находится при `W = 11` как для
  основной actual-time pipeline, так и после перехода на равномерную
  временную сетку;
- LOO-ошибка координат для основной сетки остаётся небольшой:
  `0.029221 NRMSE / IQR`;
- при `W = 11` средняя разница скоростей между двумя временными сетками
  равна `0.092036 NRMSE`, а минимальная корреляция отдельной компоненты —
  `0.995818`;
- exploratory cross-segment audit дал `W = 11` максимальную chord-distance
  на 8 из 11 сегментов и примерно 82% валидных строк; на сегментах 2, 5
  и 7 максимум равен `W = 9`, а `W = 9/13` сохраняются как симметричные
  sensitivity-альтернативы;
- более широкие окна продолжают снижать шероховатость, но одновременно
  сильнее подавляют амплитуду скорости и не образуют убедительного плато.

Поэтому `W = 11` принимается как воспроизводимый data-driven компромисс
между сохранением координат и подавлением шума, а не как истинный
физический масштаб системы: окно задано в samples, а его ширина по времени
слегка различается между сегментами. Основной pipeline остаётся прежним:
SG-сглаживание координат по номеру наблюдения, затем `np.gradient` по
настоящему `time`; равномерная сетка используется только как sensitivity
control.

Для `W = 11` математический отступ составного оператора равен
`m = (W + 1) / 2 = 6` наблюдений с каждого края сегмента. Полуокно
соответствует примерно `0.235` единицы времени; на segment 1 исключаются
12 из 7742 строк, то есть около `0.155%`.

Выбор теперь зафиксирован в параметре `SELECTED_SG_WINDOW`. Ниже единое
окно применяется ко всем сегментам и строится итоговый `kinematics_df`;
для будущей validation отдельно сохраняется проверка `W = 9/13`.


### TODO 3.B — итоговая кинематическая таблица

Для обучения SINDy состояние X будет состоять только из четырёх
SG-сглаженных координат. Производная X_dot вычисляется из той же
сглаженной траектории по настоящему time.

Исходные координаты сохраняются рядом как независимый шумный ориентир:
по ним позже можно проверить, не потеряла ли модель наблюдаемую динамику.
Скорости, центр масс и относительное движение служат диагностикой и не
расширяют состояние модели.

При окне W = 11 итоговая граница составного оператора равна
m = (W + 1) / 2 = 6 строк с каждого края каждого сегмента. На этих
строках сглаженные координаты и скорости оставляются пропусками, а
is_kinematics_valid принимает значение False.

In [ ]:
SELECTED_SG_WINDOW = 11
KINEMATICS_POLYORDER = SG_POLYORDER
KINEMATICS_MARGIN = (SELECTED_SG_WINDOW + 1) // 2

KINEMATICS_RAW_POSITION_COLUMNS = tuple(STATE_COLUMNS)
KINEMATICS_MODEL_POSITION_COLUMNS = tuple(
    f"{column}_sg" for column in KINEMATICS_RAW_POSITION_COLUMNS
)
KINEMATICS_VELOCITY_COLUMNS = tuple(TODO3_VELOCITY_COLUMNS)
KINEMATICS_MODEL_COLUMNS = (
    *KINEMATICS_MODEL_POSITION_COLUMNS,
    *KINEMATICS_VELOCITY_COLUMNS,
)

if SELECTED_SG_WINDOW != todo3_tradeoff_knee_window:
    raise AssertionError(
        "Выбранное окно не совпало с локтем actual-time pipeline."
    )
if SELECTED_SG_WINDOW != todo3_uniform_tradeoff_knee_window:
    raise AssertionError(
        "Выбранное окно не совпало с локтем uniform-grid control."
    )

kinematics_df = segmented_df.loc[
    :,
    ["segment_id", "time", *KINEMATICS_RAW_POSITION_COLUMNS],
].copy()
kinematics_df.insert(
    1,
    "segment_row",
    kinematics_df.groupby("segment_id", sort=False).cumcount(),
)

for column in KINEMATICS_MODEL_COLUMNS:
    kinematics_df[column] = np.nan
kinematics_df["is_kinematics_valid"] = False

_kinematics_segment_rows = []
for _segment_id, _segment in kinematics_df.groupby(
    "segment_id",
    sort=True,
):
    _segment_index = _segment.index
    _time = _segment["time"].to_numpy(dtype=float)
    _raw_positions = _segment.loc[
        :,
        list(KINEMATICS_RAW_POSITION_COLUMNS),
    ].to_numpy(dtype=float)
    _segment_size = len(_segment)

    if not np.all(np.diff(_time) > 0):
        raise AssertionError(
            f"time не возрастает строго в segment_id={_segment_id}."
        )
    if _segment_size <= 2 * KINEMATICS_MARGIN:
        raise AssertionError(
            f"segment_id={_segment_id} слишком короткий для маски."
        )

    _smoothed_positions = savgol_filter(
        _raw_positions,
        window_length=SELECTED_SG_WINDOW,
        polyorder=KINEMATICS_POLYORDER,
        axis=0,
        mode="interp",
    )
    _velocities = np.gradient(
        _smoothed_positions,
        _time,
        axis=0,
        edge_order=2,
    )

    _distance_to_edge = np.minimum(
        np.arange(_segment_size),
        np.arange(_segment_size)[::-1],
    )
    _valid_local = _distance_to_edge >= KINEMATICS_MARGIN
    _valid_index = _segment_index[_valid_local]

    kinematics_df.loc[
        _valid_index,
        list(KINEMATICS_MODEL_POSITION_COLUMNS),
    ] = _smoothed_positions[_valid_local]
    kinematics_df.loc[
        _valid_index,
        list(KINEMATICS_VELOCITY_COLUMNS),
    ] = _velocities[_valid_local]
    kinematics_df.loc[
        _valid_index,
        "is_kinematics_valid",
    ] = True

    _kinematics_segment_rows.append(
        {
            "segment_id": int(_segment_id),
            "rows": _segment_size,
            "valid_rows": int(_valid_local.sum()),
            "masked_rows": int((~_valid_local).sum()),
            "valid_fraction": float(_valid_local.mean()),
            "time_start": float(_time[0]),
            "time_end": float(_time[-1]),
        }
    )

kinematics_segment_summary = pd.DataFrame(_kinematics_segment_rows)

#### Контракт таблицы и автоматические проверки

Проверки ниже фиксируют не только отсутствие NaN внутри допустимой
области, но и саму семантику данных:

- X для модели: x1_sg, y1_sg, x2_sg, y2_sg;
- X_dot: vx1, vy1, vx2, vy2;
- исходные x1, y1, x2, y2: только reference для будущего rollout;
- фильтр строк перед моделированием: is_kinematics_valid.

In [ ]:
_expected_valid_rows = int(
    (
        kinematics_segment_summary["rows"]
        - 2 * KINEMATICS_MARGIN
    ).sum()
)
_valid_mask = kinematics_df["is_kinematics_valid"]
_model_values = kinematics_df.loc[
    :,
    list(KINEMATICS_MODEL_COLUMNS),
]

kinematics_checks = {
    "число строк сохранено": len(kinematics_df) == len(segmented_df),
    "индекс сохранён": kinematics_df.index.equals(segmented_df.index),
    "исходные координаты не изменены": np.array_equal(
        kinematics_df.loc[
            :,
            list(KINEMATICS_RAW_POSITION_COLUMNS),
        ].to_numpy(),
        segmented_df.loc[
            :,
            list(KINEMATICS_RAW_POSITION_COLUMNS),
        ].to_numpy(),
    ),
    "segment_row начинается с нуля": bool(
        kinematics_df.groupby("segment_id")["segment_row"].min().eq(0).all()
    ),
    "ровно m строк скрыто с каждого края": bool(
        kinematics_segment_summary["masked_rows"]
        .eq(2 * KINEMATICS_MARGIN)
        .all()
    ),
    "число валидных строк совпало с формулой": int(_valid_mask.sum())
    == _expected_valid_rows,
    "все значения модели конечны внутри маски": bool(
        np.isfinite(_model_values.loc[_valid_mask].to_numpy()).all()
    ),
    "все значения модели скрыты вне маски": bool(
        _model_values.loc[~_valid_mask].isna().all().all()
    ),
}

if not all(kinematics_checks.values()):
    raise AssertionError(
        "Не выполнены проверки: " + str(kinematics_checks)
    )

kinematics_contract = pd.Series(
    {
        "X для SINDy": ", ".join(KINEMATICS_MODEL_POSITION_COLUMNS),
        "X_dot для SINDy": ", ".join(KINEMATICS_VELOCITY_COLUMNS),
        "raw reference": ", ".join(KINEMATICS_RAW_POSITION_COLUMNS),
        "фильтр строк": "is_kinematics_valid == True",
        "выбранное окно": SELECTED_SG_WINDOW,
        "polyorder": KINEMATICS_POLYORDER,
        "краевой отступ на сторону": KINEMATICS_MARGIN,
        "всего строк": len(kinematics_df),
        "валидных строк": int(_valid_mask.sum()),
        "скрытых краевых строк": int((~_valid_mask).sum()),
    },
    name="значение",
)

display(
    pd.Series(kinematics_checks, name="пройдена").to_frame(),
    kinematics_contract.to_frame(),
    kinematics_segment_summary,
    kinematics_df.loc[_valid_mask].head(),
)

#### Итоговые скорости от времени

График строится на том же чистом segment 1, который использовался при
выборе окна. Красные области показывают по шесть исключённых строк на
каждом краю; линии в этих областях намеренно отсутствуют.

In [ ]:
_plot_segment_id = TODO3_SEGMENT_ID
_plot_segment = kinematics_df.loc[
    kinematics_df["segment_id"].eq(_plot_segment_id)
].copy()
_plot_valid = _plot_segment["is_kinematics_valid"]
_plot_time = _plot_segment["time"]

_speed_1 = np.hypot(_plot_segment["vx1"], _plot_segment["vy1"])
_speed_2 = np.hypot(_plot_segment["vx2"], _plot_segment["vy2"])

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
axes[0].plot(_plot_time, _plot_segment["vx1"], label="vx1", linewidth=1)
axes[0].plot(_plot_time, _plot_segment["vy1"], label="vy1", linewidth=1)
axes[0].set_ylabel("object 1 / coordinate per time")

axes[1].plot(_plot_time, _plot_segment["vx2"], label="vx2", linewidth=1)
axes[1].plot(_plot_time, _plot_segment["vy2"], label="vy2", linewidth=1)
axes[1].set_ylabel("object 2 / coordinate per time")

axes[2].plot(_plot_time, _speed_1, label="speed 1", linewidth=1)
axes[2].plot(_plot_time, _speed_2, label="speed 2", linewidth=1)
axes[2].set_ylabel("speed")
axes[2].set_xlabel("time")

_left_boundary = _plot_time.iloc[KINEMATICS_MARGIN]
_right_boundary = _plot_time.iloc[-KINEMATICS_MARGIN - 1]
for axis in axes:
    axis.axvspan(
        _plot_time.iloc[0],
        _left_boundary,
        color="tab:red",
        alpha=0.10,
        label="masked edge",
    )
    axis.axvspan(
        _right_boundary,
        _plot_time.iloc[-1],
        color="tab:red",
        alpha=0.10,
    )
    axis.grid(alpha=0.25)
    axis.legend(ncols=3, loc="upper right")

fig.suptitle(
    f"Итоговая кинематика: segment {_plot_segment_id}, "
    f"SG({SELECTED_SG_WINDOW}, {KINEMATICS_POLYORDER})"
)
fig.tight_layout()
plt.show()

#### Фазовые проекции

Замкнутость или повторяемость петли здесь является визуальной
диагностикой, а не доказательством физического закона. Начало и конец
валидной части траектории отмечены отдельно, чтобы направление движения
не терялось.

In [ ]:
_phase_data = _plot_segment.loc[_plot_valid]
_phase_pairs = [
    ("x1_sg", "vx1", "object 1: x1_sg — vx1"),
    ("y1_sg", "vy1", "object 1: y1_sg — vy1"),
    ("x2_sg", "vx2", "object 2: x2_sg — vx2"),
    ("y2_sg", "vy2", "object 2: y2_sg — vy2"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for axis, (_position, _velocity, _title) in zip(
    axes.flat,
    _phase_pairs,
):
    axis.plot(
        _phase_data[_position],
        _phase_data[_velocity],
        linewidth=0.8,
        alpha=0.85,
    )
    axis.scatter(
        _phase_data[_position].iloc[0],
        _phase_data[_velocity].iloc[0],
        s=45,
        color="tab:green",
        label="start",
        zorder=3,
    )
    axis.scatter(
        _phase_data[_position].iloc[-1],
        _phase_data[_velocity].iloc[-1],
        s=45,
        color="tab:red",
        label="end",
        zorder=3,
    )
    axis.set_xlabel(_position)
    axis.set_ylabel(_velocity)
    axis.set_title(_title)
    axis.grid(alpha=0.25)
    axis.legend()

fig.suptitle(f"Фазовые проекции, segment {_plot_segment_id}")
fig.tight_layout()
plt.show()

#### Центр и относительное движение

Эти величины вычисляются только для диагностики и не добавляются в
kinematics_df. Используются однозначные определения

$$
r_{mid} = \frac{r_1 + r_2}{2},
\qquad
r_{rel} = r_2 - r_1.
$$

Массы объектов неизвестны, поэтому это геометрическая середина, а не
физический центр масс. Порядок r_2 - r_1 зафиксирован явно.

In [ ]:
_diagnostic = _plot_segment.loc[_plot_valid].copy()
_mid_raw_x = (_diagnostic["x1"] + _diagnostic["x2"]) / 2
_mid_raw_y = (_diagnostic["y1"] + _diagnostic["y2"]) / 2
_mid_sg_x = (_diagnostic["x1_sg"] + _diagnostic["x2_sg"]) / 2
_mid_sg_y = (_diagnostic["y1_sg"] + _diagnostic["y2_sg"]) / 2

_rel_raw_x = _diagnostic["x2"] - _diagnostic["x1"]
_rel_raw_y = _diagnostic["y2"] - _diagnostic["y1"]
_rel_sg_x = _diagnostic["x2_sg"] - _diagnostic["x1_sg"]
_rel_sg_y = _diagnostic["y2_sg"] - _diagnostic["y1_sg"]

_mid_vx = (_diagnostic["vx1"] + _diagnostic["vx2"]) / 2
_mid_vy = (_diagnostic["vy1"] + _diagnostic["vy2"]) / 2
_rel_vx = _diagnostic["vx2"] - _diagnostic["vx1"]
_rel_vy = _diagnostic["vy2"] - _diagnostic["vy1"]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes[0, 0].plot(
    _mid_raw_x,
    _mid_raw_y,
    color="0.75",
    linewidth=0.7,
    label="raw midpoint",
)
axes[0, 0].plot(
    _mid_sg_x,
    _mid_sg_y,
    linewidth=1.0,
    label="SG midpoint",
)
axes[0, 0].set_title("Геометрическая середина")
axes[0, 0].set_xlabel("midpoint x")
axes[0, 0].set_ylabel("midpoint y")

axes[0, 1].plot(
    _rel_raw_x,
    _rel_raw_y,
    color="0.75",
    linewidth=0.7,
    label="raw relative",
)
axes[0, 1].plot(
    _rel_sg_x,
    _rel_sg_y,
    linewidth=1.0,
    label="SG relative",
)
axes[0, 1].set_title("Относительное положение: object 2 − object 1")
axes[0, 1].set_xlabel("relative x")
axes[0, 1].set_ylabel("relative y")

axes[1, 0].plot(
    _diagnostic["time"],
    np.hypot(_mid_vx, _mid_vy),
    linewidth=1.0,
    color="tab:purple",
)
axes[1, 0].set_title("Скорость геометрической середины")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("speed")

axes[1, 1].plot(
    _diagnostic["time"],
    np.hypot(_rel_vx, _rel_vy),
    linewidth=1.0,
    color="tab:orange",
)
axes[1, 1].set_title("Относительная скорость")
axes[1, 1].set_xlabel("time")
axes[1, 1].set_ylabel("speed")

for axis in axes.flat:
    axis.grid(alpha=0.25)
    if axis.get_legend_handles_labels()[0]:
        axis.legend()

fig.suptitle(
    f"Центр и относительное движение, segment {_plot_segment_id}"
)
fig.tight_layout()
plt.show()

### TODO 3 завершён

Выбор `W = 11` применён ко всем 11 непрерывным сегментам и подтверждён
как доминирующий по chord-distance кандидат на 8 из 11 сегментов;
`W = 9/13` сохранены как sensitivity-альтернативы для будущей validation.
Итоговая таблица
сохраняет исходные наблюдения, отдельно хранит сглаженное состояние и
согласованные с ним скорости, а единая маска исключает математически
обоснованные шесть строк с каждого края сегмента.

Следующий шаг — TODO 4: разбиение по segment_id и первый SINDy MVP.
Он намеренно не выполняется в этом notebook run.

## TODO 4 — первый SINDy MVP

- [x] Создать train/validation/test из разных `segment_id`.
- [x] Начать с состояния `(x1, y1, x2, y2)`.
- [x] Использовать полиномиальную библиотеку второй степени.
- [x] Подобрать разреживание на validation, не на test.
- [x] Выписать ненулевые члены найденных уравнений.

Рабочее сглаживание зафиксировано: `W = 11`, `polyorder = 3`. Окна `W = 9/13`
не участвуют в выборе модели и остаются для sensitivity-проверки в TODO 5.
Test-сегменты здесь не оцениваются и не используются для принятия решений.

### 4.A — зафиксированное разбиение по целым траекториям

Сегмент `1` обязательно находится в train: именно на нём был первоначально
выбран `W = 11`. Остальные сегменты распределены так, чтобы в каждой части
были разные длины и участки общей временной записи. Сегменты `2`, `5`, `7`,
на которых exploratory-аудит предпочёл `W = 9`, разнесены по validation,
test и train соответственно.

Это не случайный split строк: каждая непрерывная траектория целиком относится
только к одной части.

In [ ]:
TODO4_TRAIN_SEGMENTS = (1, 3, 4, 7, 8, 9)
TODO4_VALIDATION_SEGMENTS = (2, 6)
TODO4_TEST_SEGMENTS = (0, 5, 10)

_todo4_split_sets = {
    "train": set(TODO4_TRAIN_SEGMENTS),
    "validation": set(TODO4_VALIDATION_SEGMENTS),
    "test": set(TODO4_TEST_SEGMENTS),
}
_all_segment_ids = set(
    kinematics_segment_summary["segment_id"].astype(int)
)

if any(
    _todo4_split_sets[_left] & _todo4_split_sets[_right]
    for _left, _right in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]
):
    raise AssertionError("В split один segment_id попал в две части.")
if set().union(*_todo4_split_sets.values()) != _all_segment_ids:
    raise AssertionError("Split не покрывает все segment_id ровно один раз.")
if TODO3_SEGMENT_ID not in TODO4_TRAIN_SEGMENTS:
    raise AssertionError("Сегмент выбора W должен оставаться в train.")
if SELECTED_SG_WINDOW != 11:
    raise AssertionError("TODO 4 зафиксирован для W = 11.")

todo4_split_manifest = kinematics_segment_summary.loc[
    :,
    ["segment_id", "rows", "valid_rows", "time_start", "time_end"],
].copy()
todo4_split_manifest["split"] = todo4_split_manifest[
    "segment_id"
].map(
    {
        _segment_id: _split_name
        for _split_name, _segment_ids in _todo4_split_sets.items()
        for _segment_id in _segment_ids
    }
)
todo4_split_manifest["duration"] = (
    todo4_split_manifest["time_end"]
    - todo4_split_manifest["time_start"]
)

todo4_split_summary = (
    todo4_split_manifest.groupby("split", sort=False)
    .agg(
        segment_count=("segment_id", "size"),
        valid_rows=("valid_rows", "sum"),
        duration=("duration", "sum"),
    )
)
todo4_split_summary["valid_row_fraction"] = (
    todo4_split_summary["valid_rows"]
    / todo4_split_summary["valid_rows"].sum()
)
todo4_split_summary = todo4_split_summary.reindex(
    ["train", "validation", "test"]
)

display(todo4_split_manifest, todo4_split_summary)

### 4.B — что именно получает SINDy

Для каждой траектории создаются три отдельных массива:

- `X_segment` размера `(n, 4)` — только `x1_sg, y1_sg, x2_sg, y2_sg`;
- `Xdot_segment` размера `(n, 4)` — ответы `vx1, vy1, vx2, vy2`;
- `t_segment` размера `(n,)` — настоящее неравномерное время, сдвинутое к нулю.

Скорости пока не являются входными признаками. Список массивов сообщает
PySINDy, что перед ним несколько независимых траекторий: конец одного сегмента
не соединяется с началом следующего. Абсолютное время в библиотеку признаков
не входит, поэтому проверяется автономная гипотеза `X_dot = f(X)`.

In [ ]:
TODO4_STATE_COLUMNS = tuple(KINEMATICS_MODEL_POSITION_COLUMNS)
TODO4_DERIVATIVE_COLUMNS = tuple(KINEMATICS_VELOCITY_COLUMNS)
TODO4_FEATURE_NAMES = tuple(TODO4_STATE_COLUMNS)
TODO4_ALPHA = 0.05
TODO4_THRESHOLD_GRID = np.geomspace(1e-2, 1e4, 31)
TODO4_MINIMUM_RELATIVE_GAIN = 0.05
TODO4_DIAGNOSTIC_GAIN_TOLERANCE = 0.05

# Намеренно материализуем только development-часть. Test пока остаётся
# закрытым: его координаты и скорости не передаются ни одной модели.
_todo4_development_ids = (
    *TODO4_TRAIN_SEGMENTS,
    *TODO4_VALIDATION_SEGMENTS,
)
todo4_trajectories = {}
for _segment_id in _todo4_development_ids:
    _segment = kinematics_df.loc[
        kinematics_df["segment_id"].eq(_segment_id),
        [
            "time",
            "is_kinematics_valid",
            *TODO4_STATE_COLUMNS,
            *TODO4_DERIVATIVE_COLUMNS,
        ],
    ]
    _segment = _segment.loc[_segment["is_kinematics_valid"]]
    _x = _segment.loc[:, list(TODO4_STATE_COLUMNS)].to_numpy(dtype=float)
    _x_dot = _segment.loc[
        :, list(TODO4_DERIVATIVE_COLUMNS)
    ].to_numpy(dtype=float)
    _time = _segment["time"].to_numpy(dtype=float)
    _time = _time - _time[0]

    if _x.shape != _x_dot.shape or _x.shape[1] != 4:
        raise AssertionError(f"Неверная форма segment {_segment_id}.")
    if len(_time) != len(_x) or not np.all(np.diff(_time) > 0):
        raise AssertionError(f"Неверное время segment {_segment_id}.")
    if not np.isfinite(_x).all() or not np.isfinite(_x_dot).all():
        raise AssertionError(f"Неконечные данные segment {_segment_id}.")

    todo4_trajectories[_segment_id] = {
        "x": _x,
        "x_dot": _x_dot,
        "t": _time,
    }

if set(todo4_trajectories) & set(TODO4_TEST_SEGMENTS):
    raise AssertionError("Test случайно материализован в TODO 4.")

def _todo4_arrays(segment_ids, field):
    return [todo4_trajectories[_id][field] for _id in segment_ids]

def _todo4_make_model(threshold):
    return ps.SINDy(
        feature_library=ps.PolynomialLibrary(
            degree=2,
            include_bias=True,
            include_interaction=True,
        ),
        optimizer=ps.STLSQ(
            threshold=float(threshold),
            alpha=TODO4_ALPHA,
            max_iter=100,
            normalize_columns=True,
            unbias=True,
        ),
    )

def _todo4_fit(threshold, segment_ids):
    _model = _todo4_make_model(threshold)
    with warnings.catch_warnings():
        # Верх сетки намеренно включает обнулённые уравнения; их видно
        # по nonzero_terms, поэтому повторяющийся warning здесь скрыт.
        warnings.filterwarnings(
            "ignore",
            message="Sparsity parameter is too big.*",
            category=UserWarning,
        )
        _model.fit(
            x=_todo4_arrays(segment_ids, "x"),
            t=_todo4_arrays(segment_ids, "t"),
            x_dot=_todo4_arrays(segment_ids, "x_dot"),
            feature_names=list(TODO4_FEATURE_NAMES),
        )
    return _model

_todo4_train_derivative_scale = np.std(
    np.vstack(_todo4_arrays(TODO4_TRAIN_SEGMENTS, "x_dot")),
    axis=0,
)
if np.any(_todo4_train_derivative_scale <= np.finfo(float).eps):
    raise AssertionError("Нулевая train-шкала производной.")

def _todo4_score_prediction_lists(segment_ids, prediction_list):
    _detail_rows = []
    _segment_scores = []
    _integrated_squared_error = np.zeros(4, dtype=float)
    _total_duration = 0.0
    _truth_blocks = []
    _prediction_blocks = []
    for _segment_id, _prediction in zip(
        segment_ids, prediction_list, strict=True
    ):
        _truth = todo4_trajectories[_segment_id]["x_dot"]
        _time = todo4_trajectories[_segment_id]["t"]
        _prediction = np.asarray(_prediction, dtype=float)
        if _prediction.shape != _truth.shape:
            raise AssertionError(
                f"Prediction shape mismatch for segment {_segment_id}."
            )
        _duration = float(_time[-1] - _time[0])
        _squared_error_integral = np.trapezoid(
            (_prediction - _truth) ** 2, x=_time, axis=0
        )
        _truth_time_mean = (
            np.trapezoid(_truth, x=_time, axis=0) / _duration
        )
        _truth_squared_integral = np.trapezoid(
            (_truth - _truth_time_mean) ** 2, x=_time, axis=0
        )
        _r2 = 1 - _squared_error_integral / _truth_squared_integral
        _rmse = np.sqrt(_squared_error_integral / _duration)
        _nrmse = _rmse / _todo4_train_derivative_scale
        _local_scale = np.std(_truth, axis=0)
        _segment_scores.append(float(np.sqrt(np.mean(_nrmse**2))))
        _integrated_squared_error += _squared_error_integral
        _total_duration += _duration
        for _component, _rmse_value, _nrmse_value, _r2_value, _local_scale_value, _train_scale_value in zip(
            TODO4_DERIVATIVE_COLUMNS,
            _rmse,
            _nrmse,
            _r2,
            _local_scale,
            _todo4_train_derivative_scale,
            strict=True,
        ):
            _detail_rows.append(
                {
                    "segment_id": _segment_id,
                    "component": _component,
                    "rmse": float(_rmse_value),
                    "validation_std_true": float(_local_scale_value),
                    "train_std_scale": float(_train_scale_value),
                    "nrmse": float(_nrmse_value),
                    "r2": float(_r2_value),
                }
            )
        _truth_blocks.append(_truth)
        _prediction_blocks.append(_prediction)

    _details = pd.DataFrame(_detail_rows)
    _truth_all = np.vstack(_truth_blocks)
    _prediction_all = np.vstack(_prediction_blocks)
    _residual_all = _prediction_all - _truth_all
    _micro_nrmse_by_component = (
        np.sqrt(_integrated_squared_error / _total_duration)
        / _todo4_train_derivative_scale
    )
    _sse = np.sum(_residual_all**2, axis=0)
    _sst = np.sum(
        (_truth_all - _truth_all.mean(axis=0)) ** 2,
        axis=0,
    )
    _summary = {
        "macro_nrmse": float(np.mean(_segment_scores)),
        "micro_nrmse": float(
            np.sqrt(np.mean(_micro_nrmse_by_component**2))
        ),
        "mean_r2": float(np.mean(1 - _sse / _sst)),
    }
    return _details, _summary

def _todo4_evaluate(model, segment_ids):
    _predictions = model.predict(_todo4_arrays(segment_ids, "x"))
    if isinstance(_predictions, np.ndarray):
        _predictions = [_predictions]
    return _todo4_score_prediction_lists(segment_ids, _predictions)

### 4.C — выбор единственного hyperparameter на validation

Библиотека второй степени содержит 15 кандидатов на каждое из четырёх
уравнений. `normalize_columns=True` используется только внутри STLSQ для
устойчивого сравнения библиотечных столбцов; возвращаемые коэффициенты снова
выражены в исходных координатах. `alpha = 0.05` фиксируется заранее.

Перебирается только threshold. Основная метрика — time-weighted macro NRMSE:
ошибка интегрируется по настоящему `t`, делится на train-only std соответствующей
скорости, затем внутри каждого validation-сегмента берётся RMS четырёх компонент,
а итог усредняется поровну по двум сегментам.

Первый validation dry run показал дефект исходного правила «не хуже минимума
на 5%»: оно выбирало полностью нулевую модель. До какого-либо просмотра test
протокол скорректирован и зафиксирован: кандидат должен снизить macro NRMSE
хотя бы на 5% относительно более сильного из двух простых baseline — нулевой
скорости и средней train-скорости. Если gate не пройден, admissible-модели нет.
Чтобы всё же исследовать структуру слабого сигнала, отдельно выбирается
exploratory-представитель: самая разреженная ненулевая модель, сохраняющая не
менее 95% лучшего observed improvement. Она не считается validated-моделью.

In [ ]:
_train_x_dot_all = np.vstack(
    _todo4_arrays(TODO4_TRAIN_SEGMENTS, "x_dot")
)
_train_mean_velocity = _train_x_dot_all.mean(axis=0)
_baseline_predictions = [
    np.broadcast_to(
        _train_mean_velocity,
        todo4_trajectories[_segment_id]["x_dot"].shape,
    )
    for _segment_id in TODO4_VALIDATION_SEGMENTS
]
todo4_baseline_details, todo4_baseline_summary = (
    _todo4_score_prediction_lists(
        TODO4_VALIDATION_SEGMENTS,
        _baseline_predictions,
    )
)
_zero_predictions = [
    np.zeros_like(todo4_trajectories[_segment_id]["x_dot"])
    for _segment_id in TODO4_VALIDATION_SEGMENTS
]
todo4_zero_details, todo4_zero_summary = (
    _todo4_score_prediction_lists(
        TODO4_VALIDATION_SEGMENTS, _zero_predictions
    )
)
_reference_baseline_macro_nrmse = min(
    todo4_baseline_summary["macro_nrmse"],
    todo4_zero_summary["macro_nrmse"],
)

todo4_models_by_threshold = {}
_todo4_threshold_rows = []
for _threshold in TODO4_THRESHOLD_GRID:
    _model = _todo4_fit(_threshold, TODO4_TRAIN_SEGMENTS)
    _, _validation_summary = _todo4_evaluate(
        _model, TODO4_VALIDATION_SEGMENTS
    )
    _coefficients = _model.coefficients()
    _support = np.abs(_coefficients) > 1e-12
    todo4_models_by_threshold[float(_threshold)] = _model
    _todo4_threshold_rows.append(
        {
            "threshold": float(_threshold),
            **_validation_summary,
            "nonzero_terms": int(_support.sum()),
            "active_library_terms": int(_support.any(axis=0).sum()),
        }
    )

todo4_threshold_results = pd.DataFrame(_todo4_threshold_rows)
todo4_threshold_results["relative_gain_vs_baseline"] = (
    _reference_baseline_macro_nrmse
    - todo4_threshold_results["macro_nrmse"]
) / _reference_baseline_macro_nrmse
_nonzero_results = todo4_threshold_results.loc[
    todo4_threshold_results["nonzero_terms"].gt(0)
]
_admissible_results = _nonzero_results.loc[
    _nonzero_results["relative_gain_vs_baseline"].ge(
        TODO4_MINIMUM_RELATIVE_GAIN
    )
]
TODO4_COORDINATE_ONLY_PASSED = not _admissible_results.empty
TODO4_VALIDATED_THRESHOLD = None
todo4_validated_model = None
if TODO4_COORDINATE_ONLY_PASSED:
    _validated_row = (
        _admissible_results.sort_values(
            ["nonzero_terms", "threshold", "macro_nrmse"],
            ascending=[True, False, True],
        ).iloc[0]
    )
    TODO4_VALIDATED_THRESHOLD = float(_validated_row["threshold"])
    todo4_validated_model = todo4_models_by_threshold[
        TODO4_VALIDATED_THRESHOLD
    ]

_best_nonzero_gain = _nonzero_results[
    "relative_gain_vs_baseline"
].max()
if _best_nonzero_gain > 0:
    _diagnostic_gain_limit = (
        (1 - TODO4_DIAGNOSTIC_GAIN_TOLERANCE)
        * _best_nonzero_gain
    )
    _diagnostic_candidates = _nonzero_results.loc[
        _nonzero_results["relative_gain_vs_baseline"].ge(
            _diagnostic_gain_limit
        )
    ]
else:
    _diagnostic_candidates = _nonzero_results.nsmallest(
        1, "macro_nrmse"
    )
todo4_diagnostic_row = (
    _diagnostic_candidates.sort_values(
        ["nonzero_terms", "threshold", "macro_nrmse"],
        ascending=[True, False, True],
    )
    .iloc[0]
)
TODO4_DIAGNOSTIC_THRESHOLD = float(todo4_diagnostic_row["threshold"])
todo4_diagnostic_train_model = todo4_models_by_threshold[
    TODO4_DIAGNOSTIC_THRESHOLD
]

if todo4_threshold_results["nonzero_terms"].nunique() < 2:
    raise AssertionError("Threshold grid не меняет разреженность модели.")

_best_rows = todo4_threshold_results.nsmallest(8, "macro_nrmse")
display(
    pd.DataFrame(
        {
            "train-mean baseline": todo4_baseline_summary,
            "zero-velocity baseline": todo4_zero_summary,
        }
    ),
    _best_rows,
    todo4_diagnostic_row.to_frame(name="exploratory diagnostic"),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].semilogx(
    todo4_threshold_results["threshold"],
    todo4_threshold_results["macro_nrmse"],
    marker="o",
    markersize=4,
)
axes[0].axhline(
    _reference_baseline_macro_nrmse,
    color="0.4",
    linestyle="--",
    label="stronger simple baseline",
)
axes[0].axhline(
    (1 - TODO4_MINIMUM_RELATIVE_GAIN)
    * _reference_baseline_macro_nrmse,
    color="tab:orange",
    linestyle=":",
    label="5% acceptance gate",
)
axes[0].scatter(
    [TODO4_DIAGNOSTIC_THRESHOLD],
    [todo4_diagnostic_row["macro_nrmse"]],
    s=80,
    color="tab:red",
    zorder=3,
    label="exploratory diagnostic",
)
axes[0].set_xlabel("STLSQ threshold")
axes[0].set_ylabel("validation macro NRMSE")
axes[0].set_title("Ошибка производных")
axes[0].legend()

axes[1].semilogx(
    todo4_threshold_results["threshold"],
    todo4_threshold_results["nonzero_terms"],
    marker="o",
    markersize=4,
)
axes[1].scatter(
    [TODO4_DIAGNOSTIC_THRESHOLD],
    [todo4_diagnostic_row["nonzero_terms"]],
    s=80,
    color="tab:red",
    zorder=3,
)
axes[1].set_xlabel("STLSQ threshold")
axes[1].set_ylabel("ненулевых коэффициентов из 60")
axes[1].set_title("Разреженность")
for _axis in axes:
    _axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 4.D — детализация ошибки и устойчивость структуры

Для exploratory threshold показывается ошибка каждой скорости на каждом
validation-сегменте. Дополнительно проверяются соседние значения threshold и
leave-one-train-segment-out: высокая Jaccard-схожесть означает, что набор
ненулевых членов мало меняется. Это диагностика устойчивости, а не новый этап
подбора и не повод менять уже зафиксированный split. При непройденном
acceptance gate даже высокая Jaccard-схожесть не превращает diagnostic-модель
в validated-модель.

In [ ]:
todo4_diagnostic_validation_details, (
    todo4_diagnostic_validation_summary
) = _todo4_evaluate(
    todo4_diagnostic_train_model, TODO4_VALIDATION_SEGMENTS
)

def _todo4_support_set(model):
    _support = np.abs(model.coefficients()) > 1e-12
    return set(zip(*np.where(_support)))

def _todo4_jaccard(left, right):
    _union = left | right
    return 1.0 if not _union else len(left & right) / len(_union)

_diagnostic_support = _todo4_support_set(todo4_diagnostic_train_model)
_diagnostic_grid_index = int(
    np.flatnonzero(
        np.isclose(TODO4_THRESHOLD_GRID, TODO4_DIAGNOSTIC_THRESHOLD)
    )[0]
)
_neighbor_start = max(0, _diagnostic_grid_index - 2)
_neighbor_stop = min(
    len(TODO4_THRESHOLD_GRID), _diagnostic_grid_index + 3
)
_neighbor_rows = []
for _threshold in TODO4_THRESHOLD_GRID[_neighbor_start:_neighbor_stop]:
    _neighbor_model = todo4_models_by_threshold[float(_threshold)]
    _neighbor_result = todo4_threshold_results.loc[
        np.isclose(todo4_threshold_results["threshold"], _threshold)
    ].iloc[0]
    _neighbor_rows.append(
        {
            "threshold": float(_threshold),
            "macro_nrmse": _neighbor_result["macro_nrmse"],
            "nonzero_terms": int(_neighbor_result["nonzero_terms"]),
            "support_jaccard_vs_selected": _todo4_jaccard(
                _diagnostic_support, _todo4_support_set(_neighbor_model)
            ),
        }
    )
todo4_neighbor_stability = pd.DataFrame(_neighbor_rows)

_loto_rows = []
for _held_out_segment in TODO4_TRAIN_SEGMENTS:
    _remaining_segments = tuple(
        _segment_id
        for _segment_id in TODO4_TRAIN_SEGMENTS
        if _segment_id != _held_out_segment
    )
    _loto_model = _todo4_fit(
        TODO4_DIAGNOSTIC_THRESHOLD, _remaining_segments
    )
    _loto_support = _todo4_support_set(_loto_model)
    _loto_rows.append(
        {
            "held_out_train_segment": _held_out_segment,
            "remaining_valid_rows": sum(
                len(todo4_trajectories[_id]["x"])
                for _id in _remaining_segments
            ),
            "nonzero_terms": len(_loto_support),
            "support_jaccard_vs_full_train": _todo4_jaccard(
                _diagnostic_support, _loto_support
            ),
        }
    )
todo4_loto_stability = pd.DataFrame(_loto_rows)

display(
    todo4_diagnostic_validation_details.pivot(
        index="segment_id", columns="component", values="nrmse"
    ),
    todo4_diagnostic_validation_details.pivot(
        index="segment_id", columns="component", values="r2"
    ),
    todo4_neighbor_stability,
    todo4_loto_stability,
)

### 4.E — exploratory-формула перед закрытым test

Строгий 5%-й gate решает, существует ли validated coordinate-only модель.
Независимо от вердикта компактный ненулевой diagnostic threshold переобучается
на train + validation, чтобы напечатать исследуемую формулу. Если gate не
пройден, это не кандидат на физический закон, а зафиксированный отрицательный
результат и ориентир для следующей гипотезы со скоростями в состоянии.

In [ ]:
TODO4_DEVELOPMENT_SEGMENTS = (
    *TODO4_TRAIN_SEGMENTS, *TODO4_VALIDATION_SEGMENTS
)
todo4_diagnostic_refit_model = _todo4_fit(
    TODO4_DIAGNOSTIC_THRESHOLD, TODO4_DEVELOPMENT_SEGMENTS
)
_diagnostic_refit_coefficients = (
    todo4_diagnostic_refit_model.coefficients()
)
_diagnostic_feature_names = (
    todo4_diagnostic_refit_model.get_feature_names()
)
_equation_names = tuple(
    f"dot {column}" for column in TODO4_FEATURE_NAMES
)
_coefficient_rows = []
for _equation_index, _equation_name in enumerate(_equation_names):
    for _feature_index, _feature_name in enumerate(_diagnostic_feature_names):
        _coefficient = _diagnostic_refit_coefficients[
            _equation_index, _feature_index
        ]
        if abs(_coefficient) > 1e-12:
            _coefficient_rows.append(
                {
                    "equation": _equation_name,
                    "term": _feature_name,
                    "coefficient": float(_coefficient),
                }
            )
todo4_diagnostic_coefficients = pd.DataFrame(
    _coefficient_rows,
    columns=["equation", "term", "coefficient"],
)
todo4_diagnostic_equations = pd.Series(
    todo4_diagnostic_refit_model.equations(precision=6),
    index=_equation_names,
    name="right-hand side",
)
_diagnostic_refit_support = _todo4_support_set(
    todo4_diagnostic_refit_model
)
_refit_support_jaccard = _todo4_jaccard(
    _diagnostic_support, _diagnostic_refit_support
)

display(
    todo4_diagnostic_equations.to_frame(),
    todo4_diagnostic_coefficients,
)
display(
    Markdown(
        f"""
        **Результат TODO 4.** Exploratory `threshold = {TODO4_DIAGNOSTIC_THRESHOLD:.6g}`:
        validation macro NRMSE = `{todo4_diagnostic_validation_summary['macro_nrmse']:.4f}`,
        micro NRMSE = `{todo4_diagnostic_validation_summary['micro_nrmse']:.4f}`,
        mean R² = `{todo4_diagnostic_validation_summary['mean_r2']:.4f}`.
        Сильнейший простой baseline имеет macro NRMSE =
        `{_reference_baseline_macro_nrmse:.4f}`; observed gain диагностической
        модели равен
        `{100 * todo4_diagnostic_row['relative_gain_vs_baseline']:.2f}%`,
        тогда как acceptance gate требует
        `{100 * TODO4_MINIMUM_RELATIVE_GAIN:.1f}%`.
        **Validated coordinate-only model: {TODO4_COORDINATE_ONLY_PASSED}.**
        В train diagnostic-модели осталось
        **{len(_diagnostic_support)} из 60** коэффициентов; после refit на
        train + validation — **{len(_diagnostic_refit_support)}**, Jaccard структуры =
        `{_refit_support_jaccard:.3f}`.

        Если gate не пройден, coordinate-only гипотеза не дала практически
        полезной модели производных. Формула выше — только diagnostic, не
        физический закон. Test-сегменты `{TODO4_TEST_SEGMENTS}` ни разу не
        передавались модели; rollout и `W = 9/13` sensitivity остаются TODO 5.
        """
    )
)

### Вывод TODO 4: одних координат недостаточно

Проверена автономная coordinate-only гипотеза
`[x1_sg, y1_sg, x2_sg, y2_sg] → [vx1, vy1, vx2, vy2]`: можно ли по одному
текущему положению восстановить мгновенную скорость. Модель обучалась на шести
целых сегментах и оценивалась на двух других; переходы между сегментами не
склеивались.

Лучший простой baseline получил time-weighted macro NRMSE `1.0321`. Самый
компактный ненулевой diagnostic-кандидат (`threshold = 1000`, четыре члена) —
`1.0241` при `mean R² = 0.0166`. Улучшение составляет только `0.78%` и не
проходит 5%-й acceptance gate. На отдельных validation-компонентах R² близок
к нулю или отрицателен. Следовательно, практически полезной validated
coordinate-only модели не найдено.

Четырёхчленная система стабильно сохраняет структуру двух независимых линейных
вращений при leave-one-train-segment-out, но описывает лишь слабый усреднённый
рисунок движения. Устойчивость support не компенсирует отсутствие predictive
skill, поэтому формула остаётся exploratory diagnostic, а не законом движения.

Наиболее вероятная причина — координаты не образуют замкнутое Markov-состояние:
одно положение может соответствовать разным направлениям и скоростям. Следующая
проверяемая гипотеза добавляет скорости в состояние и требует ускорения как часть
`X_dot`. Она будет разработана на тех же train/validation. Test-сегменты
`(0, 5, 10)` остаются закрытыми до фиксации augmented-state кандидата.

## TODO 5 — расширенное состояние и закрытый test

- [x] На тех же train/validation подготовить состояние из координат и скоростей;
  ускорения оценивать только внутри сегментов с новой краевой маской.
- [x] Подобрать и зафиксировать augmented-state SINDy без просмотра test.
- [x] Провести `W = 9/13` sensitivity и проверку устойчивости solver на validation.
- [x] Сравнить полный augmented pipeline для raw finite differences и SG `W=11`
  на одинаковых строках train/validation, включая короткий rollout.
- [ ] Открыть test только после пройденного rollout gate; текущий кандидат его
  не прошёл, поэтому test сохранён для следующей гипотезы.
- [ ] Сравнить фазовые проекции и относительное движение.
- [x] Сформулировать validation-итог и ограничения обеих гипотез состояния.

### 5.A — ускорения и восьмимерное состояние

Основной pipeline остаётся зафиксированным: `W = 11`, `polyorder = 3`,
реальный неравномерный `time`. После SG-сглаживания координат дважды применяется
`np.gradient`: сначала для скорости, затем для ускорения. Радиус зависимости
равен `5 + 1 + 1 = 7` строкам, поэтому с каждого края сегмента исключается по
семь наблюдений.

Состояние и его производная теперь имеют вид

`X = [x1_sg, y1_sg, x2_sg, y2_sg, vx1, vy1, vx2, vy2]`,

`X_dot = [vx1, vy1, vx2, vy2, ax1, ay1, ax2, ay2]`.

На этом этапе материализуются только train и validation. Test остаётся закрытым.

In [ ]:
TODO5_MAIN_WINDOW = SELECTED_SG_WINDOW
TODO5_SENSITIVITY_WINDOWS = (9, 11, 13)
TODO5_POSITION_NAMES = tuple(KINEMATICS_MODEL_POSITION_COLUMNS)
TODO5_VELOCITY_NAMES = tuple(KINEMATICS_VELOCITY_COLUMNS)
TODO5_ACCELERATION_NAMES = ("ax1", "ay1", "ax2", "ay2")
TODO5_FEATURE_NAMES = (
    *TODO5_POSITION_NAMES, *TODO5_VELOCITY_NAMES
)
TODO5_DERIVATIVE_NAMES = (
    *TODO5_VELOCITY_NAMES, *TODO5_ACCELERATION_NAMES
)

def _todo5_margin(window):
    return (window - 1) // 2 + 2

def _todo5_build_trajectories(window, segment_ids, valid_margin=None):
    if window % 2 == 0 or window <= KINEMATICS_POLYORDER:
        raise ValueError("SG window должно быть нечётным и длиннее polyorder.")
    _trajectories = {}
    _minimum_margin = _todo5_margin(window)
    _margin = _minimum_margin if valid_margin is None else valid_margin
    if _margin < _minimum_margin:
        raise ValueError("valid_margin меньше математического minimum.")
    for _segment_id in segment_ids:
        _segment = segmented_df.loc[
            segmented_df["segment_id"].eq(_segment_id),
            ["time", *KINEMATICS_RAW_POSITION_COLUMNS],
        ]
        _time_full = _segment["time"].to_numpy(dtype=float)
        _raw_full = _segment.loc[
            :, list(KINEMATICS_RAW_POSITION_COLUMNS)
        ].to_numpy(dtype=float)
        if not np.all(np.diff(_time_full) > 0):
            raise AssertionError(f"time не возрастает в segment {_segment_id}.")
        if len(_segment) <= 2 * _margin:
            raise AssertionError(f"segment {_segment_id} слишком короткий.")

        _positions_full = savgol_filter(
            _raw_full,
            window_length=window,
            polyorder=KINEMATICS_POLYORDER,
            axis=0,
            mode="interp",
        )
        _velocities_full = np.gradient(
            _positions_full, _time_full, axis=0, edge_order=2
        )
        _accelerations_full = np.gradient(
            _velocities_full, _time_full, axis=0, edge_order=2
        )
        _distance_to_edge = np.minimum(
            np.arange(len(_segment)),
            np.arange(len(_segment))[::-1],
        )
        _valid = _distance_to_edge >= _margin
        _time = _time_full[_valid]
        _x = np.column_stack(
            (_positions_full[_valid], _velocities_full[_valid])
        )
        _x_dot = np.column_stack(
            (_velocities_full[_valid], _accelerations_full[_valid])
        )
        if _x.shape != _x_dot.shape or _x.shape[1] != 8:
            raise AssertionError(f"Неверная augmented shape segment {_segment_id}.")
        if not np.isfinite(_x).all() or not np.isfinite(_x_dot).all():
            raise AssertionError(f"Неконечные значения segment {_segment_id}.")

        _trajectories[_segment_id] = {
            "x": _x,
            "x_dot": _x_dot,
            "t": _time - _time[0],
            "time_absolute": _time,
            "raw_positions": _raw_full[_valid],
            "source_index": _segment.index.to_numpy()[_valid],
            "margin": _margin,
        }
    return _trajectories

TODO5_DEVELOPMENT_SEGMENTS = (
    *TODO4_TRAIN_SEGMENTS, *TODO4_VALIDATION_SEGMENTS
)
todo5_trajectories = _todo5_build_trajectories(
    TODO5_MAIN_WINDOW, TODO5_DEVELOPMENT_SEGMENTS
)
if set(todo5_trajectories) & set(TODO4_TEST_SEGMENTS):
    raise AssertionError("Test случайно материализован до freeze.")

# На общей внутренней области новый расчёт обязан совпасть с TODO 3.
for _segment_id in TODO5_DEVELOPMENT_SEGMENTS:
    _trajectory = todo5_trajectories[_segment_id]
    _reference = kinematics_df.loc[
        _trajectory["source_index"],
        [*TODO5_POSITION_NAMES, *TODO5_VELOCITY_NAMES],
    ].to_numpy(dtype=float)
    if not np.allclose(_trajectory["x"], _reference, rtol=1e-12, atol=1e-12):
        raise AssertionError(f"TODO 3/5 mismatch segment {_segment_id}.")

todo5_data_summary = pd.DataFrame(
    [
        {
            "segment_id": _segment_id,
            "split": (
                "train" if _segment_id in TODO4_TRAIN_SEGMENTS
                else "validation"
            ),
            "valid_rows": len(_trajectory["x"]),
            "margin_per_edge": _trajectory["margin"],
            "duration": _trajectory["t"][-1],
        }
        for _segment_id, _trajectory in todo5_trajectories.items()
    ]
)
display(
    pd.Series(
        {
            "X": ", ".join(TODO5_FEATURE_NAMES),
            "X_dot": ", ".join(TODO5_DERIVATIVE_NAMES),
            "window": TODO5_MAIN_WINDOW,
            "margin per edge": _todo5_margin(TODO5_MAIN_WINDOW),
            "test materialized": False,
        },
        name="augmented contract",
    ).to_frame(),
    todo5_data_summary,
    todo5_data_summary.groupby("split")["valid_rows"].sum(),
)

### 5.B — SINDy для ускорений

Полная degree-2 library для восьми переменных содержит 45 членов на каждое из
восьми уравнений. Первые четыре уравнения кинематически просты —
`dot position = velocity`. Чтобы они не замаскировали слабую динамику, threshold
выбирается только по ошибке последних четырёх уравнений ускорения.

Primary metric совпадает по принципу с TODO 4: time-weighted macro NRMSE,
train-only scale, равный вес validation-сегментов. Baseline — нулевое или
среднее train-ускорение, выбирается более сильный. Validated-кандидат должен
сохранить не менее 95% лучшего observed gain, улучшить baseline минимум на 5%
относительно baseline. Кинематические равенства `dot position = velocity`
задаются точно по определению; SINDy используется только как правая часть
четырёх уравнений ускорения.

In [ ]:
TODO5_ALPHA = TODO4_ALPHA
TODO5_THRESHOLD_GRID = np.geomspace(1e-2, 1e5, 36)
TODO5_MINIMUM_RELATIVE_GAIN = 0.05
TODO5_GAIN_RETENTION = 0.95

def _todo5_arrays(trajectories, segment_ids, field):
    return [trajectories[_id][field] for _id in segment_ids]

def _todo5_make_model(threshold):
    return ps.SINDy(
        feature_library=ps.PolynomialLibrary(
            degree=2, include_bias=True, include_interaction=True
        ),
        optimizer=ps.STLSQ(
            threshold=float(threshold),
            alpha=TODO5_ALPHA,
            max_iter=100,
            normalize_columns=True,
            unbias=True,
        ),
    )

def _todo5_fit(threshold, segment_ids, trajectories=todo5_trajectories):
    _model = _todo5_make_model(threshold)
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Sparsity parameter is too big.*",
            category=UserWarning,
        )
        _model.fit(
            x=_todo5_arrays(trajectories, segment_ids, "x"),
            t=_todo5_arrays(trajectories, segment_ids, "t"),
            x_dot=_todo5_arrays(trajectories, segment_ids, "x_dot"),
            feature_names=list(TODO5_FEATURE_NAMES),
        )
    return _model

_todo5_train_x_dot = np.vstack(
    _todo5_arrays(todo5_trajectories, TODO4_TRAIN_SEGMENTS, "x_dot")
)
_todo5_train_velocity_scale = np.std(_todo5_train_x_dot[:, :4], axis=0)
_todo5_train_acceleration_scale = np.std(_todo5_train_x_dot[:, 4:], axis=0)
if np.any(_todo5_train_acceleration_scale <= np.finfo(float).eps):
    raise AssertionError("Нулевая train-шкала ускорения.")

def _todo5_time_weighted_score(
    trajectories, segment_ids, prediction_list, column_slice, scale
):
    _details = []
    _segment_scores = []
    _integrated_error = np.zeros(len(scale), dtype=float)
    _total_duration = 0.0
    _truth_all = []
    _prediction_all = []
    _component_names = TODO5_DERIVATIVE_NAMES[column_slice]
    for _segment_id, _prediction in zip(
        segment_ids, prediction_list, strict=True
    ):
        _truth = trajectories[_segment_id]["x_dot"][:, column_slice]
        _prediction = np.asarray(_prediction)[:, column_slice]
        _time = trajectories[_segment_id]["t"]
        _duration = float(_time[-1] - _time[0])
        if _prediction.shape != _truth.shape:
            raise AssertionError(f"Prediction shape segment {_segment_id}.")
        _squared_integral = np.trapezoid(
            (_prediction - _truth) ** 2, x=_time, axis=0
        )
        _truth_mean = np.trapezoid(_truth, x=_time, axis=0) / _duration
        _truth_integral = np.trapezoid(
            (_truth - _truth_mean) ** 2, x=_time, axis=0
        )
        _rmse = np.sqrt(_squared_integral / _duration)
        _nrmse = _rmse / scale
        _r2 = 1 - _squared_integral / _truth_integral
        _segment_scores.append(float(np.sqrt(np.mean(_nrmse**2))))
        for _name, _rmse_value, _nrmse_value, _r2_value in zip(
            _component_names, _rmse, _nrmse, _r2, strict=True
        ):
            _details.append(
                {
                    "segment_id": _segment_id,
                    "component": _name,
                    "rmse": float(_rmse_value),
                    "nrmse": float(_nrmse_value),
                    "r2": float(_r2_value),
                }
            )
        _integrated_error += _squared_integral
        _total_duration += _duration
        _truth_all.append(_truth)
        _prediction_all.append(_prediction)
    _truth_all = np.vstack(_truth_all)
    _prediction_all = np.vstack(_prediction_all)
    _sse = np.sum((_prediction_all - _truth_all) ** 2, axis=0)
    _sst = np.sum(
        (_truth_all - _truth_all.mean(axis=0)) ** 2, axis=0
    )
    _micro_components = (
        np.sqrt(_integrated_error / _total_duration) / scale
    )
    return pd.DataFrame(_details), {
        "macro_nrmse": float(np.mean(_segment_scores)),
        "micro_nrmse": float(np.sqrt(np.mean(_micro_components**2))),
        "mean_r2": float(np.mean(1 - _sse / _sst)),
    }

def _todo5_evaluate_model(model, trajectories, segment_ids):
    _prediction = model.predict(
        _todo5_arrays(trajectories, segment_ids, "x")
    )
    if isinstance(_prediction, np.ndarray):
        _prediction = [_prediction]
    _acceleration = _todo5_time_weighted_score(
        trajectories, segment_ids, _prediction, slice(4, 8),
        _todo5_train_acceleration_scale
    )
    _kinematic = _todo5_time_weighted_score(
        trajectories, segment_ids, _prediction, slice(0, 4),
        _todo5_train_velocity_scale
    )
    return _acceleration, _kinematic

In [ ]:
_validation_acceleration_truth = [
    todo5_trajectories[_id]["x_dot"][:, 4:]
    for _id in TODO4_VALIDATION_SEGMENTS
]
_zero_acceleration_predictions = [
    np.zeros_like(_truth) for _truth in _validation_acceleration_truth
]
_train_mean_acceleration = _todo5_train_x_dot[:, 4:].mean(axis=0)
_mean_acceleration_predictions = [
    np.broadcast_to(_train_mean_acceleration, _truth.shape)
    for _truth in _validation_acceleration_truth
]

def _todo5_wrap_acceleration_predictions(acceleration_predictions):
    return [
        np.column_stack(
            (
                todo5_trajectories[_id]["x_dot"][:, :4],
                _prediction,
            )
        )
        for _id, _prediction in zip(
            TODO4_VALIDATION_SEGMENTS,
            acceleration_predictions,
            strict=True,
        )
    ]

_, todo5_zero_baseline = _todo5_time_weighted_score(
    todo5_trajectories, TODO4_VALIDATION_SEGMENTS,
    _todo5_wrap_acceleration_predictions(_zero_acceleration_predictions),
    slice(4, 8), _todo5_train_acceleration_scale
)
_, todo5_mean_baseline = _todo5_time_weighted_score(
    todo5_trajectories, TODO4_VALIDATION_SEGMENTS,
    _todo5_wrap_acceleration_predictions(_mean_acceleration_predictions),
    slice(4, 8), _todo5_train_acceleration_scale
)
_todo5_reference_baseline = min(
    todo5_zero_baseline["macro_nrmse"],
    todo5_mean_baseline["macro_nrmse"],
)

todo5_models_by_threshold = {}
_todo5_result_rows = []
for _threshold in TODO5_THRESHOLD_GRID:
    _model = _todo5_fit(_threshold, TODO4_TRAIN_SEGMENTS)
    (_, _acceleration_summary), (_, _kinematic_summary) = (
        _todo5_evaluate_model(
            _model, todo5_trajectories, TODO4_VALIDATION_SEGMENTS
        )
    )
    _support = np.abs(_model.coefficients()) > 1e-12
    todo5_models_by_threshold[float(_threshold)] = _model
    _todo5_result_rows.append(
        {
            "threshold": float(_threshold),
            "acceleration_macro_nrmse": _acceleration_summary["macro_nrmse"],
            "acceleration_micro_nrmse": _acceleration_summary["micro_nrmse"],
            "acceleration_mean_r2": _acceleration_summary["mean_r2"],
            "raw_sindy_kinematic_nrmse": _kinematic_summary["macro_nrmse"],
            "acceleration_terms": int(_support[4:].sum()),
            "raw_sindy_total_terms": int(_support.sum()),
        }
    )

todo5_threshold_results = pd.DataFrame(_todo5_result_rows)
todo5_threshold_results["relative_gain_vs_baseline"] = (
    _todo5_reference_baseline
    - todo5_threshold_results["acceleration_macro_nrmse"]
) / _todo5_reference_baseline
_todo5_nonzero = todo5_threshold_results.loc[
    todo5_threshold_results["acceleration_terms"].gt(0)
]
_todo5_best_gain = _todo5_nonzero["relative_gain_vs_baseline"].max()
if _todo5_best_gain > 0:
    _todo5_compact_candidates = _todo5_nonzero.loc[
        _todo5_nonzero["relative_gain_vs_baseline"].ge(
            TODO5_GAIN_RETENTION * _todo5_best_gain
        )
    ]
else:
    _todo5_compact_candidates = _todo5_nonzero.nsmallest(
        1, "acceleration_macro_nrmse"
    )
todo5_selected_row = _todo5_compact_candidates.sort_values(
    ["acceleration_terms", "threshold", "acceleration_macro_nrmse"],
    ascending=[True, False, True],
).iloc[0]
TODO5_SELECTED_THRESHOLD = float(todo5_selected_row["threshold"])
todo5_selected_train_model = todo5_models_by_threshold[
    TODO5_SELECTED_THRESHOLD
]
TODO5_DERIVATIVE_GATE_PASSED = bool(
    todo5_selected_row["relative_gain_vs_baseline"]
    >= TODO5_MINIMUM_RELATIVE_GAIN
)

display(
    pd.DataFrame(
        {
            "zero acceleration": todo5_zero_baseline,
            "train-mean acceleration": todo5_mean_baseline,
        }
    ),
    todo5_threshold_results.nsmallest(10, "acceleration_macro_nrmse"),
    todo5_selected_row.to_frame(name="selected before rollout"),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].semilogx(
    todo5_threshold_results["threshold"],
    todo5_threshold_results["acceleration_macro_nrmse"],
    marker="o", markersize=4,
)
axes[0].axhline(
    _todo5_reference_baseline, color="0.4", linestyle="--",
    label="stronger acceleration baseline",
)
axes[0].axhline(
    (1 - TODO5_MINIMUM_RELATIVE_GAIN) * _todo5_reference_baseline,
    color="tab:orange", linestyle=":", label="5% gate",
)
axes[0].scatter(
    [TODO5_SELECTED_THRESHOLD],
    [todo5_selected_row["acceleration_macro_nrmse"]],
    color="tab:red", s=75, zorder=3, label="selected",
)
axes[0].set_xlabel("STLSQ threshold")
axes[0].set_ylabel("validation acceleration macro NRMSE")
axes[0].set_title("Acceleration validation")
axes[0].legend()
axes[1].semilogx(
    todo5_threshold_results["threshold"],
    todo5_threshold_results["acceleration_terms"],
    marker="o", markersize=4,
)
axes[1].scatter(
    [TODO5_SELECTED_THRESHOLD],
    [todo5_selected_row["acceleration_terms"]],
    color="tab:red", s=75, zorder=3,
)
axes[1].set_xlabel("STLSQ threshold")
axes[1].set_ylabel("ненулевых acceleration coefficients")
axes[1].set_title("Dynamic sparsity")
for _axis in axes:
    _axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 5.C — validation diagnostics и устойчивость

Derivative gate проверяет локальную модель ускорения. До rollout дополнительно
показываются component-wise ошибки, соседние threshold, исключение каждого
train-сегмента и полный refit для `W = 9/11/13` при неизменном threshold.
Все три окна сравниваются на общем отступе 8 строк с каждого края.
Sensitivity не используется для повторного выбора параметров.

In [ ]:
(todo5_validation_acceleration_details,
 todo5_validation_acceleration_summary), (
    todo5_validation_kinematic_details,
    todo5_validation_kinematic_summary,
) = _todo5_evaluate_model(
    todo5_selected_train_model,
    todo5_trajectories,
    TODO4_VALIDATION_SEGMENTS,
)

def _todo5_dynamic_support(model):
    _support = np.abs(model.coefficients()[4:]) > 1e-12
    return set(zip(*np.where(_support)))

_todo5_selected_support = _todo5_dynamic_support(
    todo5_selected_train_model
)
_todo5_selected_grid_index = int(
    np.flatnonzero(
        np.isclose(TODO5_THRESHOLD_GRID, TODO5_SELECTED_THRESHOLD)
    )[0]
)
_neighbor_rows = []
for _index in range(
    max(0, _todo5_selected_grid_index - 2),
    min(len(TODO5_THRESHOLD_GRID), _todo5_selected_grid_index + 3),
):
    _threshold = float(TODO5_THRESHOLD_GRID[_index])
    _row = todo5_threshold_results.loc[
        np.isclose(todo5_threshold_results["threshold"], _threshold)
    ].iloc[0]
    _support = _todo5_dynamic_support(
        todo5_models_by_threshold[_threshold]
    )
    _neighbor_rows.append(
        {
            "threshold": _threshold,
            "acceleration_macro_nrmse": _row["acceleration_macro_nrmse"],
            "acceleration_terms": len(_support),
            "support_jaccard": _todo4_jaccard(
                _todo5_selected_support, _support
            ),
        }
    )
todo5_neighbor_stability = pd.DataFrame(_neighbor_rows)

_loto_rows = []
for _held_out in TODO4_TRAIN_SEGMENTS:
    _remaining = tuple(
        _id for _id in TODO4_TRAIN_SEGMENTS if _id != _held_out
    )
    _model = _todo5_fit(TODO5_SELECTED_THRESHOLD, _remaining)
    _support = _todo5_dynamic_support(_model)
    _loto_rows.append(
        {
            "held_out_train_segment": _held_out,
            "acceleration_terms": len(_support),
            "support_jaccard": _todo4_jaccard(
                _todo5_selected_support, _support
            ),
        }
    )
todo5_loto_stability = pd.DataFrame(_loto_rows)

_sensitivity_rows = []
todo5_sensitivity_models = {}
_sensitivity_common_margin = max(
    _todo5_margin(_candidate)
    for _candidate in TODO5_SENSITIVITY_WINDOWS
)
for _window in TODO5_SENSITIVITY_WINDOWS:
    _trajectories = _todo5_build_trajectories(
        _window, TODO5_DEVELOPMENT_SEGMENTS,
        valid_margin=_sensitivity_common_margin,
    )
    _train_x_dot = np.vstack(
        _todo5_arrays(_trajectories, TODO4_TRAIN_SEGMENTS, "x_dot")
    )
    _acceleration_scale = np.std(_train_x_dot[:, 4:], axis=0)
    _velocity_scale = np.std(_train_x_dot[:, :4], axis=0)
    _model = _todo5_fit(
        TODO5_SELECTED_THRESHOLD, TODO4_TRAIN_SEGMENTS, _trajectories
    )
    _prediction = _model.predict(
        _todo5_arrays(_trajectories, TODO4_VALIDATION_SEGMENTS, "x")
    )
    (_, _acceleration_summary) = _todo5_time_weighted_score(
        _trajectories, TODO4_VALIDATION_SEGMENTS, _prediction,
        slice(4, 8), _acceleration_scale
    )
    (_, _kinematic_summary) = _todo5_time_weighted_score(
        _trajectories, TODO4_VALIDATION_SEGMENTS, _prediction,
        slice(0, 4), _velocity_scale
    )
    _support = _todo5_dynamic_support(_model)
    todo5_sensitivity_models[_window] = _model
    _sensitivity_rows.append(
        {
            "window": _window,
            "common_margin_per_edge": _sensitivity_common_margin,
            "acceleration_macro_nrmse": _acceleration_summary["macro_nrmse"],
            "acceleration_mean_r2": _acceleration_summary["mean_r2"],
            "raw_sindy_kinematic_nrmse": _kinematic_summary["macro_nrmse"],
            "acceleration_terms": len(_support),
            "support_jaccard_vs_w11": _todo4_jaccard(
                _todo5_selected_support, _support
            ),
        }
    )
todo5_smoothing_sensitivity = pd.DataFrame(_sensitivity_rows)

display(
    todo5_validation_acceleration_details.pivot(
        index="segment_id", columns="component", values="nrmse"
    ),
    todo5_validation_acceleration_details.pivot(
        index="segment_id", columns="component", values="r2"
    ),
    todo5_neighbor_stability,
    todo5_loto_stability,
    todo5_smoothing_sensitivity,
)

### 5.D — короткий rollout на validation

Локально хорошее ускорение ещё не гарантирует устойчивую ODE. Для каждого
validation-сегмента берутся пять равномерно распределённых стартов и горизонты
`0.25`, `0.5`, `1.0` единицы времени. Интегрирование идёт на настоящих timestamps.

Baseline сохраняет начальную скорость постоянной. Модель проходит rollout gate,
если все интеграции конечны и её macro NRMSE одновременно по координатам и
скоростям ниже baseline на каждом из трёх горизонтов.

In [ ]:
TODO5_ROLLOUT_HORIZONS = (0.25, 0.5, 1.0)
TODO5_ROLLOUT_STARTS_PER_SEGMENT = 5
_todo5_train_x = np.vstack(
    _todo5_arrays(todo5_trajectories, TODO4_TRAIN_SEGMENTS, "x")
)
_todo5_rollout_scale = np.std(_todo5_train_x, axis=0)
_todo5_state_limit = 10 * np.maximum(
    np.max(np.abs(_todo5_train_x), axis=0), 1.0
)

def _todo5_model_rhs(model):
    def _rhs(_time, _state):
        _sindy_rhs = np.asarray(
            model.predict(np.asarray(_state).reshape(1, -1))
        )[0]
        return np.concatenate((_state[4:], _sindy_rhs[4:]))
    return _rhs

def _todo5_rollout(model, x0, t_eval, rtol=1e-9, atol=1e-10):
    def _limit_event(_time, _state):
        return 1 - np.max(np.abs(_state) / _todo5_state_limit)
    _limit_event.terminal = True
    _limit_event.direction = -1
    _solution = solve_ivp(
        _todo5_model_rhs(model),
        (float(t_eval[0]), float(t_eval[-1])),
        np.asarray(x0, dtype=float),
        t_eval=np.asarray(t_eval, dtype=float),
        method="LSODA",
        rtol=rtol,
        atol=atol,
        events=_limit_event,
    )
    _prediction = _solution.y.T
    _ok = bool(
        _solution.success
        and _prediction.shape == (len(t_eval), 8)
        and np.isfinite(_prediction).all()
    )
    return _prediction, _ok, _solution.message

def _todo5_rollout_error(prediction, truth, t_eval):
    _duration = float(t_eval[-1] - t_eval[0])
    _rmse = np.sqrt(
        np.trapezoid(
            (prediction - truth) ** 2, x=t_eval, axis=0
        ) / _duration
    )
    _nrmse = _rmse / _todo5_rollout_scale
    return {
        "position_nrmse": float(np.sqrt(np.mean(_nrmse[:4] ** 2))),
        "velocity_nrmse": float(np.sqrt(np.mean(_nrmse[4:] ** 2))),
        "state_nrmse": float(np.sqrt(np.mean(_nrmse**2))),
    }

def _todo5_validation_rollouts(model, rtol=1e-9, atol=1e-10):
    _rows = []
    _max_horizon = max(TODO5_ROLLOUT_HORIZONS)
    for _segment_id in TODO4_VALIDATION_SEGMENTS:
        _trajectory = todo5_trajectories[_segment_id]
        _time = _trajectory["t"]
        _state = _trajectory["x"]
        _latest_start = _time[-1] - _max_horizon
        _target_start_times = np.linspace(
            _time[0], _latest_start, TODO5_ROLLOUT_STARTS_PER_SEGMENT
        )
        _start_indices = np.unique(
            np.searchsorted(_time, _target_start_times)
        )
        if len(_start_indices) != TODO5_ROLLOUT_STARTS_PER_SEGMENT:
            raise AssertionError("Не получено пять уникальных rollout starts.")
        for _start_number, _start_index in enumerate(_start_indices):
            _x0 = _state[_start_index]
            _t0 = _time[_start_index]
            for _horizon in TODO5_ROLLOUT_HORIZONS:
                _stop = np.searchsorted(
                    _time, _t0 + _horizon, side="right"
                )
                _truth = _state[_start_index:_stop]
                _t_eval = _time[_start_index:_stop] - _t0
                if len(_t_eval) < 3:
                    raise AssertionError("Rollout horizon слишком короткий.")
                _prediction, _ok, _message = _todo5_rollout(
                    model, _x0, _t_eval, rtol=rtol, atol=atol
                )
                _baseline = np.broadcast_to(_x0, _truth.shape).copy()
                _baseline[:, :4] = (
                    _x0[:4] + _t_eval[:, None] * _x0[4:]
                )
                if _ok:
                    _model_error = _todo5_rollout_error(
                        _prediction, _truth, _t_eval
                    )
                else:
                    _model_error = {
                        "position_nrmse": np.inf,
                        "velocity_nrmse": np.inf,
                        "state_nrmse": np.inf,
                    }
                _baseline_error = _todo5_rollout_error(
                    _baseline, _truth, _t_eval
                )
                for _kind, _error, _finite in [
                    ("SINDy", _model_error, _ok),
                    ("constant velocity", _baseline_error, True),
                ]:
                    _rows.append(
                        {
                            "segment_id": _segment_id,
                            "start_number": _start_number,
                            "horizon": _horizon,
                            "kind": _kind,
                            "finite": _finite,
                            **_error,
                            "solver_message": (
                                _message if _kind == "SINDy" else "baseline"
                            ),
                        }
                    )
    return pd.DataFrame(_rows)

todo5_validation_rollout_details = _todo5_validation_rollouts(
    todo5_selected_train_model
)
todo5_validation_rollout_summary = (
    todo5_validation_rollout_details.groupby(
        ["horizon", "kind"], as_index=False
    )
    .agg(
        finite_fraction=("finite", "mean"),
        position_nrmse=("position_nrmse", "mean"),
        velocity_nrmse=("velocity_nrmse", "mean"),
        state_nrmse=("state_nrmse", "mean"),
    )
)
_rollout_pivot = todo5_validation_rollout_summary.pivot(
    index="horizon", columns="kind",
    values=["finite_fraction", "position_nrmse", "velocity_nrmse"],
)
TODO5_ROLLOUT_GATE_PASSED = bool(
    _rollout_pivot["finite_fraction"]["SINDy"].eq(1).all()
    and (
        _rollout_pivot["position_nrmse"]["SINDy"]
        < _rollout_pivot["position_nrmse"]["constant velocity"]
    ).all()
    and (
        _rollout_pivot["velocity_nrmse"]["SINDy"]
        < _rollout_pivot["velocity_nrmse"]["constant velocity"]
    ).all()
)
TODO5_AUGMENTED_VALIDATED = bool(
    TODO5_DERIVATIVE_GATE_PASSED and TODO5_ROLLOUT_GATE_PASSED
)
display(todo5_validation_rollout_summary, _rollout_pivot)
display(
    Markdown(
        f"**Validation gates:** derivative = `{TODO5_DERIVATIVE_GATE_PASSED}`, "
        f"rollout = `{TODO5_ROLLOUT_GATE_PASSED}`, "
        f"augmented validated = **`{TODO5_AUGMENTED_VALIDATED}`**."
    )
)

_solver_sensitivity_rows = []
for _rtol, _atol in [(1e-7, 1e-9), (1e-9, 1e-11)]:
    _details = _todo5_validation_rollouts(
        todo5_selected_train_model, rtol=_rtol, atol=_atol
    )
    _summary = (
        _details.loc[_details["kind"].eq("SINDy")]
        .groupby("horizon", as_index=False)
        .agg(
            finite_fraction=("finite", "mean"),
            position_nrmse=("position_nrmse", "mean"),
            velocity_nrmse=("velocity_nrmse", "mean"),
        )
    )
    for _row in _summary.itertuples(index=False):
        _solver_sensitivity_rows.append(
            {
                "rtol": _rtol,
                "atol": _atol,
                "horizon": _row.horizon,
                "finite_fraction": _row.finite_fraction,
                "position_nrmse": _row.position_nrmse,
                "velocity_nrmse": _row.velocity_nrmse,
            }
        )
todo5_solver_sensitivity = pd.DataFrame(_solver_sensitivity_rows)
display(todo5_solver_sensitivity)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for _kind, _group in todo5_validation_rollout_summary.groupby("kind"):
    axes[0].plot(
        _group["horizon"], _group["position_nrmse"],
        marker="o", label=_kind,
    )
    axes[1].plot(
        _group["horizon"], _group["velocity_nrmse"],
        marker="o", label=_kind,
    )
axes[0].set_title("Validation position rollout")
axes[1].set_title("Validation velocity rollout")
for _axis in axes:
    _axis.set_xlabel("physical horizon")
    _axis.set_ylabel("macro NRMSE")
    _axis.grid(alpha=0.25)
    _axis.legend()
fig.tight_layout()
plt.show()

### 5.E — производные без сглаживания против SG

Предыдущая sensitivity-проверка сравнивала только разные SG-окна. Это не
заменяет контроль полностью несглаженных finite differences. Ниже параллельно
строятся два полных pipeline на одних и тех же строках (`margin = 8`):

1. `raw position → gradient → gradient`;
2. `SG W=11 position → gradient → gradient`.

Для каждого варианта независимо выбирается sparse threshold по тому же
validation-rule, затем выполняется тот же rollout против constant velocity.
Сравнение является development-аудитом; test-сегменты не читаются.

In [ ]:
TODO5_DERIVATIVE_AUDIT_MARGIN = max(
    _todo5_margin(_window) for _window in TODO5_SENSITIVITY_WINDOWS
)

def _todo5_build_raw_trajectories(segment_ids, valid_margin):
    _trajectories = {}
    if valid_margin < 2:
        raise ValueError("Две finite differences требуют margin >= 2.")
    for _segment_id in segment_ids:
        _segment = segmented_df.loc[
            segmented_df["segment_id"].eq(_segment_id),
            ["time", *KINEMATICS_RAW_POSITION_COLUMNS],
        ]
        _time_full = _segment["time"].to_numpy(dtype=float)
        _positions_full = _segment.loc[
            :, list(KINEMATICS_RAW_POSITION_COLUMNS)
        ].to_numpy(dtype=float)
        _velocities_full = np.gradient(
            _positions_full, _time_full, axis=0, edge_order=2
        )
        _accelerations_full = np.gradient(
            _velocities_full, _time_full, axis=0, edge_order=2
        )
        _distance_to_edge = np.minimum(
            np.arange(len(_segment)), np.arange(len(_segment))[::-1]
        )
        _valid = _distance_to_edge >= valid_margin
        _x = np.column_stack(
            (_positions_full[_valid], _velocities_full[_valid])
        )
        _x_dot = np.column_stack(
            (_velocities_full[_valid], _accelerations_full[_valid])
        )
        if not np.isfinite(_x).all() or not np.isfinite(_x_dot).all():
            raise AssertionError(
                f"Raw pipeline nonfinite segment {_segment_id}."
            )
        _time = _time_full[_valid]
        _trajectories[_segment_id] = {
            "x": _x,
            "x_dot": _x_dot,
            "t": _time - _time[0],
            "time_absolute": _time,
            "raw_positions": _positions_full[_valid],
            "source_index": _segment.index.to_numpy()[_valid],
            "margin": valid_margin,
        }
    return _trajectories

todo5_raw_trajectories = _todo5_build_raw_trajectories(
    TODO5_DEVELOPMENT_SEGMENTS, TODO5_DERIVATIVE_AUDIT_MARGIN
)
todo5_sg_common_trajectories = _todo5_build_trajectories(
    TODO5_MAIN_WINDOW, TODO5_DEVELOPMENT_SEGMENTS,
    valid_margin=TODO5_DERIVATIVE_AUDIT_MARGIN,
)
if (set(todo5_raw_trajectories) | set(todo5_sg_common_trajectories)) & set(
    TODO4_TEST_SEGMENTS
):
    raise AssertionError("Derivative audit случайно открыл test.")
for _segment_id in TODO5_DEVELOPMENT_SEGMENTS:
    if not np.array_equal(
        todo5_raw_trajectories[_segment_id]["source_index"],
        todo5_sg_common_trajectories[_segment_id]["source_index"],
    ):
        raise AssertionError("Raw/SG audit использует разные строки.")

def _todo5_select_derivative_pipeline(trajectories):
    _train_x_dot = np.vstack(
        _todo5_arrays(trajectories, TODO4_TRAIN_SEGMENTS, "x_dot")
    )
    _velocity_scale = np.std(_train_x_dot[:, :4], axis=0)
    _acceleration_scale = np.std(_train_x_dot[:, 4:], axis=0)
    if np.any(_acceleration_scale <= np.finfo(float).eps):
        raise AssertionError("Нулевая acceleration scale в audit.")
    _truth = [
        trajectories[_id]["x_dot"][:, 4:]
        for _id in TODO4_VALIDATION_SEGMENTS
    ]
    _train_mean = _train_x_dot[:, 4:].mean(axis=0)

    def _wrap(_acceleration_predictions):
        return [
            np.column_stack(
                (trajectories[_id]["x_dot"][:, :4], _prediction)
            )
            for _id, _prediction in zip(
                TODO4_VALIDATION_SEGMENTS, _acceleration_predictions,
                strict=True,
            )
        ]

    _baseline_summaries = {}
    for _name, _prediction in {
        "zero": [np.zeros_like(_item) for _item in _truth],
        "train mean": [
            np.broadcast_to(_train_mean, _item.shape) for _item in _truth
        ],
    }.items():
        _, _summary = _todo5_time_weighted_score(
            trajectories, TODO4_VALIDATION_SEGMENTS, _wrap(_prediction),
            slice(4, 8), _acceleration_scale,
        )
        _baseline_summaries[_name] = _summary
    _baseline = min(
        _item["macro_nrmse"] for _item in _baseline_summaries.values()
    )

    _models = {}
    _rows = []
    for _threshold in TODO5_THRESHOLD_GRID:
        _model = _todo5_fit(
            _threshold, TODO4_TRAIN_SEGMENTS, trajectories
        )
        _prediction = _model.predict(
            _todo5_arrays(trajectories, TODO4_VALIDATION_SEGMENTS, "x")
        )
        _, _summary = _todo5_time_weighted_score(
            trajectories, TODO4_VALIDATION_SEGMENTS, _prediction,
            slice(4, 8), _acceleration_scale,
        )
        _terms = int((np.abs(_model.coefficients()[4:]) > 1e-12).sum())
        _models[float(_threshold)] = _model
        _rows.append(
            {
                "threshold": float(_threshold),
                "acceleration_macro_nrmse": _summary["macro_nrmse"],
                "acceleration_micro_nrmse": _summary["micro_nrmse"],
                "acceleration_mean_r2": _summary["mean_r2"],
                "acceleration_terms": _terms,
            }
        )
    _results = pd.DataFrame(_rows)
    _results["relative_gain_vs_baseline"] = (
        _baseline - _results["acceleration_macro_nrmse"]
    ) / _baseline
    _nonzero = _results.loc[_results["acceleration_terms"].gt(0)]
    _best_gain = _nonzero["relative_gain_vs_baseline"].max()
    if _best_gain > 0:
        _candidates = _nonzero.loc[
            _nonzero["relative_gain_vs_baseline"].ge(
                TODO5_GAIN_RETENTION * _best_gain
            )
        ]
    else:
        _candidates = _nonzero.nsmallest(1, "acceleration_macro_nrmse")
    _selected = _candidates.sort_values(
        ["acceleration_terms", "threshold", "acceleration_macro_nrmse"],
        ascending=[True, False, True],
    ).iloc[0]
    _threshold = float(_selected["threshold"])
    return {
        "model": _models[_threshold],
        "selected": _selected,
        "results": _results,
        "baseline": _baseline,
        "baseline_summaries": _baseline_summaries,
        "velocity_scale": _velocity_scale,
        "acceleration_scale": _acceleration_scale,
        "derivative_gate": bool(
            _selected["relative_gain_vs_baseline"]
            >= TODO5_MINIMUM_RELATIVE_GAIN
        ),
    }

todo5_derivative_audit_pipelines = {
    "raw finite differences": todo5_raw_trajectories,
    "SG W=11": todo5_sg_common_trajectories,
}
todo5_derivative_audit_fits = {
    _name: _todo5_select_derivative_pipeline(_trajectories)
    for _name, _trajectories in todo5_derivative_audit_pipelines.items()
}

_target_agreement_rows = []
for _segment_id in TODO4_VALIDATION_SEGMENTS:
    _time = todo5_raw_trajectories[_segment_id]["t"]
    _duration = float(_time[-1] - _time[0])
    _raw_acceleration = todo5_raw_trajectories[_segment_id]["x_dot"][:, 4:]
    _sg_acceleration = todo5_sg_common_trajectories[_segment_id]["x_dot"][:, 4:]
    _raw_rms = np.sqrt(
        np.trapezoid(_raw_acceleration**2, x=_time, axis=0) / _duration
    )
    _sg_rms = np.sqrt(
        np.trapezoid(_sg_acceleration**2, x=_time, axis=0) / _duration
    )
    for _column, _component in enumerate(TODO5_ACCELERATION_NAMES):
        _target_agreement_rows.append(
            {
                "segment_id": _segment_id,
                "component": _component,
                "raw_to_sg_rms_ratio": float(
                    _raw_rms[_column] / _sg_rms[_column]
                ),
                "raw_sg_correlation": float(
                    np.corrcoef(
                        _raw_acceleration[:, _column],
                        _sg_acceleration[:, _column],
                    )[0, 1]
                ),
            }
        )
todo5_derivative_target_agreement = pd.DataFrame(_target_agreement_rows)

def _todo5_pipeline_rollouts(model, trajectories):
    _train_x = np.vstack(
        _todo5_arrays(trajectories, TODO4_TRAIN_SEGMENTS, "x")
    )
    _scale = np.maximum(np.std(_train_x, axis=0), np.finfo(float).eps)
    _state_limit = 10 * np.maximum(np.max(np.abs(_train_x), axis=0), 1.0)

    def _error(_prediction, _truth, _time):
        _duration = float(_time[-1] - _time[0])
        _rmse = np.sqrt(
            np.trapezoid(
                (_prediction - _truth) ** 2, x=_time, axis=0
            ) / _duration
        )
        _nrmse = _rmse / _scale
        return {
            "position_nrmse": float(np.sqrt(np.mean(_nrmse[:4] ** 2))),
            "velocity_nrmse": float(np.sqrt(np.mean(_nrmse[4:] ** 2))),
        }

    _rows = []
    for _segment_id in TODO4_VALIDATION_SEGMENTS:
        _trajectory = trajectories[_segment_id]
        _time = _trajectory["t"]
        _state = _trajectory["x"]
        _latest_start = _time[-1] - max(TODO5_ROLLOUT_HORIZONS)
        _starts = np.unique(
            np.searchsorted(
                _time,
                np.linspace(
                    _time[0], _latest_start, TODO5_ROLLOUT_STARTS_PER_SEGMENT
                ),
            )
        )
        if len(_starts) != TODO5_ROLLOUT_STARTS_PER_SEGMENT:
            raise AssertionError("Derivative audit rollout starts mismatch.")
        for _start_number, _start_index in enumerate(_starts):
            _x0 = _state[_start_index]
            _t0 = _time[_start_index]
            for _horizon in TODO5_ROLLOUT_HORIZONS:
                _stop = np.searchsorted(
                    _time, _t0 + _horizon, side="right"
                )
                _truth = _state[_start_index:_stop]
                _t_eval = _time[_start_index:_stop] - _t0

                def _limit_event(_solver_time, _solver_state):
                    return 1 - np.max(np.abs(_solver_state) / _state_limit)
                _limit_event.terminal = True
                _limit_event.direction = -1
                _solution = solve_ivp(
                    _todo5_model_rhs(model),
                    (float(_t_eval[0]), float(_t_eval[-1])),
                    _x0, t_eval=_t_eval, method="LSODA",
                    rtol=1e-9, atol=1e-10, events=_limit_event,
                )
                _model_prediction = _solution.y.T
                _finite = bool(
                    _solution.success
                    and _model_prediction.shape == _truth.shape
                    and np.isfinite(_model_prediction).all()
                )
                _baseline = np.broadcast_to(_x0, _truth.shape).copy()
                _baseline[:, :4] = (
                    _x0[:4] + _t_eval[:, None] * _x0[4:]
                )
                _model_error = (
                    _error(_model_prediction, _truth, _t_eval)
                    if _finite
                    else {"position_nrmse": np.inf, "velocity_nrmse": np.inf}
                )
                for _kind, _prediction, _ok, _errors in [
                    ("SINDy", _model_prediction, _finite, _model_error),
                    ("constant velocity", _baseline, True,
                     _error(_baseline, _truth, _t_eval)),
                ]:
                    _rows.append(
                        {
                            "segment_id": _segment_id,
                            "start_number": _start_number,
                            "horizon": _horizon,
                            "kind": _kind,
                            "finite": _ok,
                            **_errors,
                        }
                    )
    return pd.DataFrame(_rows)

_audit_rollout_frames = []
for _pipeline_name, _trajectories in todo5_derivative_audit_pipelines.items():
    _details = _todo5_pipeline_rollouts(
        todo5_derivative_audit_fits[_pipeline_name]["model"],
        _trajectories,
    )
    _details.insert(0, "pipeline", _pipeline_name)
    _audit_rollout_frames.append(_details)
todo5_derivative_audit_rollout_details = pd.concat(
    _audit_rollout_frames, ignore_index=True
)
todo5_derivative_audit_rollout_summary = (
    todo5_derivative_audit_rollout_details.groupby(
        ["pipeline", "horizon", "kind"], as_index=False
    ).agg(
        finite_fraction=("finite", "mean"),
        position_nrmse=("position_nrmse", "mean"),
        velocity_nrmse=("velocity_nrmse", "mean"),
    )
)

_audit_selection_rows = []
_audit_rollout_ratio_rows = []
for _pipeline_name, _fit in todo5_derivative_audit_fits.items():
    _summary = todo5_derivative_audit_rollout_summary.loc[
        todo5_derivative_audit_rollout_summary["pipeline"].eq(_pipeline_name)
    ]
    _pivot = _summary.pivot(index="horizon", columns="kind")
    _rollout_gate = bool(
        _pivot["finite_fraction"]["SINDy"].eq(1).all()
        and (
            _pivot["position_nrmse"]["SINDy"]
            < _pivot["position_nrmse"]["constant velocity"]
        ).all()
        and (
            _pivot["velocity_nrmse"]["SINDy"]
            < _pivot["velocity_nrmse"]["constant velocity"]
        ).all()
    )
    _selected = _fit["selected"]
    _audit_selection_rows.append(
        {
            "pipeline": _pipeline_name,
            "threshold": _selected["threshold"],
            "acceleration_terms": int(_selected["acceleration_terms"]),
            "baseline_macro_nrmse": _fit["baseline"],
            "acceleration_macro_nrmse": _selected["acceleration_macro_nrmse"],
            "acceleration_mean_r2": _selected["acceleration_mean_r2"],
            "relative_gain_vs_baseline": _selected["relative_gain_vs_baseline"],
            "derivative_gate": _fit["derivative_gate"],
            "rollout_gate": _rollout_gate,
            "validated": bool(_fit["derivative_gate"] and _rollout_gate),
        }
    )
    for _horizon in TODO5_ROLLOUT_HORIZONS:
        _audit_rollout_ratio_rows.append(
            {
                "pipeline": _pipeline_name,
                "horizon": _horizon,
                "position_error_ratio_vs_constant_velocity": float(
                    _pivot.loc[_horizon, ("position_nrmse", "SINDy")]
                    / _pivot.loc[
                        _horizon, ("position_nrmse", "constant velocity")
                    ]
                ),
                "velocity_error_ratio_vs_constant_velocity": float(
                    _pivot.loc[_horizon, ("velocity_nrmse", "SINDy")]
                    / _pivot.loc[
                        _horizon, ("velocity_nrmse", "constant velocity")
                    ]
                ),
            }
        )
todo5_derivative_audit_selection = pd.DataFrame(
    _audit_selection_rows
).set_index("pipeline")
todo5_derivative_audit_rollout_ratios = pd.DataFrame(
    _audit_rollout_ratio_rows
)
TODO5_RAW_DERIVATIVE_AUDIT_VALIDATED = bool(
    todo5_derivative_audit_selection.loc[
        "raw finite differences", "validated"
    ]
)
todo5_derivative_audit_equations = pd.DataFrame(
    {
        _pipeline_name: _fit["model"].equations(precision=6)[4:]
        for _pipeline_name, _fit in todo5_derivative_audit_fits.items()
    },
    index=TODO5_ACCELERATION_NAMES,
)

display(
    todo5_derivative_audit_selection,
    todo5_derivative_audit_equations,
    todo5_derivative_target_agreement,
    todo5_derivative_audit_rollout_summary,
    todo5_derivative_audit_rollout_ratios,
)
_raw_row = todo5_derivative_audit_selection.loc["raw finite differences"]
_sg_row = todo5_derivative_audit_selection.loc["SG W=11"]
display(
    Markdown(
        f"""
        **Raw-vs-SG verdict.** Raw acceleration имеет в среднем
        `{todo5_derivative_target_agreement['raw_to_sg_rms_ratio'].mean():.2f}×`
        больший RMS, а средняя корреляция raw и SG acceleration равна
        `{todo5_derivative_target_agreement['raw_sg_correlation'].mean():.3f}`.
        
        Raw pipeline: derivative gain `{100 * _raw_row['relative_gain_vs_baseline']:.2f}%`,
        derivative gate `{bool(_raw_row['derivative_gate'])}`, rollout gate
        `{bool(_raw_row['rollout_gate'])}`. SG `W=11` на тех же строках:
        derivative gain `{100 * _sg_row['relative_gain_vs_baseline']:.2f}%`,
        derivative gate `{bool(_sg_row['derivative_gate'])}`, rollout gate
        `{bool(_sg_row['rollout_gate'])}`.
        
        Ни один результат этого audit не открывает test автоматически.
        """
    )
)

### 5.F — freeze decision до test

Test можно открыть только для кандидата, прошедшего одновременно derivative и
rollout gates. Если локальные ускорения лучше baseline, но интегрированная модель
хуже constant velocity, конфигурация фиксируется как `rejected_on_validation`,
а test остаётся неиспользованным для следующей гипотезы.

In [ ]:
_todo5_library_terms = todo5_selected_train_model.get_feature_names()
_todo5_coefficients = todo5_selected_train_model.coefficients()[4:]
todo5_acceleration_equations = pd.Series(
    todo5_selected_train_model.equations(precision=6)[4:],
    index=TODO5_ACCELERATION_NAMES,
    name="right-hand side",
)
_todo5_coefficient_rows = []
for _equation_index, _equation_name in enumerate(
    TODO5_ACCELERATION_NAMES
):
    for _term_index, _term_name in enumerate(_todo5_library_terms):
        _coefficient = _todo5_coefficients[
            _equation_index, _term_index
        ]
        if abs(_coefficient) > 1e-12:
            _todo5_coefficient_rows.append(
                {
                    "equation": _equation_name,
                    "term": _term_name,
                    "coefficient": float(_coefficient),
                }
            )
todo5_acceleration_coefficients = pd.DataFrame(
    _todo5_coefficient_rows
)

TODO5_STATUS = (
    "validated_before_test"
    if TODO5_AUGMENTED_VALIDATED
    else "rejected_on_validation"
)
TODO5_CONFIG_FROZEN = True
TODO5_TEST_OPENED = False
todo5_freeze_manifest = pd.Series(
    {
        "status": TODO5_STATUS,
        "window": TODO5_MAIN_WINDOW,
        "polyorder": KINEMATICS_POLYORDER,
        "acceleration margin per edge": _todo5_margin(TODO5_MAIN_WINDOW),
        "state dimension": len(TODO5_FEATURE_NAMES),
        "library degree": 2,
        "alpha": TODO5_ALPHA,
        "threshold": TODO5_SELECTED_THRESHOLD,
        "train segments": TODO4_TRAIN_SEGMENTS,
        "validation segments": TODO4_VALIDATION_SEGMENTS,
        "reserved test segments": TODO4_TEST_SEGMENTS,
        "rollout horizons": TODO5_ROLLOUT_HORIZONS,
        "raw-vs-SG common margin": TODO5_DERIVATIVE_AUDIT_MARGIN,
        "raw derivative audit validated": TODO5_RAW_DERIVATIVE_AUDIT_VALIDATED,
        "test opened": TODO5_TEST_OPENED,
    },
    name="frozen value",
)

_model_rollout = todo5_validation_rollout_summary.loc[
    todo5_validation_rollout_summary["kind"].eq("SINDy")
].set_index("horizon")
_baseline_rollout = todo5_validation_rollout_summary.loc[
    todo5_validation_rollout_summary["kind"].eq("constant velocity")
].set_index("horizon")
display(
    todo5_freeze_manifest.to_frame(),
    todo5_acceleration_equations.to_frame(),
    todo5_acceleration_coefficients,
)
display(
    Markdown(
        f"""
        **TODO 5 validation result:** `{TODO5_STATUS}`.

        Локальная acceleration model при `threshold = {TODO5_SELECTED_THRESHOLD:.6g}`
        получила macro NRMSE `{todo5_selected_row['acceleration_macro_nrmse']:.4f}`
        против baseline `{_todo5_reference_baseline:.4f}`: gain
        `{100 * todo5_selected_row['relative_gain_vs_baseline']:.2f}%`,
        mean R² `{todo5_selected_row['acceleration_mean_r2']:.3f}`.

        Однако validation rollout хуже constant velocity на всех горизонтах.
        На `H = 0.5` position NRMSE: SINDy
        `{_model_rollout.loc[0.5, 'position_nrmse']:.4f}` против
        `{_baseline_rollout.loc[0.5, 'position_nrmse']:.4f}`; velocity NRMSE:
        `{_model_rollout.loc[0.5, 'velocity_nrmse']:.4f}` против
        `{_baseline_rollout.loc[0.5, 'velocity_nrmse']:.4f}`.

        Поэтому derivative gate = `{TODO5_DERIVATIVE_GATE_PASSED}`, rollout gate =
        `{TODO5_ROLLOUT_GATE_PASSED}`, общий verdict =
        **`{TODO5_AUGMENTED_VALIDATED}`**. Test-сегменты
        `{TODO4_TEST_SEGMENTS}` не материализованы и остаются доступными для
        следующей, заранее зафиксированной гипотезы.
        """
    )
)

### Промежуточный вывод TODO 5

Полная проверка теперь включает и SG-производные, и raw finite differences.
Raw-ускорения почти втрое крупнее SG-ускорений и слабо с ними коррелируют,
поэтому локальный fit raw acceleration нельзя автоматически считать более
физичным. При этом raw-модель улучшила rollout наблюдаемых координат относительно
constant velocity на всех трёх горизонтах, но проиграла по вычисленным raw-
скоростям; SG-модель проиграла по обеим группам. Это делает raw-ветку полезным
exploratory сигналом, но не validated результатом: правило оценки нельзя менять
задним числом после просмотра validation. Test остаётся закрытым. Следующий
эксперимент должен заранее объявить координатный rollout primary-метрикой и
избежать нестабильной второй производной через weak/integral или direct rollout fit.

## TODO 6 — coordinate-first модель вращения скорости

- [x] До запуска зафиксировать coordinate rollout как primary target.
- [x] Оценивать начальную скорость причинно, только по прошлым точкам.
- [x] Обучить двухпараметрическую модель непосредственно по train-координатам.
- [x] Проверить validation gate, LOTO и чувствительность causal window.
- [x] Зафиксировать conditional test rule; первый run выявил отсутствовавший
  boundary guard, поэтому показанный тогда test объявлен скомпрометированным.

Модель намеренно ограничена найденной в TODO 5 структурой:

`dot p_i = v_i`, `dot v_i = omega_i J v_i`,

где `J(vx, vy) = (vy, -vx)`. Параметров всего два: `omega_1` и
`omega_2`. Обучающая функция — ошибка интегрированных raw-координат, а не
ошибка шумной второй производной.

In [ ]:
TODO6_CAUSAL_WINDOW = 11
TODO6_CAUSAL_POLYORDER = 3
TODO6_SENSITIVITY_WINDOWS = (9, 11, 13)
TODO6_COMMON_HISTORY = max(TODO6_SENSITIVITY_WINDOWS)
TODO6_TRAIN_STARTS_PER_SEGMENT = 40
TODO6_EVAL_STARTS_PER_SEGMENT = 5
TODO6_HORIZONS = (0.25, 0.5, 1.0)
TODO6_OMEGA_BOUNDS = (-30.0, 30.0)
TODO6_OMEGA_GRID_SIZE = 241
TODO6_MIN_POSITION_GAIN = 0.05
TODO6_MAX_SEGMENT_ERROR_RATIO = 1.10
TODO6_MAX_LOTO_COEFFICIENT_OF_VARIATION = 0.20

def _todo6_build_trajectories(segment_ids):
    _trajectories = {}
    for _segment_id in segment_ids:
        _segment = segmented_df.loc[
            segmented_df["segment_id"].eq(_segment_id),
            ["time", *KINEMATICS_RAW_POSITION_COLUMNS],
        ]
        _time = _segment["time"].to_numpy(dtype=float)
        _positions = _segment.loc[
            :, list(KINEMATICS_RAW_POSITION_COLUMNS)
        ].to_numpy(dtype=float)
        _positions_sg = savgol_filter(
            _positions, window_length=TODO6_CAUSAL_WINDOW,
            polyorder=KINEMATICS_POLYORDER, axis=0, mode="interp",
        )
        _velocity_sg = np.gradient(
            _positions_sg, _time, axis=0, edge_order=2
        )
        if not np.all(np.diff(_time) > 0):
            raise AssertionError(f"Non-increasing time segment {_segment_id}.")
        _trajectories[_segment_id] = {
            "t": _time,
            "position": _positions,
            "velocity_sg": _velocity_sg,
            "source_index": _segment.index.to_numpy(),
        }
    return _trajectories

def _todo6_causal_velocity(time, positions, start_index, window):
    _first = start_index - window + 1
    if _first < 0:
        raise ValueError("Недостаточно прошлых точек для causal velocity.")
    _time_history = time[_first : start_index + 1] - time[start_index]
    _position_history = positions[_first : start_index + 1]
    _time_scale = max(abs(float(_time_history[0])), np.finfo(float).eps)
    _u = _time_history / _time_scale
    _design = np.vander(
        _u, N=TODO6_CAUSAL_POLYORDER + 1, increasing=True
    )
    _coefficients, *_ = np.linalg.lstsq(
        _design, _position_history, rcond=None
    )
    return _coefficients[1] / _time_scale

def _todo6_build_windows(
    trajectories, segment_ids, starts_per_segment, causal_window
):
    _windows = []
    _history_index = TODO6_COMMON_HISTORY - 1
    _max_horizon = max(TODO6_HORIZONS)
    for _segment_id in segment_ids:
        _trajectory = trajectories[_segment_id]
        _time = _trajectory["t"]
        _positions = _trajectory["position"]
        _latest_time = _time[-1] - _max_horizon
        if _time[_history_index] >= _latest_time:
            raise AssertionError(f"Segment {_segment_id} слишком короткий.")
        _targets = np.linspace(
            _time[_history_index], _latest_time, starts_per_segment
        )
        _starts = np.unique(np.searchsorted(_time, _targets))
        if len(_starts) != starts_per_segment:
            raise AssertionError(
                f"Не получено {starts_per_segment} starts segment {_segment_id}."
            )
        for _start_number, _start_index in enumerate(_starts):
            _stop = np.searchsorted(
                _time, _time[_start_index] + _max_horizon, side="right"
            )
            _t_eval = _time[_start_index:_stop] - _time[_start_index]
            if len(_t_eval) < 3:
                raise AssertionError("TODO6 rollout window слишком короткий.")
            _velocity0 = _todo6_causal_velocity(
                _time, _positions, _start_index, causal_window
            )
            _velocity_truth = _trajectory["velocity_sg"][_start_index:_stop]
            if not np.isfinite(_velocity_truth).all():
                raise AssertionError("SG velocity diagnostic nonfinite.")
            _windows.append(
                {
                    "segment_id": _segment_id,
                    "start_number": _start_number,
                    "start_index": int(_start_index),
                    "t": _t_eval,
                    "position_truth": _positions[_start_index:_stop],
                    "velocity_truth_sg": _velocity_truth,
                    "position0": _positions[_start_index].copy(),
                    "velocity0": _velocity0,
                }
            )
    return _windows

def _todo6_predict_rotation(omegas, position0, velocity0, t_eval):
    _time = np.asarray(t_eval, dtype=float)
    _positions = np.empty((_time.size, 4), dtype=float)
    _velocities = np.empty((_time.size, 4), dtype=float)
    for _object_index, _omega in enumerate(np.asarray(omegas, dtype=float)):
        _columns = slice(2 * _object_index, 2 * _object_index + 2)
        _p0 = np.asarray(position0[_columns], dtype=float)
        _v0 = np.asarray(velocity0[_columns], dtype=float)
        if abs(_omega) <= 1e-10:
            _positions[:, _columns] = _p0 + _time[:, None] * _v0
            _velocities[:, _columns] = _v0
            continue
        _angle = _omega * _time
        _cosine = np.cos(_angle)
        _sine = np.sin(_angle)
        _one_minus_cosine = 1 - _cosine
        _velocities[:, 2 * _object_index] = (
            _cosine * _v0[0] + _sine * _v0[1]
        )
        _velocities[:, 2 * _object_index + 1] = (
            -_sine * _v0[0] + _cosine * _v0[1]
        )
        _positions[:, 2 * _object_index] = (
            _p0[0] + _sine / _omega * _v0[0]
            + _one_minus_cosine / _omega * _v0[1]
        )
        _positions[:, 2 * _object_index + 1] = (
            _p0[1] - _one_minus_cosine / _omega * _v0[0]
            + _sine / _omega * _v0[1]
        )
    return _positions, _velocities

def _todo6_position_scale(trajectories, segment_ids):
    return np.maximum(
        np.std(
            np.vstack([trajectories[_id]["position"] for _id in segment_ids]),
            axis=0,
        ),
        np.finfo(float).eps,
    )

def _todo6_velocity_scale(trajectories, segment_ids):
    _values = np.vstack(
        [trajectories[_id]["velocity_sg"] for _id in segment_ids]
    )
    _values = _values[np.isfinite(_values).all(axis=1)]
    return np.maximum(np.std(_values, axis=0), np.finfo(float).eps)

def _todo6_prepare_object_fit(windows, object_index, position_scale):
    _columns = slice(2 * object_index, 2 * object_index + 2)
    _time_parts = []
    _position0_parts = []
    _velocity0_parts = []
    _truth_parts = []
    _weight_parts = []
    for _window in windows:
        _time = _window["t"]
        _weights = np.empty_like(_time)
        _weights[0] = (_time[1] - _time[0]) / 2
        _weights[-1] = (_time[-1] - _time[-2]) / 2
        _weights[1:-1] = (_time[2:] - _time[:-2]) / 2
        _weights /= float(_time[-1] - _time[0])
        _time_parts.append(_time)
        _position0_parts.append(
            np.broadcast_to(
                _window["position0"][_columns], (_time.size, 2)
            )
        )
        _velocity0_parts.append(
            np.broadcast_to(
                _window["velocity0"][_columns], (_time.size, 2)
            )
        )
        _truth_parts.append(_window["position_truth"][:, _columns])
        _weight_parts.append(_weights / len(windows))
    return {
        "time": np.concatenate(_time_parts),
        "position0": np.vstack(_position0_parts),
        "velocity0": np.vstack(_velocity0_parts),
        "truth": np.vstack(_truth_parts),
        "weights": np.concatenate(_weight_parts),
        "scale": position_scale[_columns],
    }

def _todo6_objective(omega, prepared):
    _time = prepared["time"]
    _position0 = prepared["position0"]
    _velocity0 = prepared["velocity0"]
    if abs(omega) <= 1e-10:
        _prediction = _position0 + _time[:, None] * _velocity0
    else:
        _angle = float(omega) * _time
        _sine = np.sin(_angle)
        _one_minus_cosine = 1 - np.cos(_angle)
        _prediction = np.empty_like(_position0)
        _prediction[:, 0] = (
            _position0[:, 0] + _sine / omega * _velocity0[:, 0]
            + _one_minus_cosine / omega * _velocity0[:, 1]
        )
        _prediction[:, 1] = (
            _position0[:, 1] - _one_minus_cosine / omega * _velocity0[:, 0]
            + _sine / omega * _velocity0[:, 1]
        )
    _scaled_squared_error = (
        (_prediction - prepared["truth"]) / prepared["scale"]
    ) ** 2
    return float(
        np.sum(prepared["weights"][:, None] * _scaled_squared_error) / 2
    )

def _todo6_fit_omegas(windows, position_scale):
    _grid = np.linspace(
        TODO6_OMEGA_BOUNDS[0], TODO6_OMEGA_BOUNDS[1],
        TODO6_OMEGA_GRID_SIZE,
    )
    _omegas = []
    _fit_rows = []
    for _object_index in range(2):
        _prepared = _todo6_prepare_object_fit(
            windows, _object_index, position_scale
        )
        _values = np.array(
            [
                _todo6_objective(_omega, _prepared)
                for _omega in _grid
            ]
        )
        _best_index = int(np.argmin(_values))
        _left = _grid[max(0, _best_index - 1)]
        _right = _grid[min(len(_grid) - 1, _best_index + 1)]
        if _left == _right:
            _omega = float(_grid[_best_index])
            _loss = float(_values[_best_index])
            _success = False
        else:
            _result = minimize_scalar(
                _todo6_objective,
                bounds=(float(_left), float(_right)),
                args=(_prepared,),
                method="bounded",
                options={"xatol": 1e-9},
            )
            _omega = float(_result.x)
            _loss = float(_result.fun)
            _success = bool(_result.success)
        _omegas.append(_omega)
        _fit_rows.append(
            {
                "object": _object_index + 1,
                "omega": _omega,
                "train_coordinate_mse": _loss,
                "optimizer_success": _success,
                "grid_boundary_selected": bool(
                    _best_index in (0, len(_grid) - 1)
                ),
            }
        )
    return np.asarray(_omegas), pd.DataFrame(_fit_rows)

TODO6_DEVELOPMENT_SEGMENTS = (
    *TODO4_TRAIN_SEGMENTS, *TODO4_VALIDATION_SEGMENTS
)
todo6_development_trajectories = _todo6_build_trajectories(
    TODO6_DEVELOPMENT_SEGMENTS
)
if set(todo6_development_trajectories) & set(TODO4_TEST_SEGMENTS):
    raise AssertionError("TODO6 открыл test до validation gate.")
todo6_train_position_scale = _todo6_position_scale(
    todo6_development_trajectories, TODO4_TRAIN_SEGMENTS
)
todo6_train_velocity_scale = _todo6_velocity_scale(
    todo6_development_trajectories, TODO4_TRAIN_SEGMENTS
)
todo6_train_windows = _todo6_build_windows(
    todo6_development_trajectories, TODO4_TRAIN_SEGMENTS,
    TODO6_TRAIN_STARTS_PER_SEGMENT, TODO6_CAUSAL_WINDOW,
)
todo6_validation_windows = _todo6_build_windows(
    todo6_development_trajectories, TODO4_VALIDATION_SEGMENTS,
    TODO6_EVAL_STARTS_PER_SEGMENT, TODO6_CAUSAL_WINDOW,
)
display(
    pd.Series(
        {
            "causal window": TODO6_CAUSAL_WINDOW,
            "causal polyorder": TODO6_CAUSAL_POLYORDER,
            "train windows": len(todo6_train_windows),
            "validation windows": len(todo6_validation_windows),
            "horizons": TODO6_HORIZONS,
            "omega bounds": TODO6_OMEGA_BOUNDS,
            "test materialized": False,
        },
        name="pre-registered coordinate-first protocol",
    ).to_frame()
)

### 6.A — direct coordinate fit и validation gate

Начальная скорость в каждом rollout start получается локальным полиномом
третьей степени по последним 11 точкам, включая текущую, но без будущих
наблюдений. Тот же initial state используется моделью и baseline.

Gate требует, чтобы модель была конечной, уменьшала macro position NRMSE минимум
на 5% относительно более сильного из constant-velocity и constant-position
baseline на каждом горизонте и не проигрывала ему более чем на 10% ни на одном
отдельном validation-сегменте. SG-скорости оцениваются только
как secondary diagnostic и в acceptance не входят.

In [ ]:
def _todo6_nrmse(prediction, truth, time, scale):
    _duration = float(time[-1] - time[0])
    _rmse = np.sqrt(
        np.trapezoid((prediction - truth) ** 2, x=time, axis=0)
        / _duration
    )
    _components = _rmse / scale
    return (
        float(np.sqrt(np.mean(_components**2))),
        float(np.sqrt(np.mean(_components[:2] ** 2))),
        float(np.sqrt(np.mean(_components[2:] ** 2))),
    )

def _todo6_evaluate_windows(
    omegas, windows, position_scale, velocity_scale
):
    _rows = []
    for _window in windows:
        for _horizon in TODO6_HORIZONS:
            _stop = np.searchsorted(
                _window["t"], _horizon, side="right"
            )
            _time = _window["t"][:_stop]
            _position_truth = _window["position_truth"][:_stop]
            _velocity_truth = _window["velocity_truth_sg"][:_stop]
            _model_position, _model_velocity = _todo6_predict_rotation(
                omegas, _window["position0"], _window["velocity0"],
                _time,
            )
            _baseline_position = (
                _window["position0"]
                + _time[:, None] * _window["velocity0"]
            )
            _baseline_velocity = np.broadcast_to(
                _window["velocity0"], _velocity_truth.shape
            )
            _static_position = np.broadcast_to(
                _window["position0"], _position_truth.shape
            )
            _static_velocity = np.zeros_like(_velocity_truth)
            for _kind, _position, _velocity in [
                ("rotation model", _model_position, _model_velocity),
                ("constant velocity", _baseline_position, _baseline_velocity),
                ("constant position", _static_position, _static_velocity),
            ]:
                _position_scores = _todo6_nrmse(
                    _position, _position_truth, _time, position_scale
                )
                _velocity_scores = _todo6_nrmse(
                    _velocity, _velocity_truth, _time, velocity_scale
                )
                _rows.append(
                    {
                        "segment_id": _window["segment_id"],
                        "start_number": _window["start_number"],
                        "horizon": _horizon,
                        "kind": _kind,
                        "finite": bool(
                            np.isfinite(_position).all()
                            and np.isfinite(_velocity).all()
                        ),
                        "position_nrmse": _position_scores[0],
                        "object1_position_nrmse": _position_scores[1],
                        "object2_position_nrmse": _position_scores[2],
                        "velocity_sg_nrmse": _velocity_scores[0],
                    }
                )
    return pd.DataFrame(_rows)

def _todo6_summarize(details):
    _macro = (
        details.groupby(["horizon", "kind"], as_index=False)
        .agg(
            finite_fraction=("finite", "mean"),
            position_nrmse=("position_nrmse", "mean"),
            object1_position_nrmse=("object1_position_nrmse", "mean"),
            object2_position_nrmse=("object2_position_nrmse", "mean"),
            velocity_sg_nrmse=("velocity_sg_nrmse", "mean"),
        )
    )
    _per_segment = (
        details.groupby(
            ["segment_id", "horizon", "kind"], as_index=False
        ).agg(
            position_nrmse=("position_nrmse", "mean"),
            velocity_sg_nrmse=("velocity_sg_nrmse", "mean"),
        )
    )
    return _macro, _per_segment

def _todo6_coordinate_gate(macro_summary, per_segment_summary):
    _macro_pivot = macro_summary.pivot(
        index="horizon", columns="kind", values=[
            "finite_fraction", "position_nrmse", "velocity_sg_nrmse"
        ]
    )
    _segment_pivot = per_segment_summary.pivot(
        index=["segment_id", "horizon"], columns="kind",
        values="position_nrmse",
    )
    _macro_baseline = _macro_pivot["position_nrmse"][
        ["constant velocity", "constant position"]
    ].min(axis=1)
    _segment_baseline = _segment_pivot[
        ["constant velocity", "constant position"]
    ].min(axis=1)
    _macro_gain = 1 - (
        _macro_pivot["position_nrmse"]["rotation model"]
        / _macro_baseline
    )
    _segment_ratio = _segment_pivot["rotation model"] / _segment_baseline
    _gate = bool(
        _macro_pivot["finite_fraction"]["rotation model"].eq(1).all()
        and _macro_gain.ge(TODO6_MIN_POSITION_GAIN).all()
        and _segment_ratio.le(TODO6_MAX_SEGMENT_ERROR_RATIO).all()
    )
    return _gate, _macro_gain, _segment_ratio, _macro_pivot

TODO6_TRAIN_OMEGAS, todo6_train_fit = _todo6_fit_omegas(
    todo6_train_windows, todo6_train_position_scale
)
TODO6_BOUNDARY_GUARD_PASSED = bool(
    not todo6_train_fit["grid_boundary_selected"].any()
)
todo6_validation_details = _todo6_evaluate_windows(
    TODO6_TRAIN_OMEGAS, todo6_validation_windows,
    todo6_train_position_scale, todo6_train_velocity_scale,
)
(todo6_validation_summary,
 todo6_validation_per_segment) = _todo6_summarize(
    todo6_validation_details
)
(TODO6_COORDINATE_GATE_PASSED,
 todo6_validation_macro_gain,
 todo6_validation_segment_ratio,
 todo6_validation_pivot) = _todo6_coordinate_gate(
    todo6_validation_summary, todo6_validation_per_segment
)

display(
    todo6_train_fit,
    todo6_validation_summary,
    todo6_validation_macro_gain.rename("position gain").to_frame(),
    todo6_validation_segment_ratio.rename("model / baseline").to_frame(),
)
display(
    Markdown(
        f"**Coordinate validation gate:** `{TODO6_COORDINATE_GATE_PASSED}`; "
        f"boundary guard: `{TODO6_BOUNDARY_GUARD_PASSED}`; "
        f"omegas = `({TODO6_TRAIN_OMEGAS[0]:.6f}, "
        f"{TODO6_TRAIN_OMEGAS[1]:.6f})`."
    )
)

### 6.B — устойчивость параметров

Для LOTO каждый train-сегмент по очереди исключается целиком. Stability gate
требует сохранения знаков обеих частот и coefficient of variation не выше
20%. Отдельно causal velocity и весь fit повторяются для окон `9/11/13`;
эта таблица является sensitivity diagnostic и не используется для выбора нового
окна после просмотра validation.

In [ ]:
_loto_rows = []
for _held_out in TODO4_TRAIN_SEGMENTS:
    _remaining = tuple(
        _id for _id in TODO4_TRAIN_SEGMENTS if _id != _held_out
    )
    _windows = [
        _window for _window in todo6_train_windows
        if _window["segment_id"] in _remaining
    ]
    _scale = _todo6_position_scale(
        todo6_development_trajectories, _remaining
    )
    _omegas, _fit = _todo6_fit_omegas(_windows, _scale)
    _loto_rows.append(
        {
            "held_out_train_segment": _held_out,
            "omega1": _omegas[0],
            "omega2": _omegas[1],
            "omega1_relative_change": (
                (_omegas[0] - TODO6_TRAIN_OMEGAS[0])
                / abs(TODO6_TRAIN_OMEGAS[0])
            ),
            "omega2_relative_change": (
                (_omegas[1] - TODO6_TRAIN_OMEGAS[1])
                / abs(TODO6_TRAIN_OMEGAS[1])
            ),
            "boundary_selected": bool(
                _fit["grid_boundary_selected"].any()
            ),
        }
    )
todo6_loto_stability = pd.DataFrame(_loto_rows)
todo6_loto_coefficient_of_variation = pd.Series(
    {
        "omega1": (
            todo6_loto_stability["omega1"].std(ddof=1)
            / abs(todo6_loto_stability["omega1"].mean())
        ),
        "omega2": (
            todo6_loto_stability["omega2"].std(ddof=1)
            / abs(todo6_loto_stability["omega2"].mean())
        ),
    },
    name="LOTO coefficient of variation",
)
_loto_signs_stable = bool(
    (np.sign(todo6_loto_stability["omega1"]) == np.sign(TODO6_TRAIN_OMEGAS[0])).all()
    and (np.sign(todo6_loto_stability["omega2"]) == np.sign(TODO6_TRAIN_OMEGAS[1])).all()
)
TODO6_STABILITY_GATE_PASSED = bool(
    _loto_signs_stable
    and not todo6_loto_stability["boundary_selected"].any()
    and todo6_loto_coefficient_of_variation.le(
        TODO6_MAX_LOTO_COEFFICIENT_OF_VARIATION
    ).all()
)

_sensitivity_rows = []
_primary_start_contract = [
    (_window["segment_id"], _window["start_index"])
    for _window in todo6_validation_windows
]
for _causal_window in TODO6_SENSITIVITY_WINDOWS:
    _train_windows = _todo6_build_windows(
        todo6_development_trajectories, TODO4_TRAIN_SEGMENTS,
        TODO6_TRAIN_STARTS_PER_SEGMENT, _causal_window,
    )
    _validation_windows = _todo6_build_windows(
        todo6_development_trajectories, TODO4_VALIDATION_SEGMENTS,
        TODO6_EVAL_STARTS_PER_SEGMENT, _causal_window,
    )
    if [
        (_window["segment_id"], _window["start_index"])
        for _window in _validation_windows
    ] != _primary_start_contract:
        raise AssertionError("Sensitivity использует другие rollout starts.")
    _omegas, _fit = _todo6_fit_omegas(
        _train_windows, todo6_train_position_scale
    )
    _details = _todo6_evaluate_windows(
        _omegas, _validation_windows, todo6_train_position_scale,
        todo6_train_velocity_scale,
    )
    _summary, _per_segment = _todo6_summarize(_details)
    _gate, _macro_gain, _segment_ratio, _ = _todo6_coordinate_gate(
        _summary, _per_segment
    )
    _sensitivity_rows.append(
        {
            "causal_window": _causal_window,
            "omega1": _omegas[0],
            "omega2": _omegas[1],
            "minimum_macro_position_gain": float(_macro_gain.min()),
            "maximum_segment_error_ratio": float(_segment_ratio.max()),
            "coordinate_gate": _gate,
            "boundary_selected": bool(
                _fit["grid_boundary_selected"].any()
            ),
        }
    )
todo6_causal_window_sensitivity = pd.DataFrame(_sensitivity_rows)
TODO6_VALIDATED_BEFORE_TEST = bool(
    TODO6_COORDINATE_GATE_PASSED
    and TODO6_STABILITY_GATE_PASSED
    and TODO6_BOUNDARY_GUARD_PASSED
)

display(
    todo6_loto_stability,
    todo6_loto_coefficient_of_variation.to_frame(),
    todo6_causal_window_sensitivity,
)
display(
    Markdown(
        f"**Pre-test decision:** coordinate gate = "
        f"`{TODO6_COORDINATE_GATE_PASSED}`, stability gate = "
        f"`{TODO6_STABILITY_GATE_PASSED}`, boundary guard = "
        f"`{TODO6_BOUNDARY_GUARD_PASSED}`, validated before test = "
        f"**`{TODO6_VALIDATED_BEFORE_TEST}`**."
    )
)

### 6.C — условный freeze и единственный test

Если оба development gate пройдены, спецификация модели и evaluation protocol
считаются замороженными. Параметры один раз переоцениваются на train+validation,
после чего test открывается и оценивается без дальнейшей настройки. Если хотя бы
один gate не пройден, test остаётся нематериализованным. Первый автоматический
run обнаружил, что частоты упёрлись в границы поиска, но boundary guard ещё не
входил в условие. Поэтому одна test-таблица была показана ошибочно; эти сегменты
больше не считаются pristine holdout и результат того run не является validation.

In [ ]:
TODO6_CONFIG_FROZEN = True
TODO6_TEST_PREVIOUSLY_EXPOSED = True
TODO6_TEST_HOLDOUT_PRISTINE = False
TODO6_TEST_OPENED = bool(
    TODO6_VALIDATED_BEFORE_TEST and TODO6_TEST_HOLDOUT_PRISTINE
)
TODO6_TEST_GATE_PASSED = False
TODO6_REFIT_OMEGAS = None
todo6_test_summary = None
todo6_test_per_segment = None
todo6_test_macro_gain = None
todo6_test_segment_ratio = None

if TODO6_TEST_OPENED:
    _refit_segment_ids = TODO6_DEVELOPMENT_SEGMENTS
    _refit_windows = _todo6_build_windows(
        todo6_development_trajectories, _refit_segment_ids,
        TODO6_TRAIN_STARTS_PER_SEGMENT, TODO6_CAUSAL_WINDOW,
    )
    _refit_position_scale = _todo6_position_scale(
        todo6_development_trajectories, _refit_segment_ids
    )
    _refit_velocity_scale = _todo6_velocity_scale(
        todo6_development_trajectories, _refit_segment_ids
    )
    TODO6_REFIT_OMEGAS, todo6_refit_fit = _todo6_fit_omegas(
        _refit_windows, _refit_position_scale
    )
    todo6_test_trajectories = _todo6_build_trajectories(
        TODO4_TEST_SEGMENTS
    )
    todo6_test_windows = _todo6_build_windows(
        todo6_test_trajectories, TODO4_TEST_SEGMENTS,
        TODO6_EVAL_STARTS_PER_SEGMENT, TODO6_CAUSAL_WINDOW,
    )
    todo6_test_details = _todo6_evaluate_windows(
        TODO6_REFIT_OMEGAS, todo6_test_windows,
        _refit_position_scale, _refit_velocity_scale,
    )
    todo6_test_summary, todo6_test_per_segment = _todo6_summarize(
        todo6_test_details
    )
    (TODO6_TEST_GATE_PASSED,
     todo6_test_macro_gain,
     todo6_test_segment_ratio,
     todo6_test_pivot) = _todo6_coordinate_gate(
        todo6_test_summary, todo6_test_per_segment
    )
    TODO6_FINAL_STATUS = (
        "validated_on_test"
        if TODO6_TEST_GATE_PASSED
        else "failed_on_test"
    )
else:
    TODO6_FINAL_STATUS = (
        "eligible_but_test_holdout_compromised"
        if TODO6_VALIDATED_BEFORE_TEST
        else "rejected_before_test_test_holdout_compromised"
    )

_reported_omegas = (
    TODO6_REFIT_OMEGAS
    if TODO6_REFIT_OMEGAS is not None
    else TODO6_TRAIN_OMEGAS
)
todo6_equations = pd.Series(
    {
        "dot x1": "vx1",
        "dot y1": "vy1",
        "dot vx1": f"{_reported_omegas[0]:.6f} vy1",
        "dot vy1": f"{-_reported_omegas[0]:.6f} vx1",
        "dot x2": "vx2",
        "dot y2": "vy2",
        "dot vx2": f"{_reported_omegas[1]:.6f} vy2",
        "dot vy2": f"{-_reported_omegas[1]:.6f} vx2",
    },
    name="rejected boundary diagnostic RHS",
)
todo6_freeze_manifest = pd.Series(
    {
        "status": TODO6_FINAL_STATUS,
        "primary target": "raw coordinate rollout",
        "model": "two-frequency rotating velocity",
        "causal window": TODO6_CAUSAL_WINDOW,
        "causal polyorder": TODO6_CAUSAL_POLYORDER,
        "train starts per segment": TODO6_TRAIN_STARTS_PER_SEGMENT,
        "eval starts per segment": TODO6_EVAL_STARTS_PER_SEGMENT,
        "horizons": TODO6_HORIZONS,
        "minimum position gain": TODO6_MIN_POSITION_GAIN,
        "maximum per-segment ratio": TODO6_MAX_SEGMENT_ERROR_RATIO,
        "omega bounds": TODO6_OMEGA_BOUNDS,
        "train omegas": tuple(TODO6_TRAIN_OMEGAS),
        "refit omegas": (
            None if TODO6_REFIT_OMEGAS is None else tuple(TODO6_REFIT_OMEGAS)
        ),
        "validation coordinate gate": TODO6_COORDINATE_GATE_PASSED,
        "stability gate": TODO6_STABILITY_GATE_PASSED,
        "boundary guard": TODO6_BOUNDARY_GUARD_PASSED,
        "test previously exposed": TODO6_TEST_PREVIOUSLY_EXPOSED,
        "test holdout pristine": TODO6_TEST_HOLDOUT_PRISTINE,
        "test opened": TODO6_TEST_OPENED,
        "test gate": TODO6_TEST_GATE_PASSED,
    },
    name="frozen value",
)
display(todo6_freeze_manifest.to_frame(), todo6_equations.to_frame())
if TODO6_TEST_OPENED:
    display(
        todo6_refit_fit,
        todo6_test_summary,
        todo6_test_macro_gain.rename("position gain").to_frame(),
        todo6_test_segment_ratio.rename("model / baseline").to_frame(),
    )
display(
    Markdown(
        f"**TODO6 final status:** `{TODO6_FINAL_STATUS}`. "
        f"Test opened = `{TODO6_TEST_OPENED}`, test gate = "
        f"`{TODO6_TEST_GATE_PASSED}`."
    )
)

### Итог TODO 6

Двухпараметрическая модель вращения не подтверждена. Оптимум обеих частот
упёрся в границы `+30/-30`: при дальнейшем росте частоты интегрированное
смещение стремится к нулю, поэтому модель фактически имитирует constant-position
baseline. После добавления этого обязательного baseline rotation model оказалась
хуже на validation на `14.8%`, `12.0%` и `7.5%` для горизонтов
`0.25`, `0.5`, `1.0`. Все `W=9/11/13` варианты также отклонены.

Первый автоматический conditional-test run произошёл до добавления boundary
guard и успел показать test-таблицу. Поэтому прежний holdout больше не считается
чистым и никакой test-verdict из него не принимается. Дальнейшие гипотезы нужно
сравнивать nested/leave-segment-out validation, всегда включая constant-position
baseline; для динамики предпочтителен weak/integral fit без второй производной.

## TODO 7 — forensic reverse engineering дискретного симулятора

- [x] Проверить временную семантику `distance` вместо same-row предположения.
- [x] Проверить, задают ли `angle1/angle2` направление отдельного update.
- [x] Восстановить свободную дискретную ветвь без производных и сглаживания.
- [x] Проверить найденные тождества отдельно на всех 11 reset-сегментах.
- [x] Локализовать несовпадения относительно границ экранной области.
- [ ] Восстановить точное boundary/reset-правило и собрать полный simulator.

TODO 4–6 остаются в notebook как честный отрицательный эксперимент. Здесь
не подбирается ещё одна ODE: проверяется гипотеза, что строка записана между
последовательными обновлениями двух объектов, а естественное время системы —
номер кадра. Все сегменты уже участвовали в exploratory discovery, поэтому
результат называется cross-segment identity audit, а не blind test.

In [ ]:
TODO7_ANGLE_MODULUS = 179.0
TODO7_BASE_STEP = 10.0
TODO7_DISTANCE_SCALE = 100.0
TODO7_COORDINATE_LIMITS = np.array([239.0, 179.0])
TODO7_IDENTITY_TOLERANCE = 2e-6
TODO7_DISCOVERY_USES_ALL_SEGMENTS = True

def _todo7_wrap_degrees(values):
    return (np.asarray(values) + 180.0) % 360.0 - 180.0

todo7_df = segmented_df.sort_index().copy()
todo7_p1 = todo7_df[["x1", "y1"]].to_numpy(dtype=float)
todo7_p2 = todo7_df[["x2", "y2"]].to_numpy(dtype=float)
todo7_angle1 = todo7_df["angle1"].to_numpy(dtype=float)
todo7_angle2 = todo7_df["angle2"].to_numpy(dtype=float)
todo7_distance = todo7_df["distance"].to_numpy(dtype=float)
todo7_time = todo7_df["time"].to_numpy(dtype=float)
todo7_segment = todo7_df["segment_id"].to_numpy(dtype=int)
todo7_same_segment = todo7_segment[1:] == todo7_segment[:-1]

# Записанный distance вычислен после update объекта 1, но до update объекта 2.
todo7_distance_async_prediction = np.linalg.norm(
    todo7_p1[1:] - todo7_p2[:-1], axis=1
)
todo7_distance_same_row_prediction = np.linalg.norm(
    todo7_p1[1:] - todo7_p2[1:], axis=1
)
todo7_distance_async_error = np.abs(
    todo7_distance[1:] - todo7_distance_async_prediction
)
todo7_distance_same_row_error = np.abs(
    todo7_distance[1:] - todo7_distance_same_row_prediction
)

todo7_distance_audit = pd.DataFrame(
    {
        "candidate": [
            "||p1[i] - p2[i-1]||",
            "||p1[i] - p2[i]||",
        ],
        "RMSE": [
            np.sqrt(np.mean(todo7_distance_async_error[todo7_same_segment] ** 2)),
            np.sqrt(np.mean(todo7_distance_same_row_error[todo7_same_segment] ** 2)),
        ],
        "median absolute error": [
            np.median(todo7_distance_async_error[todo7_same_segment]),
            np.median(todo7_distance_same_row_error[todo7_same_segment]),
        ],
        "maximum absolute error": [
            np.max(todo7_distance_async_error[todo7_same_segment]),
            np.max(todo7_distance_same_row_error[todo7_same_segment]),
        ],
    }
).set_index("candidate")

assert todo7_distance_async_error[todo7_same_segment].max() < 2e-6
todo7_distance_audit

### 7.A — строка является промежуточным, а не синхронным снимком

Точное async-равенство задаёт порядок операций:

1. взять обе позиции из предыдущей строки;
2. обновить объект 1;
3. записать расстояние между новым объектом 1 и старым объектом 2;
4. используя это расстояние, обновить объект 2.

Поэтому состояние `(x1[i], y1[i], x2[i], y2[i])`, использованное в ранних
ODE-моделях, смешивает два разных подмомента update loop.

In [ ]:
def _todo7_direction(angle_degrees):
    _angle = np.deg2rad(np.asarray(angle_degrees))
    return np.column_stack((np.sin(_angle), np.cos(_angle)))

# Полный one-step prediction свободной ветви только из предыдущей строки.
todo7_rho_before = np.linalg.norm(todo7_p2[:-1] - todo7_p1[:-1], axis=1)
todo7_predicted_angle1 = _todo7_wrap_degrees(
    todo7_angle1[:-1] + np.mod(todo7_rho_before, TODO7_ANGLE_MODULUS)
)
todo7_predicted_p1 = (
    todo7_p1[:-1]
    + (TODO7_BASE_STEP + todo7_rho_before / TODO7_DISTANCE_SCALE)[:, None]
    * _todo7_direction(todo7_predicted_angle1)
)
todo7_predicted_distance = np.linalg.norm(
    todo7_predicted_p1 - todo7_p2[:-1], axis=1
)
todo7_predicted_angle2 = _todo7_wrap_degrees(
    todo7_angle2[:-1] - np.mod(todo7_predicted_distance, TODO7_ANGLE_MODULUS)
)
todo7_predicted_p2 = (
    todo7_p2[:-1]
    + (TODO7_BASE_STEP + todo7_predicted_distance / TODO7_DISTANCE_SCALE)[:, None]
    * _todo7_direction(todo7_predicted_angle2)
)

# Для отдельной проверки object-2 branch используем реально записанный
# промежуточный distance, даже если object 1 встретил границу.
todo7_conditional_angle2 = _todo7_wrap_degrees(
    todo7_angle2[:-1] - np.mod(todo7_distance[1:], TODO7_ANGLE_MODULUS)
)
todo7_conditional_p2 = (
    todo7_p2[:-1]
    + (TODO7_BASE_STEP + todo7_distance[1:] / TODO7_DISTANCE_SCALE)[:, None]
    * _todo7_direction(todo7_conditional_angle2)
)

todo7_p1_error = np.linalg.norm(todo7_predicted_p1 - todo7_p1[1:], axis=1)
todo7_p2_conditional_error = np.linalg.norm(
    todo7_conditional_p2 - todo7_p2[1:], axis=1
)
todo7_p2_end_to_end_error = np.linalg.norm(
    todo7_predicted_p2 - todo7_p2[1:], axis=1
)
todo7_angle1_error = np.abs(_todo7_wrap_degrees(
    todo7_predicted_angle1 - todo7_angle1[1:]
))
todo7_angle2_conditional_error = np.abs(_todo7_wrap_degrees(
    todo7_conditional_angle2 - todo7_angle2[1:]
))

todo7_object1_match = (
    todo7_same_segment
    & (todo7_p1_error < TODO7_IDENTITY_TOLERANCE)
    & (todo7_angle1_error < TODO7_IDENTITY_TOLERANCE)
)
todo7_object2_conditional_match = (
    todo7_same_segment
    & (todo7_p2_conditional_error < TODO7_IDENTITY_TOLERANCE)
    & (todo7_angle2_conditional_error < TODO7_IDENTITY_TOLERANCE)
)
todo7_end_to_end_match = (
    todo7_object1_match
    & todo7_object2_conditional_match
    & (todo7_p2_end_to_end_error < TODO7_IDENTITY_TOLERANCE)
)

_todo7_segment_rows = []
for _segment_id in sorted(np.unique(todo7_segment)):
    _mask = todo7_same_segment & (todo7_segment[1:] == _segment_id)
    _todo7_segment_rows.append(
        {
            "segment_id": _segment_id,
            "transitions": int(_mask.sum()),
            "object1 free-law share": float(
                todo7_object1_match[_mask].mean()
            ),
            "object2 conditional free-law share": float(
                todo7_object2_conditional_match[_mask].mean()
            ),
            "complete free-step share": float(
                todo7_end_to_end_match[_mask].mean()
            ),
            "distance max abs error": float(
                todo7_distance_async_error[_mask].max()
            ),
        }
    )
todo7_formula_by_segment = pd.DataFrame(_todo7_segment_rows).set_index(
    "segment_id"
)

todo7_formula_summary = pd.Series(
    {
        "object1 free-law share": todo7_object1_match[
            todo7_same_segment
        ].mean(),
        "object2 conditional free-law share": (
            todo7_object2_conditional_match[todo7_same_segment].mean()
        ),
        "complete free-step share": todo7_end_to_end_match[
            todo7_same_segment
        ].mean(),
        "async distance max abs error": todo7_distance_async_error[
            todo7_same_segment
        ].max(),
    },
    name="value",
)

assert todo7_object1_match[todo7_same_segment].mean() > 0.70
assert todo7_object2_conditional_match[todo7_same_segment].mean() > 0.68
display(todo7_formula_summary.to_frame())
todo7_formula_by_segment.style.format(precision=6)

### 7.B — свободная ветвь точна, но система гибридная

Во внутренней области `angle` не является приблизительным признаком: он
задаёт направление в экранной системе координат через вектор
`(sin(angle), cos(angle))`. Длина шага линейно растёт с промежуточным
расстоянием: `10 + distance/100`. Число `179` одновременно входит в angular
modulo и совпадает с половиной наблюдаемой высоты области. Это сильный признак
алгоритмической canvas-симуляции, а не физической системы в известных единицах.

Несовпавшие строки нельзя считать обычным шумом и нельзя сглаживать вместе со
свободной ветвью: ниже проверяется их концентрация у прямоугольной границы.

In [ ]:
def _todo7_boundary_margin(positions):
    return np.min(TODO7_COORDINATE_LIMITS - np.abs(positions), axis=1)

todo7_p1_previous_margin = _todo7_boundary_margin(todo7_p1[:-1])
todo7_p2_previous_margin = _todo7_boundary_margin(todo7_p2[:-1])
_todo7_boundary_rows = []
for _name, _match, _margin in [
    ("object1", todo7_object1_match, todo7_p1_previous_margin),
    ("object2", todo7_object2_conditional_match, todo7_p2_previous_margin),
]:
    _valid = todo7_same_segment
    _failed = _valid & ~_match
    _matched = _valid & _match
    _todo7_boundary_rows.append(
        {
            "object": _name,
            "matched transitions": int(_matched.sum()),
            "non-matched transitions": int(_failed.sum()),
            "median margin, matched": float(np.median(_margin[_matched])),
            "median margin, non-matched": float(np.median(_margin[_failed])),
            "non-matched with margin <= 0.5": float(
                np.mean(_margin[_failed] <= 0.5)
            ),
        }
    )
todo7_boundary_summary = pd.DataFrame(_todo7_boundary_rows).set_index(
    "object"
)

todo7_bounds = pd.DataFrame(
    {
        "minimum": todo7_df[["x1", "y1", "x2", "y2"]].min(),
        "maximum": todo7_df[["x1", "y1", "x2", "y2"]].max(),
    }
)

_todo7_fig, _todo7_axes = plt.subplots(1, 2, figsize=(14, 5.5))
for _ax, _positions, _match, _title in [
    (_todo7_axes[0], todo7_p1, todo7_object1_match, "Объект 1"),
    (
        _todo7_axes[1], todo7_p2, todo7_object2_conditional_match,
        "Объект 2",
    ),
]:
    _failed_rows = np.flatnonzero(todo7_same_segment & ~_match) + 1
    _ax.scatter(
        _positions[:, 0], _positions[:, 1], s=0.6, alpha=0.10,
        color="slategray", label="все положения",
    )
    _ax.scatter(
        _positions[_failed_rows, 0], _positions[_failed_rows, 1],
        s=2.0, alpha=0.35, color="crimson",
        label="free law не совпал",
    )
    _ax.set_xlim(-245, 245)
    _ax.set_ylim(-185, 185)
    _ax.set_aspect("equal", adjustable="box")
    _ax.set_xlabel("x")
    _ax.set_ylabel("y")
    _ax.set_title(_title)
    _ax.grid(alpha=0.2)
    _ax.legend(markerscale=4)
_todo7_fig.suptitle(
    "Несовпадения свободного update law концентрируются у границ"
)
_todo7_fig.tight_layout()

display(todo7_bounds, todo7_boundary_summary.style.format(precision=4))
_todo7_fig

### 7.C — стандартное отражение и геометрия объекта

Для boundary-перехода проверяются четыре заранее заданных варианта угла:
свободный, отражённый от вертикальной стены `-angle`, отражённый от
горизонтальной стены `180°-angle` и двойное отражение `angle+180°`. Это не
регрессионный fit: каждый вариант следует из геометрии зеркального отражения.

Отдельно проверяется положение центра у стены. Формула
`g(angle)=|sin(angle)|+|cos(angle)|-1` является добавочной полушириной
axis-aligned bounding box повёрнутого единичного квадрата. Поэтому clamp для
центра равен `(239-g, 179-g)`. Геометрическая проверка ниже использует
записанный следующий угол и доказывает форму объекта/clamp, но сама по себе не
решает редкий выбор corner/loop-ветви.

In [ ]:
def _todo7_angle_candidates(free_angle):
    return np.column_stack(
        (
            free_angle,
            _todo7_wrap_degrees(-free_angle),
            _todo7_wrap_degrees(180.0 - free_angle),
            _todo7_wrap_degrees(free_angle + 180.0),
        )
    )

_todo7_reflection_rows = []
_todo7_branch_rows = []
todo7_standard_angle_match = {}
for _name, _previous_position, _actual_position, _actual_angle, _free_angle, _distance_for_step in [
    (
        "object1", todo7_p1[:-1], todo7_p1[1:], todo7_angle1[1:],
        todo7_predicted_angle1, todo7_rho_before,
    ),
    (
        "object2", todo7_p2[:-1], todo7_p2[1:], todo7_angle2[1:],
        todo7_conditional_angle2, todo7_distance[1:],
    ),
]:
    _candidates = _todo7_angle_candidates(_free_angle)
    _candidate_error = np.abs(
        _todo7_wrap_degrees(_actual_angle[:, None] - _candidates)
    )
    _branch_index = np.argmin(_candidate_error, axis=1)
    _best_angle_error = np.min(_candidate_error, axis=1)
    _standard_match = (
        todo7_same_segment
        & (_best_angle_error < TODO7_IDENTITY_TOLERANCE)
    )
    todo7_standard_angle_match[_name] = _standard_match

    _free_position = (
        _previous_position
        + (
            TODO7_BASE_STEP
            + _distance_for_step / TODO7_DISTANCE_SCALE
        )[:, None]
        * _todo7_direction(_free_angle)
    )
    _actual_radians = np.deg2rad(_actual_angle)
    _footprint_gap = (
        np.abs(np.sin(_actual_radians))
        + np.abs(np.cos(_actual_radians))
        - 1.0
    )
    _oriented_limits = (
        TODO7_COORDINATE_LIMITS - _footprint_gap[:, None]
    )
    _geometry_prediction = np.clip(
        _free_position, -_oriented_limits, _oriented_limits
    )
    _geometry_error = np.linalg.norm(
        _geometry_prediction - _actual_position, axis=1
    )

    _todo7_reflection_rows.append(
        {
            "object": _name,
            "standard angle/reflection share": float(
                _standard_match[todo7_same_segment].mean()
            ),
            "geometry-with-recorded-angle share": float(
                np.mean(
                    _geometry_error[todo7_same_segment] < 1e-5
                )
            ),
            "geometry error q99": float(
                np.quantile(_geometry_error[todo7_same_segment], 0.99)
            ),
            "unexplained angle transitions": int(
                np.sum(todo7_same_segment & ~_standard_match)
            ),
        }
    )
    for _branch_id, _branch_name in enumerate(
        ("free", "vertical reflection", "horizontal reflection", "double reflection")
    ):
        _todo7_branch_rows.append(
            {
                "object": _name,
                "branch": _branch_name,
                "exact transitions": int(
                    np.sum(_standard_match & (_branch_index == _branch_id))
                ),
            }
        )

todo7_reflection_summary = pd.DataFrame(
    _todo7_reflection_rows
).set_index("object")
todo7_reflection_branches = pd.DataFrame(
    _todo7_branch_rows
).pivot(index="object", columns="branch", values="exact transitions")

assert todo7_reflection_summary[
    "standard angle/reflection share"
].min() > 0.98
assert todo7_reflection_summary[
    "geometry-with-recorded-angle share"
].min() > 0.999
display(
    todo7_reflection_summary.style.format(precision=6),
    todo7_reflection_branches,
)

In [ ]:
TODO7_STATUS = (
    "free_and_standard_reflection_laws_recovered_corner_reset_open"
)
todo7_manifest = pd.Series(
    {
        "status": TODO7_STATUS,
        "model class": "sequential discrete hybrid map",
        "natural time": "row/update index",
        "row order": "object1 -> distance -> object2",
        "angle convention": "direction=(sin(angle), cos(angle))",
        "angle modulus": TODO7_ANGLE_MODULUS,
        "step law": "10 + distance / 100",
        "coordinate bounds": tuple(TODO7_COORDINATE_LIMITS),
        "segments audited": tuple(sorted(np.unique(todo7_segment))),
        "all segments used in discovery": (
            TODO7_DISCOVERY_USES_ALL_SEGMENTS
        ),
        "boundary rule": (
            "standard reflection recovered; rare corner/loop branch open"
        ),
        "previous holdout pristine": False,
    },
    name="value",
)
display(todo7_manifest.to_frame(), todo7_formula_summary.to_frame())
display(
    Markdown(
        f"**TODO7 status:** `{TODO7_STATUS}`. Свободная ветвь и обычное "
        "отражение найдены; следующая проверяемая задача — редкое "
        "corner/loop- и reset-правило и "
        "длинный rollout собственного дискретного simulator."
    )
)

### Итог TODO 7

В данных есть сильный и интерпретируемый закон, но это не автономная ODE для
четырёх синхронных координат. Это последовательная дискретная карта: первый
объект поворачивается и перемещается, затем измеряется промежуточное расстояние,
после чего поворачивается и перемещается второй объект. В свободной области
порядок операций, angular modulo и длина шага восстановлены до численной
точности.

Различие рисунков между сегментами совместимо с одной и той же хаотической
картой при разных начальных состояниях. На границе обычное зеркальное отражение
и clamp центра повёрнутого квадрата объясняют подавляющее большинство
переходов. Открытыми остаются редкая wall/corner/loop-ветвь и последующий
глобальный reset. Поэтому следующий TODO должен моделировать именно эти
события, а не усложнять SINDy или сильнее сглаживать координаты.

## TODO 8 — causal baseline simulator и карта отказов

Найденные в TODO 7 тождества превращаются в исполняемый покадровый simulator.
Это **cross-segment forensic audit**, а не blind validation: все 11 сегментов
уже участвовали в discovery, а прежний holdout не pristine.

Контракт:

- состояние: сырые `p1, angle1, p2, angle2`; без сглаживания и производных;
- естественный шаг — номер update, `dt` не входит в карту;
- free law, standard reflection и rotated-square clamp фиксированы TODO 7;
- при пересечении обеих осей действует найденный corner priority: в правом
  верхнем углу horizontal reflection, в остальных углах vertical;
- special wall/loop остаётся неизвестной ветвью и выявляется по first mismatch;
- teacher-forced one-step видит лишь предыдущую наблюдаемую строку;
- recursive rollout после старта не получает truth.

Из-за округления clamped coordinates position tolerance равен `1e-5`, а
angle/distance tolerance — `2e-6`. Для rollout материальный
отказ означает position/distance error `>=0.1` или angle error `>=1°`.

In [ ]:
TODO8_BRANCH_NAMES = (
    "free",
    "vertical reflection",
    "horizontal reflection",
)
TODO8_ALL_ANGLE_BRANCHES = (
    *TODO8_BRANCH_NAMES,
    "double reflection",
)
TODO8_CROSSING_TOLERANCE = 1e-9
TODO8_POSITION_TOLERANCE = 1e-5
TODO8_ANGLE_TOLERANCE = TODO7_IDENTITY_TOLERANCE
TODO8_DISTANCE_TOLERANCE = TODO7_IDENTITY_TOLERANCE
TODO8_MATERIAL_POSITION_ERROR = 0.1
TODO8_MATERIAL_DISTANCE_ERROR = 0.1
TODO8_MATERIAL_ANGLE_ERROR = 1.0
TODO8_ROLLOUT_HORIZONS = (10, 50, 100, 500)


def _todo8_direction_scalar(angle_degrees):
    _angle = np.deg2rad(float(angle_degrees))
    return np.array([np.sin(_angle), np.cos(_angle)], dtype=float)


def _todo8_oriented_limits(angle_degrees):
    _angle = np.deg2rad(float(angle_degrees))
    _gap = abs(np.sin(_angle)) + abs(np.cos(_angle)) - 1.0
    return TODO7_COORDINATE_LIMITS - _gap


def _todo8_identify_actual_branch(free_angle, actual_angle):
    _candidates = np.array(
        [
            free_angle,
            _todo7_wrap_degrees(-free_angle),
            _todo7_wrap_degrees(180.0 - free_angle),
            _todo7_wrap_degrees(free_angle + 180.0),
        ],
        dtype=float,
    )
    _errors = np.abs(
        _todo7_wrap_degrees(float(actual_angle) - _candidates)
    )
    _best = int(np.argmin(_errors))
    if _errors[_best] >= TODO7_IDENTITY_TOLERANCE:
        return "corner/loop"
    return TODO8_ALL_ANGLE_BRANCHES[_best]


def _todo8_update_object(
    previous_position,
    previous_angle,
    driver_distance,
    angle_sign,
):
    _previous_position = np.asarray(previous_position, dtype=float)
    _free_angle = float(
        _todo7_wrap_degrees(
            float(previous_angle)
            + angle_sign
            * np.mod(float(driver_distance), TODO7_ANGLE_MODULUS)
        )
    )
    _step_length = (
        TODO7_BASE_STEP
        + float(driver_distance) / TODO7_DISTANCE_SCALE
    )
    _displacement = (
        _step_length * _todo8_direction_scalar(_free_angle)
    )
    _free_position = _previous_position + _displacement
    _limits = _todo8_oriented_limits(_free_angle)
    _crossing = (
        np.abs(_free_position)
        > _limits + TODO8_CROSSING_TOLERANCE
    )
    _hit_fraction = np.full(2, np.inf, dtype=float)

    for _axis in range(2):
        if not _crossing[_axis]:
            continue
        _axis_delta = _displacement[_axis]
        if abs(_axis_delta) <= np.finfo(float).eps:
            continue
        _wall_coordinate = (
            np.sign(_free_position[_axis]) * _limits[_axis]
        )
        _candidate_fraction = (
            (_wall_coordinate - _previous_position[_axis])
            / _axis_delta
        )
        if _candidate_fraction >= -TODO8_CROSSING_TOLERANCE:
            _hit_fraction[_axis] = _candidate_fraction

    if _crossing[0] and _crossing[1]:
        # Empirical canvas priority, найденный на standard transitions.
        _branch_index = (
            2
            if _free_position[0] > 0 and _free_position[1] > 0
            else 1
        )
    elif _crossing[0]:
        _branch_index = 1
    elif _crossing[1]:
        _branch_index = 2
    else:
        _branch_index = 0

    _candidate_angles = np.array(
        [
            _free_angle,
            _todo7_wrap_degrees(-_free_angle),
            _todo7_wrap_degrees(180.0 - _free_angle),
        ],
        dtype=float,
    )
    _next_angle = float(_candidate_angles[_branch_index])
    _next_limits = _todo8_oriented_limits(_next_angle)
    _next_position = np.clip(
        _free_position, -_next_limits, _next_limits
    )
    return {
        "position": _next_position,
        "angle": _next_angle,
        "branch": TODO8_BRANCH_NAMES[_branch_index],
        "free_angle": _free_angle,
        "free_position": _free_position,
        "step_length": _step_length,
        "limits": _next_limits,
        "crossing": _crossing,
        "hit_fraction": _hit_fraction,
        "penetration": np.maximum(
            np.abs(_free_position) - _next_limits, 0.0
        ),
    }


def _todo8_predict_step(p1, angle1, p2, angle2):
    _p1 = np.asarray(p1, dtype=float)
    _p2 = np.asarray(p2, dtype=float)
    _rho_before = float(np.linalg.norm(_p2 - _p1))
    _object1 = _todo8_update_object(
        _p1, angle1, _rho_before, angle_sign=1.0
    )
    _distance = float(
        np.linalg.norm(_object1["position"] - _p2)
    )
    _object2 = _todo8_update_object(
        _p2, angle2, _distance, angle_sign=-1.0
    )
    return {
        "p1": _object1["position"],
        "angle1": _object1["angle"],
        "p2": _object2["position"],
        "angle2": _object2["angle"],
        "distance": _distance,
        "object1": _object1,
        "object2": _object2,
    }


def _todo8_angle_error(prediction, truth):
    return float(
        abs(_todo7_wrap_degrees(float(prediction) - float(truth)))
    )


### 8.A — causal one-step и conditional event audit

Полный prediction использует только строку `i-1`. Object 2 дополнительно
проверяется conditionally с записанным промежуточным `distance[i]`, чтобы
отделить его collision-логику от upstream-ошибки object 1. Сохранённая
`todo8_event_table` содержит геометрию каждого события для следующего
stateful mode model.

In [ ]:
todo8_transition_rows = []
todo8_event_rows = []
todo8_segment_step = (
    todo7_df.groupby("segment_id", sort=False).cumcount().to_numpy()
)

for _i in range(1, len(todo7_df)):
    if not todo7_same_segment[_i - 1]:
        continue
    _prediction = _todo8_predict_step(
        todo7_p1[_i - 1],
        todo7_angle1[_i - 1],
        todo7_p2[_i - 1],
        todo7_angle2[_i - 1],
    )
    _p1_error = float(
        np.linalg.norm(_prediction["p1"] - todo7_p1[_i])
    )
    _p2_error = float(
        np.linalg.norm(_prediction["p2"] - todo7_p2[_i])
    )
    _angle1_error = _todo8_angle_error(
        _prediction["angle1"], todo7_angle1[_i]
    )
    _angle2_error = _todo8_angle_error(
        _prediction["angle2"], todo7_angle2[_i]
    )
    _distance_error = float(
        abs(_prediction["distance"] - todo7_distance[_i])
    )
    todo8_transition_rows.append(
        {
            "segment_id": int(todo7_segment[_i]),
            "segment_step": int(todo8_segment_step[_i]),
            "source_row": int(todo7_df.index[_i]),
            "p1_error": _p1_error,
            "p2_error": _p2_error,
            "angle1_error": _angle1_error,
            "angle2_error": _angle2_error,
            "distance_error": _distance_error,
            "maximum_position_error": max(
                _p1_error, _p2_error
            ),
            "maximum_angle_error": max(
                _angle1_error, _angle2_error
            ),
        }
    )

    _rho_before = float(
        np.linalg.norm(todo7_p2[_i - 1] - todo7_p1[_i - 1])
    )
    _object_specs = (
        (
            "object1",
            todo7_p1[_i - 1],
            todo7_angle1[_i - 1],
            todo7_p1[_i],
            todo7_angle1[_i],
            _rho_before,
            1.0,
        ),
        (
            "object2",
            todo7_p2[_i - 1],
            todo7_angle2[_i - 1],
            todo7_p2[_i],
            todo7_angle2[_i],
            float(todo7_distance[_i]),
            -1.0,
        ),
    )
    for (
        _object_name,
        _previous_position,
        _previous_angle,
        _actual_position,
        _actual_angle,
        _driver_distance,
        _angle_sign,
    ) in _object_specs:
        _conditional = _todo8_update_object(
            _previous_position,
            _previous_angle,
            _driver_distance,
            _angle_sign,
        )
        _actual_branch = _todo8_identify_actual_branch(
            _conditional["free_angle"], _actual_angle
        )
        _position_error = float(
            np.linalg.norm(
                _conditional["position"] - _actual_position
            )
        )
        _angle_error = _todo8_angle_error(
            _conditional["angle"], _actual_angle
        )
        _previous_gap = (
            _todo8_oriented_limits(_previous_angle)
            - np.abs(_previous_position)
        )
        todo8_event_rows.append(
            {
                "segment_id": int(todo7_segment[_i]),
                "segment_step": int(todo8_segment_step[_i]),
                "source_row": int(todo7_df.index[_i]),
                "object": _object_name,
                "previous_x": float(_previous_position[0]),
                "previous_y": float(_previous_position[1]),
                "previous_angle": float(_previous_angle),
                "driver_distance": _driver_distance,
                "free_angle": _conditional["free_angle"],
                "step_length": _conditional["step_length"],
                "free_x": float(
                    _conditional["free_position"][0]
                ),
                "free_y": float(
                    _conditional["free_position"][1]
                ),
                "x_penetration": float(
                    _conditional["penetration"][0]
                ),
                "y_penetration": float(
                    _conditional["penetration"][1]
                ),
                "x_hit_fraction": float(
                    _conditional["hit_fraction"][0]
                ),
                "y_hit_fraction": float(
                    _conditional["hit_fraction"][1]
                ),
                "predicted_x_crossing": bool(
                    _conditional["crossing"][0]
                ),
                "predicted_y_crossing": bool(
                    _conditional["crossing"][1]
                ),
                "previous_x_gap": float(_previous_gap[0]),
                "previous_y_gap": float(_previous_gap[1]),
                "previous_wall_contact": bool(
                    np.min(_previous_gap) <= 1e-5
                ),
                "predicted_branch": _conditional["branch"],
                "actual_branch": _actual_branch,
                "branch_correct": bool(
                    _conditional["branch"] == _actual_branch
                ),
                "conditional_position_error": _position_error,
                "conditional_angle_error": _angle_error,
                "conditional_state_exact": bool(
                    _position_error < TODO8_POSITION_TOLERANCE
                    and _angle_error < TODO8_ANGLE_TOLERANCE
                ),
            }
        )

todo8_transition_audit = pd.DataFrame(todo8_transition_rows)
todo8_event_table = pd.DataFrame(todo8_event_rows)
todo8_event_table["both_axis_proposal"] = (
    todo8_event_table["predicted_x_crossing"]
    & todo8_event_table["predicted_y_crossing"]
)
todo8_event_table["boundary_candidate"] = (
    todo8_event_table["predicted_x_crossing"]
    | todo8_event_table["predicted_y_crossing"]
    | todo8_event_table["actual_branch"].ne("free")
)

_todo8_exact_positions = (
    todo8_transition_audit["maximum_position_error"]
    < TODO8_POSITION_TOLERANCE
)
_todo8_exact_angles = (
    todo8_transition_audit["maximum_angle_error"]
    < TODO8_ANGLE_TOLERANCE
)
_todo8_exact_distance = (
    todo8_transition_audit["distance_error"]
    < TODO8_DISTANCE_TOLERANCE
)
_todo8_exact_complete = (
    _todo8_exact_positions
    & _todo8_exact_angles
    & _todo8_exact_distance
)
todo8_teacher_summary = pd.Series(
    {
        "transitions": len(todo8_transition_audit),
        "diagnostic exact positions share": float(
            _todo8_exact_positions.mean()
        ),
        "diagnostic exact angles share": float(
            _todo8_exact_angles.mean()
        ),
        "diagnostic exact complete-state share": float(
            _todo8_exact_complete.mean()
        ),
        "median maximum position error": float(
            todo8_transition_audit[
                "maximum_position_error"
            ].median()
        ),
        "position error q95": float(
            todo8_transition_audit[
                "maximum_position_error"
            ].quantile(0.95)
        ),
        "angle error q95": float(
            todo8_transition_audit[
                "maximum_angle_error"
            ].quantile(0.95)
        ),
    },
    name="value",
)

_todo8_object_rows = []
for _object_name, _events in todo8_event_table.groupby(
    "object", sort=True
):
    _standard = _events["actual_branch"].ne("corner/loop")
    _both_standard = _standard & _events["both_axis_proposal"]
    _todo8_object_rows.append(
        {
            "object": _object_name,
            "transitions": len(_events),
            "conditional exact-state share": float(
                _events["conditional_state_exact"].mean()
            ),
            "standard branch-selection accuracy": float(
                _events.loc[_standard, "branch_correct"].mean()
            ),
            "both-axis proposals": int(
                _events["both_axis_proposal"].sum()
            ),
            "both-axis heuristic accuracy": float(
                _events.loc[
                    _both_standard, "branch_correct"
                ].mean()
            ),
            "unexplained actual transitions": int(
                (~_standard).sum()
            ),
        }
    )
todo8_object_summary = pd.DataFrame(
    _todo8_object_rows
).set_index("object")
todo8_failure_events = todo8_event_table.loc[
    ~todo8_event_table["conditional_state_exact"]
].copy()

assert len(todo8_transition_audit) == int(todo7_same_segment.sum())
assert (
    todo8_teacher_summary[
        "diagnostic exact complete-state share"
    ] > 0.96
)
assert set(todo8_event_table["object"]) == {"object1", "object2"}

display(
    todo8_teacher_summary.to_frame(),
    todo8_object_summary.style.format(precision=6),
)
display(
    Markdown(
        "### Первые conditional failures для будущего mode model"
    ),
    todo8_failure_events.loc[
        :,
        [
            "segment_id",
            "segment_step",
            "source_row",
            "object",
            "previous_x",
            "previous_y",
            "free_angle",
            "predicted_branch",
            "actual_branch",
            "both_axis_proposal",
            "previous_wall_contact",
            "conditional_position_error",
            "conditional_angle_error",
        ],
    ].head(16),
)


In [ ]:
# Stateful boundary-признаки для TODO 9: они только описывают события
# и не меняют baseline policy или уже рассчитанные one-step scores.
_todo8_source_to_position = pd.Series(
    np.arange(len(todo7_df), dtype=int),
    index=todo7_df.index,
)
_todo8_event_locations = _todo8_source_to_position.loc[
    todo8_event_table["source_row"]
].to_numpy(dtype=int)
_todo8_is_object1 = (
    todo8_event_table["object"].to_numpy() == "object1"
)
_todo8_actual_position = np.empty(
    (len(todo8_event_table), 2), dtype=float
)
_todo8_actual_angle = np.empty(len(todo8_event_table), dtype=float)
_todo8_actual_position[_todo8_is_object1] = todo7_p1[
    _todo8_event_locations[_todo8_is_object1]
]
_todo8_actual_position[~_todo8_is_object1] = todo7_p2[
    _todo8_event_locations[~_todo8_is_object1]
]
_todo8_actual_angle[_todo8_is_object1] = todo7_angle1[
    _todo8_event_locations[_todo8_is_object1]
]
_todo8_actual_angle[~_todo8_is_object1] = todo7_angle2[
    _todo8_event_locations[~_todo8_is_object1]
]

_todo8_actual_radians = np.deg2rad(_todo8_actual_angle)
_todo8_actual_gap = (
    np.abs(np.sin(_todo8_actual_radians))
    + np.abs(np.cos(_todo8_actual_radians))
    - 1.0
)
_todo8_actual_limits = (
    TODO7_COORDINATE_LIMITS - _todo8_actual_gap[:, None]
)
_todo8_edge_error = (
    _todo8_actual_limits - np.abs(_todo8_actual_position)
)
_todo8_x_wall = np.abs(_todo8_edge_error[:, 0]) < 1e-5
_todo8_y_wall = np.abs(_todo8_edge_error[:, 1]) < 1e-5
todo8_event_table["current_x_wall"] = _todo8_x_wall
todo8_event_table["current_y_wall"] = _todo8_y_wall
todo8_event_table["current_wall_contact"] = (
    _todo8_x_wall | _todo8_y_wall
)
todo8_event_table["wall_mask"] = np.select(
    [
        _todo8_x_wall & _todo8_y_wall,
        _todo8_x_wall,
        _todo8_y_wall,
    ],
    ["xy", "x", "y"],
    default="interior",
)

_todo8_lag2_valid = (
    (_todo8_event_locations >= 2)
    & (
        todo7_segment[_todo8_event_locations]
        == todo7_segment[
            np.maximum(_todo8_event_locations - 2, 0)
        ]
    )
)
_todo8_lag2_position = np.full(
    (len(todo8_event_table), 2), np.nan
)
_todo8_lag2_angle = np.full(len(todo8_event_table), np.nan)
_todo8_lag2_locations = np.maximum(
    _todo8_event_locations - 2, 0
)
_todo8_lag2_position[_todo8_is_object1] = todo7_p1[
    _todo8_lag2_locations[_todo8_is_object1]
]
_todo8_lag2_position[~_todo8_is_object1] = todo7_p2[
    _todo8_lag2_locations[~_todo8_is_object1]
]
_todo8_lag2_angle[_todo8_is_object1] = todo7_angle1[
    _todo8_lag2_locations[_todo8_is_object1]
]
_todo8_lag2_angle[~_todo8_is_object1] = todo7_angle2[
    _todo8_lag2_locations[~_todo8_is_object1]
]
_todo8_lag2_position_linf = np.max(
    np.abs(_todo8_actual_position - _todo8_lag2_position),
    axis=1,
)
_todo8_lag2_angle_error = np.abs(
    _todo7_wrap_degrees(
        _todo8_actual_angle - _todo8_lag2_angle
    )
)
_todo8_lag2_position_linf[~_todo8_lag2_valid] = np.nan
_todo8_lag2_angle_error[~_todo8_lag2_valid] = np.nan
todo8_event_table["lag2_position_linf"] = (
    _todo8_lag2_position_linf
)
todo8_event_table["lag2_angle_error"] = (
    _todo8_lag2_angle_error
)

_todo8_boundary_run_length = np.zeros(
    len(todo8_event_table), dtype=int
)
for _, _indices in todo8_event_table.groupby(
    ["object", "segment_id"], sort=False
).indices.items():
    _run_length = 0
    for _row_index in np.sort(_indices):
        if todo8_event_table.at[
            _row_index, "current_wall_contact"
        ]:
            _run_length += 1
        else:
            _run_length = 0
        _todo8_boundary_run_length[_row_index] = _run_length
todo8_event_table["boundary_run_length"] = (
    _todo8_boundary_run_length
)
todo8_event_table["lag2_loop_candidate"] = (
    todo8_event_table["current_wall_contact"]
    & todo8_event_table["lag2_position_linf"].lt(1e-5)
    & todo8_event_table["lag2_angle_error"].lt(1e-5)
    & todo8_event_table["boundary_run_length"].ge(4)
)

_todo8_special = todo8_event_table["actual_branch"].eq(
    "corner/loop"
)
todo8_event_table["event_mode"] = np.select(
    [
        _todo8_special
        & todo8_event_table["segment_step"].eq(1),
        _todo8_special
        & todo8_event_table["lag2_loop_candidate"],
        _todo8_special
        & todo8_event_table["current_wall_contact"],
        _todo8_special,
    ],
    [
        "segment_start_special",
        "period2_wall_loop",
        "wall_loop_special",
        "interior_special",
    ],
    default=todo8_event_table["actual_branch"],
)
todo8_event_mode_summary = (
    todo8_event_table.groupby(
        ["object", "event_mode"], as_index=False
    ).agg(transitions=("source_row", "size"))
)

todo8_failure_events = todo8_event_table.loc[
    ~todo8_event_table["conditional_state_exact"]
].copy()
todo8_loop2_summary = (
    todo8_event_table.loc[
        todo8_event_table["lag2_loop_candidate"]
    ]
    .groupby(["segment_id", "object"], as_index=False)
    .agg(
        loop2_rows=("source_row", "size"),
        first_row=("source_row", "min"),
        last_row=("source_row", "max"),
        maximum_boundary_run=("boundary_run_length", "max"),
        maximum_lag2_position_error=(
            "lag2_position_linf", "max"
        ),
        maximum_lag2_angle_error=(
            "lag2_angle_error", "max"
        ),
    )
)
TODO8_RESET_ROWS = tuple(
    int(_index) for _index in break_rows.index
)

display(
    Markdown(
        "### Найденные period-2 boundary candidates"
    ),
    todo8_loop2_summary.style.format(precision=8),
    todo8_event_mode_summary,
)
display(
    Markdown(
        f"Reset transitions не моделируются; строки после jump: "
        f"`{TODO8_RESET_ROWS}`."
    )
)


### 8.B — exact-rule survival и recursive rollout

Каждый сегмент стартует из первой наблюдаемой строки. Exact-rule survival
заканчивается при первом превышении фиксированных position/angle/distance
tolerances; это post-hoc метка первой special wall/loop ошибки. Полный recursive
rollout после неё продолжается только как диагностика хаотического расхождения
и не считается blind validation score.

In [ ]:
def _todo8_first_failure(mask):
    _indices = np.flatnonzero(np.asarray(mask, dtype=bool))
    return None if len(_indices) == 0 else int(_indices[0] + 1)


def _todo8_rollout_segment(segment_frame):
    _segment_frame = segment_frame.sort_index()
    _truth_p1 = _segment_frame[["x1", "y1"]].to_numpy(dtype=float)
    _truth_p2 = _segment_frame[["x2", "y2"]].to_numpy(dtype=float)
    _truth_angle1 = _segment_frame["angle1"].to_numpy(dtype=float)
    _truth_angle2 = _segment_frame["angle2"].to_numpy(dtype=float)
    _truth_distance = _segment_frame["distance"].to_numpy(dtype=float)

    _predicted_p1 = [_truth_p1[0].copy()]
    _predicted_p2 = [_truth_p2[0].copy()]
    _predicted_angle1 = [float(_truth_angle1[0])]
    _predicted_angle2 = [float(_truth_angle2[0])]
    _predicted_distance = [float(_truth_distance[0])]
    _predicted_branch1 = ["initial"]
    _predicted_branch2 = ["initial"]

    for _ in range(1, len(_segment_frame)):
        _step = _todo8_predict_step(
            _predicted_p1[-1],
            _predicted_angle1[-1],
            _predicted_p2[-1],
            _predicted_angle2[-1],
        )
        _predicted_p1.append(_step["p1"])
        _predicted_p2.append(_step["p2"])
        _predicted_angle1.append(_step["angle1"])
        _predicted_angle2.append(_step["angle2"])
        _predicted_distance.append(_step["distance"])
        _predicted_branch1.append(_step["object1"]["branch"])
        _predicted_branch2.append(_step["object2"]["branch"])

    _predicted_p1 = np.asarray(_predicted_p1)
    _predicted_p2 = np.asarray(_predicted_p2)
    _predicted_angle1 = np.asarray(_predicted_angle1)
    _predicted_angle2 = np.asarray(_predicted_angle2)
    _predicted_distance = np.asarray(_predicted_distance)

    _p1_error = np.linalg.norm(
        _predicted_p1[1:] - _truth_p1[1:], axis=1
    )
    _p2_error = np.linalg.norm(
        _predicted_p2[1:] - _truth_p2[1:], axis=1
    )
    _position_error = np.maximum(_p1_error, _p2_error)
    _angle_error = np.maximum(
        np.abs(
            _todo7_wrap_degrees(
                _predicted_angle1[1:] - _truth_angle1[1:]
            )
        ),
        np.abs(
            _todo7_wrap_degrees(
                _predicted_angle2[1:] - _truth_angle2[1:]
            )
        ),
    )
    _distance_error = np.abs(
        _predicted_distance[1:] - _truth_distance[1:]
    )
    _numerical_failure = (
        (_position_error >= TODO8_POSITION_TOLERANCE)
        | (_angle_error >= TODO8_ANGLE_TOLERANCE)
        | (_distance_error >= TODO8_DISTANCE_TOLERANCE)
    )
    _material_failure = (
        (_position_error >= TODO8_MATERIAL_POSITION_ERROR)
        | (_angle_error >= TODO8_MATERIAL_ANGLE_ERROR)
        | (_distance_error >= TODO8_MATERIAL_DISTANCE_ERROR)
    )

    _trajectory = pd.DataFrame(
        {
            "source_row": _segment_frame.index.to_numpy(dtype=int),
            "truth_x1": _truth_p1[:, 0],
            "truth_y1": _truth_p1[:, 1],
            "truth_x2": _truth_p2[:, 0],
            "truth_y2": _truth_p2[:, 1],
            "predicted_x1": _predicted_p1[:, 0],
            "predicted_y1": _predicted_p1[:, 1],
            "predicted_x2": _predicted_p2[:, 0],
            "predicted_y2": _predicted_p2[:, 1],
            "predicted_angle1": _predicted_angle1,
            "predicted_angle2": _predicted_angle2,
            "predicted_distance": _predicted_distance,
            "predicted_branch1": _predicted_branch1,
            "predicted_branch2": _predicted_branch2,
        }
    )
    return {
        "trajectory": _trajectory,
        "position_error": _position_error,
        "angle_error": _angle_error,
        "distance_error": _distance_error,
        "first_numerical_failure": _todo8_first_failure(
            _numerical_failure
        ),
        "first_material_failure": _todo8_first_failure(
            _material_failure
        ),
    }


todo8_rollouts = {}
_todo8_rollout_rows = []
_todo8_horizon_rows = []

for _segment_id, _segment_frame in todo7_df.groupby(
    "segment_id", sort=True
):
    _rollout = _todo8_rollout_segment(_segment_frame)
    todo8_rollouts[int(_segment_id)] = _rollout
    _trajectory = _rollout["trajectory"]
    _transition_count = len(_segment_frame) - 1
    _predicted_positions = _trajectory[
        [
            "predicted_x1",
            "predicted_y1",
            "predicted_x2",
            "predicted_y2",
        ]
    ].to_numpy(dtype=float)
    _truth_positions = _segment_frame[
        ["x1", "y1", "x2", "y2"]
    ].to_numpy(dtype=float)
    _todo8_rollout_rows.append(
        {
            "segment_id": int(_segment_id),
            "transitions": _transition_count,
            "first numerical failure": _rollout[
                "first_numerical_failure"
            ],
            "first material failure": _rollout[
                "first_material_failure"
            ],
            "final position error": float(
                _rollout["position_error"][-1]
            ),
            "maximum position error": float(
                np.max(_rollout["position_error"])
            ),
            "finite": bool(
                np.isfinite(_predicted_positions).all()
            ),
            "maximum outer-bound excess": float(
                np.max(
                    np.abs(_predicted_positions)
                    - np.tile(TODO7_COORDINATE_LIMITS, 2)
                )
            ),
        }
    )

    _available = [
        _horizon
        for _horizon in TODO8_ROLLOUT_HORIZONS
        if _horizon <= _transition_count
    ]
    for _label, _horizon in [
        *[(str(_h), _h) for _h in _available],
        ("full", _transition_count),
    ]:
        _truth = _truth_positions[1 : _horizon + 1]
        _model = _predicted_positions[1 : _horizon + 1]
        _constant = np.broadcast_to(
            _truth_positions[0], _truth.shape
        )
        _model_rmse = float(
            np.sqrt(np.mean((_model - _truth) ** 2))
        )
        _baseline_rmse = float(
            np.sqrt(np.mean((_constant - _truth) ** 2))
        )
        _todo8_horizon_rows.append(
            {
                "segment_id": int(_segment_id),
                "horizon": _label,
                "evaluated transitions": _horizon,
                "diagnostic simulator position RMSE": _model_rmse,
                "constant-position RMSE": _baseline_rmse,
                "simulator / baseline": (
                    _model_rmse / _baseline_rmse
                ),
            }
        )

todo8_rollout_summary = pd.DataFrame(
    _todo8_rollout_rows
).set_index("segment_id")
todo8_rollout_horizons = pd.DataFrame(_todo8_horizon_rows)
todo8_rollout_macro = (
    todo8_rollout_horizons.groupby("horizon", sort=False)
    .agg(
        segments=("segment_id", "nunique"),
        diagnostic_simulator_RMSE=(
            "diagnostic simulator position RMSE", "mean"
        ),
        constant_position_RMSE=(
            "constant-position RMSE", "mean"
        ),
        median_simulator_to_baseline=(
            "simulator / baseline", "median"
        ),
    )
)

assert todo8_rollout_summary["finite"].all()
assert (
    todo8_rollout_summary["maximum outer-bound excess"].max()
    <= 1e-9
)

_todo8_display_segment = int(
    todo8_rollout_summary["first material failure"].idxmax()
)
_todo8_display_trajectory = todo8_rollouts[
    _todo8_display_segment
]["trajectory"]
_todo8_display_count = min(500, len(_todo8_display_trajectory))

_todo8_figure, _todo8_axes = plt.subplots(
    2, 2, figsize=(15, 11)
)
todo8_rollout_summary[
    ["first numerical failure", "first material failure"]
].plot.bar(ax=_todo8_axes[0, 0])
_todo8_axes[0, 0].set_title(
    "Exact-rule failure и material divergence"
)
_todo8_axes[0, 0].set_xlabel("segment_id")
_todo8_axes[0, 0].set_ylabel("update step")
_todo8_axes[0, 0].grid(axis="y", alpha=0.25)

for _object_name, _color in [
    ("object1", "tab:orange"),
    ("object2", "tab:green"),
]:
    _events = todo8_failure_events.loc[
        todo8_failure_events["object"].eq(_object_name)
    ]
    _todo8_axes[0, 1].scatter(
        _events["previous_x"],
        _events["previous_y"],
        s=3,
        alpha=0.30,
        label=_object_name,
        color=_color,
    )
_todo8_axes[0, 1].set_title(
    "Conditional failures сосредоточены у стен"
)
_todo8_axes[0, 1].set_xlim(-245, 245)
_todo8_axes[0, 1].set_ylim(-185, 185)
_todo8_axes[0, 1].set_aspect("equal", adjustable="box")
_todo8_axes[0, 1].legend()
_todo8_axes[0, 1].grid(alpha=0.2)

for _axis, _object_index in [
    (_todo8_axes[1, 0], 1),
    (_todo8_axes[1, 1], 2),
]:
    _axis.plot(
        _todo8_display_trajectory[
            f"truth_x{_object_index}"
        ].iloc[:_todo8_display_count],
        _todo8_display_trajectory[
            f"truth_y{_object_index}"
        ].iloc[:_todo8_display_count],
        color="black",
        linewidth=1.0,
        label="data",
    )
    _axis.plot(
        _todo8_display_trajectory[
            f"predicted_x{_object_index}"
        ].iloc[:_todo8_display_count],
        _todo8_display_trajectory[
            f"predicted_y{_object_index}"
        ].iloc[:_todo8_display_count],
        color="crimson",
        linewidth=0.8,
        alpha=0.8,
        label="diagnostic continuation",
    )
    _axis.set_title(
        f"segment {_todo8_display_segment}, object {_object_index}, "
        f"first {_todo8_display_count - 1} updates"
    )
    _axis.set_aspect("equal", adjustable="box")
    _axis.grid(alpha=0.2)
    _axis.legend()

_todo8_figure.tight_layout()
display(
    todo8_rollout_summary.style.format(precision=6),
    todo8_rollout_macro.style.format(precision=6),
)
_todo8_figure


In [ ]:
TODO8_STATUS = "cross_segment_baseline_audit_no_blind_holdout"
TODO8_MEDIAN_EXACT_RULE_PREFIX = float(
    todo8_rollout_summary["first numerical failure"].median()
)
TODO8_MEDIAN_MATERIAL_PREFIX = float(
    todo8_rollout_summary["first material failure"].median()
)
TODO8_MINIMUM_MATERIAL_PREFIX = int(
    todo8_rollout_summary["first material failure"].min()
)
TODO8_MAXIMUM_MATERIAL_PREFIX = int(
    todo8_rollout_summary["first material failure"].max()
)
TODO8_TEST_OPENED = False
TODO8_HOLDOUT_PRISTINE = False
todo8_manifest = pd.Series(
    {
        "status": TODO8_STATUS,
        "evaluation role": "cross-segment forensic audit",
        "teacher-forced transitions": len(
            todo8_transition_audit
        ),
        "diagnostic exact complete-state share": float(
            todo8_teacher_summary[
                "diagnostic exact complete-state share"
            ]
        ),
        "diagnostic exact positions share": float(
            todo8_teacher_summary[
                "diagnostic exact positions share"
            ]
        ),
        "recursive segments": len(todo8_rollout_summary),
        "median first exact-rule failure": (
            TODO8_MEDIAN_EXACT_RULE_PREFIX
        ),
        "median first material failure": (
            TODO8_MEDIAN_MATERIAL_PREFIX
        ),
        "material failure range": (
            TODO8_MINIMUM_MATERIAL_PREFIX,
            TODO8_MAXIMUM_MATERIAL_PREFIX,
        ),
        "all diagnostic rollouts finite": bool(
            todo8_rollout_summary["finite"].all()
        ),
        "simultaneous-hit selector": (
            "upper-right -> horizontal; otherwise vertical"
        ),
        "reset modeled": False,
        "test opened": TODO8_TEST_OPENED,
        "holdout pristine": TODO8_HOLDOUT_PRISTINE,
    },
    name="value",
)
display(todo8_manifest.to_frame())
display(
    Markdown(
        f"**TODO8 status:** `{TODO8_STATUS}`. Diagnostic causal "
        f"one-step воспроизводит полный state точно в "
        f"**{100 * todo8_teacher_summary['diagnostic exact complete-state share']:.2f}%** "
        f"переходов. First exact-rule failure: median "
        f"**{TODO8_MEDIAN_EXACT_RULE_PREFIX:.0f}** updates. "
        f"Diagnostic material failure: median "
        f"**{TODO8_MEDIAN_MATERIAL_PREFIX:.0f}**, range "
        f"**{TODO8_MINIMUM_MATERIAL_PREFIX}–"
        f"{TODO8_MAXIMUM_MATERIAL_PREFIX}**."
    )
)


### Итог TODO 8

Получен первый causal baseline simulator. Diagnostic teacher-forced one-step
точно воспроизводит полный записанный state примерно в 96% переходов и
координаты примерно в 97%. Для standard collision найден причинный обеосевой
priority selector, поэтому будущий угол для one-step prediction не используется.
Но это ещё не готовый генератор: специальная wall/loop-ветвь не восстановлена.

Recursive continuation показывает механизм ошибки: один неверно выбранный
collision-angle меняет последующее состояние, и хаотическая траектория
расходится. `todo8_event_table` и `todo8_failure_events` теперь являются
готовым датасетом для интерпретируемого stateful mode model. Reset остаётся
внешней границей эпизода.

## TODO 9 — causal boundary state machine и residual ML gate

TODO 9 проверяет следующий уровень гибридной модели: точная кинематика и
геометрия остаются детерминированными, а отдельный causal gate выбирает
редкий wall/loop-режим. Это leave-one-segment-out forensic evaluation, а не
новый blind test: все сегменты уже участвовали в discovery.

Контракт:

- target и post-hoc event labels могут использовать текущую строку только для
  оценки; classifier получает состояние и историю до предсказываемого update;
- сначала проверяется явная period-2 state machine;
- shallow tree используется как rule-discovery probe;
- fixed Random Forest проверяет, хватает ли causal признакам ёмкости для
  residual special gate;
- special angle аппроксимируется train-only семейством `k*90° ± theta`;
- object 2 получает distance, вычисленный из OOF-предсказания object 1;
- numerical exact и material tolerances показываются раздельно;
- recursive rollout остаётся закрыт, если exact one-step gate не пройден.

Реализация вынесена в `notebooks/todo9_hybrid.py`, чтобы сложный event-model
оставался тестируемым и не скрывался внутри большого JSON notebook.

In [ ]:
_todo9_companion = PROJECT_ROOT / "notebooks" / "todo9_hybrid.py"
exec(
    compile(
        _todo9_companion.read_text(encoding="utf-8"),
        str(_todo9_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 9

Period-2 оказался двухфазным состоянием, а не одним классом special-строк:
явное causal continuation-rule восстанавливает его почти полностью. Остальные
special angles лежат около восьми направлений `k*90° ± 11.43°`; это даёт
точную структурную подсказку, но пока только приблизительное значение угла.

Residual Random Forest проходит event-level F1 gate и вместе с state-machine
резко уменьшает practically material one-step error. Однако strict numerical
exact complete-state gate не пройден: приблизительный angular snap иногда
портит ранее точные standard transitions. Поэтому модель является сильным
teacher-forced OOF diagnostic, но recursive rollout в TODO 9 намеренно не
запускается. Следующая задача — вывести точную формулу special angle или заранее
зафиксировать отдельный prediction-oriented material gate до нового rollout.

## TODO 10 — точная special-angle формула и условно открытый rollout

TODO 10 сначала классифицирует оставшиеся 69 material one-step ошибок, затем
проверяет найденное аналитическое тождество special-angle. LOSO Random Forest
из TODO 9 и его признаки не меняются: так улучшение нельзя приписать новому
подбору classifier после просмотра результата.

Контракт, зафиксированный до нового запуска:

- для proposed free-angle `alpha_free` величина `phi` — расстояние до ближайшей
  оси, а проверяемая формула равна
  `theta = atan((2 / base_step) / cos(phi))`; при `base_step=10` коэффициент
  равен `0.2`;
- current/future angle используется только как target для post-hoc проверки;
  формула на prediction получает только causal `free_angle`;
- формула должна быть numerical-exact на всех обычных special wall-событиях
  во всех 11 сегментах; segment-start hidden state учитывается отдельно;
- полный sequential LOSO one-step должен пройти прежний exact gate `98.17%`;
- recursive rollout открывается автоматически только после прохождения gate;
- rollout сравнивается с TODO 8 и constant-position baseline на горизонтах
  10, 50, 100, 500 и full; прежний target для медианы первого material failure
  остаётся 178 update.

Реализация вынесена в `notebooks/todo10_exact_special.py`.

In [ ]:
_todo10_companion = PROJECT_ROOT / "notebooks" / "todo10_exact_special.py"
exec(
    compile(
        _todo10_companion.read_text(encoding="utf-8"),
        str(_todo10_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 10

Для 1 146 обычных special wall-событий восстановлена numerical-exact causal
формула `tan(theta) cos(phi) = 2 / 10 = 0.2`. Максимальная ошибка `theta`
равна примерно `5.3e-8°`; current/future angle в правой части не используется.

С неизменённым LOSO Random Forest полный sequential exact one-step вырос
`96.21% → 98.69%` и прошёл прежний gate `98.17%`; material share достиг
`99.84%` (58 ошибок из 36 664). Поэтому recursive rollout был открыт.
Медианный first material failure вырос `89 → 189` update. Macro coordinate
RMSE TODO10 против TODO8 равен `4.09 vs 8.60` на горизонте 100 и
`61.66 vs 95.35` на горизонте 500; на full horizon `158.84` также лучше
constant-position `169.25`. Все заранее записанные rollout gates пройдены.

Это сильный результат для восстановленного алгоритма данного синтетического
симулятора, но не blind generalization: все 11 сегментов уже участвовали в
forensic discovery. Основной остаток теперь находится в ML selector
(false-positive/false-negative special-mode), а один segment-start transition
зависит от скрытого состояния до начала записи.

## TODO 11 — stateful special selector с nested LOSO

Точная special-angle формула из TODO 10 остаётся неизменной. TODO 11 меняет
только процесс включения residual special-ветви: вместо одного порога `0.5`
используются разные пороги входа `T_enter` и продолжения `T_stay`.

Контракт, зафиксированный до просмотра TODO 11 результатов:

- `T_enter` применяется, если собственное предыдущее решение selector было не
  special; `T_stay <= T_enter` удерживает собственный predicted special-state;
- RF architecture и causal features из TODO 9 не подбираются заново;
- для каждого outer held-out сегмента пороги выбираются только по остальным
  сегментам через inner LOSO; pairwise models исключают одновременно outer и
  inner сегмент, чтобы outer labels не влияли на threshold;
- primary inner objective — заранее зафиксированный object-level surrogate:
  macro conditional material error по сегментам при фиксированном causal
  baseline; special F1 и exact error используются как следующие tie-break;
  окончательный outer score считается уже полным sequential проходом
  `object1 -> predicted distance -> object2`;
- outer one-step принимает selector, только если осталось не более 40 material
  failures, residual special F1 не ниже 0.90, exact/material shares выше TODO 10
  и улучшены хотя бы 8 из 11 сегментов;
- recursive rollout открывается только после полного one-step gate; затем он
  должен превысить TODO 10 median 189 и уменьшить RMSE 4.09/61.66 на
  горизонтах 100/500, не ухудшив full horizon;
- current/future target angle, branch и event label не являются model inputs;
  в recursive режиме past observed mode заменяется собственным predicted mode.

Реализация вынесена в `notebooks/todo11_stateful_selector.py`.

In [ ]:
_todo11_companion = PROJECT_ROOT / "notebooks" / "todo11_stateful_selector.py"
exec(
    compile(
        _todo11_companion.read_text(encoding="utf-8"),
        str(_todo11_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 11

Nested-LOSO hysteresis улучшил residual selector: `F1 0.8810 -> 0.9159`.
Во внешнем полном sequential one-step exact complete-state share выросла
`98.6935% -> 98.7508%`, material share — `99.8418% -> 99.8664%`, а число
material failures уменьшилось `58 -> 49` (`-15.5%`). Exact share улучшена
на 9 из 11 held-out сегментов.

Inner folds независимо выбрали `T_enter=0.55-0.60` и более низкий
`T_stay=0.30-0.45`: данные действительно поддерживают идею, что войти в
special-state следует осторожнее, чем продолжать уже начавшийся режим.
Остаток, однако, всё ещё состоит из 24 false positive, 24 false negative и
одного segment-start hidden-state перехода. Maximum-angle MAE также не
улучшилась (`0.009962° -> 0.010013°`).

Заранее установленный gate требовал не более 40 material failures. Получено
49, поэтому TODO 11 считается полезным, но не принятым улучшением TODO 10.
Recursive rollout намеренно не запускался. Следующая гипотеза должна
разделить **entry** и **continuation** как разные задачи, а не продолжать
подбирать два порога одной и той же RF-вероятности.

## TODO 12 — entry/continuation causal automaton

TODO 12 сохраняет exact transition map TODO 10 и hysteresis TODO 11 как
обязательный baseline, но разделяет включение special-ветви на две разные
задачи: первый `entry` и последующие `continuation/exit`. Trajectory ensemble
в этот эксперимент не входит.

Контракт, зафиксированный до просмотра TODO 12 результатов:

- автомат имеет состояния `NORMAL`, `SPECIAL_ACTIVE` и приоритетный
  детерминированный `PERIOD2`; state сбрасывается на границе сегмента, а
  выход из PERIOD2 проходит через continuation/exit policy;
- entry RF обучается только на causal candidate-событиях после неактивной
  наблюдавшейся истории; continuation RF — на событиях сразу после
  наблюдавшегося active-state и учится как продолжению, так и выходу;
- target-derived state используется только для формирования train population
  и метрик. При inference модель выбирается по собственному predicted state;
  current/future targets не входят в признаки;
- RF architecture и exact special-angle formula не подбираются заново;
  entry thresholds `0.50..0.95`, continuation thresholds `0.15..0.80`;
- для каждого outer segment thresholds выбираются по остальным сегментам
  pairwise inner LOSO; каждая пара entry/continuation models исключает и
  outer, и inner segment;
- primary inner objective — macro material complete-state error полного
  teacher-forced one-step прохода `object1 -> predicted distance -> object2`;
  прошлое наблюдаемо, но внутри текущей строки используется predicted object1;
  tie-break: residual-run F1,
  exact error, residual event F1, затем более строгие thresholds;
- promotion gate относительно TODO 11: не более 40 material failures,
  residual F1 не ниже 0.915, exact и material shares строго выше TODO 11,
  exact лучше TODO 10 минимум на 8 из 11 outer segments; residual-run F1
  не ниже 0.80, median absolute onset/exit delay не больше 2 updates;
- recursive rollout открывается только после полного one-step gate: median
  first material failure не ниже 190, RMSE ниже TODO 10 на горизонтах 100 и
  500, full-horizon RMSE не хуже TODO 10.

Реализация вынесена в `notebooks/todo12_entry_continuation.py`.

In [ ]:
_todo12_companion = PROJECT_ROOT / "notebooks" / "todo12_entry_continuation.py"
exec(
    compile(
        _todo12_companion.read_text(encoding="utf-8"),
        str(_todo12_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 12

Полностью раздельные entry/continuation RF не улучшили TODO 11. Residual
event F1 составил `0.9030`, residual-run F1 — `0.8997`; median absolute
onset и exit delays равны нулю. Однако exact complete-state share снизилась
`98.7508% -> 98.6744%`, material share — `99.8664% -> 99.8227%`, а число
material failures выросло `49 -> 65`.

Остаток: 31 missed entry, 6 missed continuation, 27 false-special/late-exit
и один hidden segment-start. Entry recall равен `0.8835`, continuation
recall `0.9259`, correct-exit share `0.9961`. Следовательно, раздельный
continuation head потенциально полезен, но новый entry head слабее общего
TODO 11 detector. Все promotion gates провалены, recursive rollout не
запускался. Следующая заранее отделённая ablation сохраняет TODO 11 RF для
entry и применяет отдельный classifier только в active-state.

## TODO 13 — TODO11-entry + continuation automaton

TODO 13 — заранее отделённая ablation после отрицательного TODO 12. Это не
ансамбль: в каждый момент вызывается ровно один classifier. В состоянии
`NORMAL` используется проверенный общий residual RF из TODO 11; в состоянии
`SPECIAL_ACTIVE/PERIOD2` — отдельный continuation/exit RF из TODO 12.

Контракт до просмотра TODO 13 результатов:

- exact transition geometry, period-2 rule и special-angle formula frozen;
- entry thresholds `0.50..0.95`, continuation thresholds `0.15..0.80`;
- inner pairwise general RF исключает outer и inner segments; continuation
  probabilities и deterministic outcomes переиспользуются из TODO 12 folds;
- thresholds выбираются по полному teacher-forced sequential проходу
  `object1 -> predicted distance -> object2`; primary macro material error,
  затем residual-run F1, exact error, event F1 и строгие thresholds;
- promotion gates неизменны: не более 40 material failures, residual F1
  не ниже 0.915, exact/material строго выше TODO 11, exact лучше TODO 10 на
  8/11 сегментов, residual-run F1 >= 0.80 и median onset/exit delay <=2;
- recursive rollout только после полного one-step gate и по прежним TODO 10
  gates: median >=190, меньший RMSE на 100/500, full не хуже TODO 10;
- trajectory ensemble и post-hoc расширение grid запрещены.

Реализация: `notebooks/todo13_hybrid_automaton.py`.

In [ ]:
_todo13_companion = PROJECT_ROOT / "notebooks" / "todo13_hybrid_automaton.py"
exec(
    compile(
        _todo13_companion.read_text(encoding="utf-8"),
        str(_todo13_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 13

Hybrid автомат полностью устранил деградацию раздельного TODO 12, но не
превзошёл более простой TODO 11. Material failures вернулись `65 -> 49`,
однако это ровно уровень TODO 11. Exact share равна `98.7481%` против
`98.7508%` (`459` против `458` exact failures), residual F1 — `0.91549`
против `0.91586`. Residual-run F1 составила `0.91343`, median onset/exit
delays — 0 updates.

Entry recall `0.8993`, continuation recall `0.9630`, correct-exit share
`0.9980`. Среди 49 material failures: 24 missed entry, 23 false/late
special, 1 missed continuation и 1 hidden segment-start. Значит отдельный
continuation head почти идеально выполняет узкую задачу, но меняет лишь
две ошибки и не даёт net gain. TODO 11 остаётся лучшим и более простым
stateful selector; TODO 12/13 сохраняются как честные ablations. Promotion
gate не пройден, recursive rollout не запускался. Дальше нужен не новый
classifier continuation, а анализ причинных различий 24 missed-entry и 24
ложных entry-событий TODO 11.

## TODO 14. Forensic-анализ ошибок лучшего TODO 11 selector

Это отдельная диагностическая ячейка, а не новый этап подбора модели. Она
разбирает уже полученные outer-LOSO предсказания TODO 11 после оценки и не
меняет признаки, Random Forest, `T_enter/T_stay` или promotion gates.

Единица анализа — решение `object × transition`. `PERIOD2`, скрытый старт
сегмента, отсутствие causal candidate и вызванный RF разделяются. Для
object 2 признаки заново строятся с предсказанным после object 1 расстоянием:
иначе анализировался бы не тот sequential pipeline, который дал 49 material
failures.

Проверяются: FP/FN по сегментам и фазам entry/continuation, signed probability
margin относительно активного порога, физические collision-features и
ближайшие состояния противоположного класса из другого held-out сегмента.
Robust scaling для каждого focal segment считается только по остальным
сегментам. Близкие пары и feature shifts — источники следующих гипотез, но не
доказательство скрытой физической переменной и не разрешение подогнать TODO 11.

In [ ]:
_todo14_companion = PROJECT_ROOT / "notebooks" / "todo14_error_forensics.py"
exec(
    compile(
        _todo14_companion.read_text(encoding="utf-8"),
        str(_todo14_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 14

Все 49 material failures TODO 11 локализованы в выборе residual-special
режима: 24 обычных FN, 24 FP и 1 hidden/reset FN в первой строке сегмента,
где RF не вызывался. Все обычные FN — пропущенные входы (`entry`); среди FP
22 ложных включения из NORMAL и только 2 поздних выхода. Поэтому проблема
точно не в continuation head: TODO 12/13 проверяли не главный bottleneck.

Threshold объясняет лишь часть остатка. Из 48 вызванных RF ошибок 17 (`35.4%`)
находятся не дальше 0.05 от активного порога и 29 (`60.4%`) — не дальше 0.10,
но 19 (`39.6%`) являются уверенными ошибками с `|margin| > 0.10`. Медианный
absolute margin равен `0.0891` для FN и `0.0748` для FP. Значит одним общим
сдвигом threshold одновременно убрать FP и FN нельзя.

Ошибки неоднородны: три сегмента содержат 29 из 49 failures (`59.2%`), а
сегмент 3 не содержит ни одной. FN особенно связаны с object 2 (`16/24`), FP
— с object 1 (`15/24`). Для FN ближайший корректный пример противоположного
класса из другого сегмента оказывается не дальше корректного TP в `91.7%`
случаев (median distance ratio `0.601`): доступные causal-признаки входа сильно
перекрываются. Для FP картина обратная: только `8.3%`, median ratio `1.374`;
они больше похожи на обычные TN и указывают на переобобщение RF.

Практический вывод: TODO 11 остаётся лучшим selector, но остаток смешанный.
Пропущенным entry, вероятно, не хватает причинной истории/скрытой фазы; ложные
entry требуют более консервативного или abstaining решения. Это диагностическая
гипотеза, а не новый fitted result: TODO 14 ничего не переобучает и не меняет
зафиксированные пороги.

## TODO 15. History-aware entry detector

Проверяется ровно одна гипотеза TODO 14: для распознавания перехода
`NORMAL → SPECIAL` одной текущей строки недостаточно. Entry RF получает
текущий causal proposal и предыдущие `K ∈ {2, 4, 8}` строк того же объекта
в том же reset-сегменте. Для отсутствующих лагов используются validity flags;
история никогда не пересекает границу объекта или сегмента.

`K` и `T_entry` выбираются для каждого outer segment полным sequential nested
LOSO без его labels. Inner simulation сохраняет порядок `object1 → predicted
distance → object2`. PERIOD2, collision geometry, special-angle formula, общий
TODO 11 RF в активном состоянии и его `T_stay` заморожены. Поэтому отличие от
TODO 11 измеряет именно ценность дополнительных строк для входа.

Это teacher-forced causal one-step проверка: предыдущие наблюдённые строки
доступны, будущие значения и текущие target/actual branch не используются.
Recursive rollout не запускается в этой ячейке; сначала требуется прежний gate
`≤40` material failures, улучшение global exact/material и минимум 8/11
сегментов.

In [ ]:
_todo15_companion = PROJECT_ROOT / "notebooks" / "todo15_history_entry.py"
exec(
    compile(
        _todo15_companion.read_text(encoding="utf-8"),
        str(_todo15_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 15

Короткая causal history в виде плоского окна не улучшила распознавание входа.
Nested LOSO во всех 11 outer folds выбрал минимальное окно `K=2`; более
длинные окна ухудшались монотонно. Относительно замороженного TODO 11 exact
share снизилась `98.7508% → 98.5463%`, material share —
`99.8664% → 99.6591%`, failures выросли `49 → 125`, а residual F1 упал
`0.9162 → 0.8777`. Улучшены только 2 из 11 сегментов, поэтому не пройден
ни один acceptance gate и recursive rollout не открывается.

Это не доказывает, что прошлое бесполезно. Оно показывает более узкий
результат: RF не умеет надёжно извлечь фазу входа из простого concatenation
2–8 сырых предыдущих строк. Следующая проверка должна сначала отделить эффект
новой entry-population с помощью `K=0` control, а затем заменить сырые лаги
компактными динамическими признаками (causal differences, approaching-wall
trend, run phase/time-since-event) или отдельной sequence-моделью.

## TODO 16. Compact causal dynamics control

TODO 15 смешивал два изменения: отдельную training population для entry-head
и добавление сырых lag-блоков. Здесь они разделены. `current_only` и
`compact_dynamics` используют одну entry-population, одну RF-архитектуру,
одну threshold grid и один полный sequential nested LOSO. Отличаются только
входные признаки.

Compact-вариант не получает прошлые строки целиком. Из двух предыдущих строк
того же объекта/reset-сегмента вычисляются только causal differences:
перемещение и разности по update (не физическая acceleration), wrapped angular change, penetration/gap trend,
crossing/contact transition и boundary-run phase. Для object 2 текущие
признаки пересчитываются после predicted object 1 distance.

Geometry, PERIOD2, exact special-angle law, TODO 11 continuation RF и
`T_stay` заморожены; inner selection условна на этих ранее выбранных порогах.
Одинаковая сетка `T_entry=0.50–0.95` сохраняет `T_entry ≥ T_stay`.
В обоих entry-head текущие обучающие признаки object2 строятся через
фиксированный base/PERIOD2 teacher object1, а не его текущую truth-позицию.
Это дополнительное отличие от TODO15: прямой matched contrast здесь —
`compact_dynamics` против `current_only`, а не чистая реплика старого head.
Прошлые записи остаются наблюдёнными; age-счётчики используют более длинный
прошлый prefix с cap=8. Успех требует победить оба baseline, получить не более
40 material failures и улучшить exact минимум на 8/11 сегментов.
Это всё ещё teacher-forced causal one-step experiment; rollout закрыт до
прохождения gate.

In [ ]:
_todo16_companion = PROJECT_ROOT / "notebooks" / "todo16_compact_dynamics.py"
exec(
    compile(
        _todo16_companion.read_text(encoding="utf-8"),
        str(_todo16_companion),
        "exec",
    ),
    globals(),
)


### Итог TODO 16

Выполнен чистый запуск: 76/76 code cells, 0 errors. `current_only` —
контроль без нового блока истории (старые признаки прошлого режима TODO 11
сохраняются); `compact_dynamics` добавляет 31 причинный признак.

| Вариант | Exact share | Material failures | Residual F1 |
|---|---:|---:|---:|
| TODO 11 | 98.7508% | 49 | 0.9162 |
| current_only | 98.6881% | 62 | 0.9053 |
| compact_dynamics | 98.6935% | 86 | 0.9064 |

Compact dynamics даёт всего два дополнительных numerical-exact перехода
из 36 664 относительно current_only, но 24 дополнительные material ошибки
(+38.7%). Event recall немного вырос (0.8927 → 0.9040), а precision снизилась
(0.9183 → 0.9088). Значит небольшой выигрыш в распознавании некоторых событий
не превращается в более надёжное движение: exact и material имеют разные
пороги ошибки. По material dynamics лучше control лишь на одном сегменте,
равен на трёх и хуже на семи. Против TODO 11 exact лучше только на 3/11,
material — на 1/11; все acceptance gates не пройдены.

В 86 material-failure transitions код отмечает контекст missed
entry/continuation в 47 случаях, false/late special в 38 и hidden/reset
initialization в одном. Frozen continuation RF не гарантирует прежние
решения: изменённый entry меняет последовательность active/inactive states.

TODO 11 остаётся лучшим stateful selector, TODO 10 — моделью с проверенным
recursive rollout. TODO 16 сохранён как ablation, rollout не запускался.
Отрицательный результат относится к данному RF и representation, а не
доказывает отсутствие закономерности или бесполезность любой истории.
Feature importance использовалась только для отчёта, не для повторного
подбора. Оценка teacher-forced и условна на frozen TODO 11 T_stay; все
сегменты ранее исследовались, поэтому это не независимый pristine test.